# Evo2 variant-effect fine-tuning — one notebook

Kaggle port of the Modal pipeline in `evo2-backend/`. Same `finetune/` code
(embedded below), notebooks instead of Modal functions, and **notebook output as
input** instead of a persistent Volume.

## How to run

Set `STAGE` in the config cell and *Save Version* (Run All). Between runs, add the
previous version's output as input:

> Add Input → **Notebook Output** → this notebook → latest version

Order: `datasets` → `extract` (repeat until all shards done) → `merge` → `train`
→ `publish`. Or set `STAGE = "auto"` and it does the next unfinished thing each
run, extracting shards until the session is nearly out of time.

## Accelerator / settings

| stage | accelerator | internet |
| --- | --- | --- |
| datasets | None (save quota) | **on** |
| extract | **GPU T4 x2** — both T4s required, see below | **on** |
| merge / train / publish | None | off |

**`extract` needs `GPU T4 x2`, not a single GPU.** evo2_7b's weights are ~14 GB
and don't fit one 16 GB T4. vortex pipeline-parallelises the 32 blocks across
every visible CUDA device (~7 GB of weights per T4), and `features._forward`
feeds inputs on `cuda:0` to match — so two T4s work with no extra wiring, one
does not. The cell aborts early with a clear message if it sees < 2 GPUs.

On the T4 (compute capability 7.5) `loader.py` also (a) disables FP8 autocast
(`evo2_7b` sets `use_fp8_input_projections`, which needs CC ≥ 8.9) and (b) casts
the model to float16 (vortex's Triton kernels emit `.bf16` PTX, which needs
sm_80+). Both are approximations of the reference numerics and are no-ops on
Modal's Ampere+ GPUs. Watch the `extract` smoke test for VRAM headroom and for
inf/nan from the fp16 exponent range; `window_size` is held at 2048.

### 1 · Write the embedded `finetune/` package to disk

In [ ]:
import base64, pathlib, sys

SRC = "/kaggle/working/src"
FILES = {
"finetune/__init__.py": "IiIiRmluZS10dW5pbmcgcGlwZWxpbmUgZm9yIEV2bzIgdmFyaWFudCBlZmZlY3QgcHJlZGljdGlvbi4KCkV2bzIgc3RheXMgZnJvemVuLiBUaGUgcGlwZWxpbmUgY2FjaGVzIGl0cyBlbWJlZGRpbmdzIGFuZCBwZXItdG9rZW4KbG9nLXByb2JhYmlsaXRpZXMgZm9yIGxhYmVsbGVkIHZhcmlhbnRzLCB0aGVuIHRyYWlucyBhIHNtYWxsIGNsYXNzaWZpZXIgaGVhZCBvbgp0b3Agb2YgdGhlbSwgcmVwbGFjaW5nIHRoZSBoYW5kLWZpdHRlZCBkZWx0YS1saWtlbGlob29kIHRocmVzaG9sZCB0aGF0IHRoZQp6ZXJvLXNob3QgZW5kcG9pbnQgdXNlcy4KClN0YWdlcywgaW4gb3JkZXI6CgoxLiBgYGRhdGFgYCAgICAgIC0gYnVpbGQgbGFiZWxsZWQgU05WIHRhYmxlcyBmcm9tIENsaW5WYXIgYW5kIHRoZSBCUkNBMSBETVMgc2V0CjIuIGBgZmVhdHVyZXNgYCAgLSBvbmUgR1BVIHBhc3MgcGVyIHZhcmlhbnQsIHNoYXJkZWQgYW5kIHJlc3VtYWJsZQozLiBgYHRyYWluYGAgICAgIC0gZml0IHRoZSBoZWFkLCBwaWNrIGEgdGhyZXNob2xkLCBzY29yZSBhZ2FpbnN0IHRoZSB6ZXJvLXNob3QgYmFzZWxpbmUKNC4gYGBoZWFkYGAgICAgICAtIHRoZSBzZXJpYWxpc2VkIGFydGlmYWN0IHRoZSBlbmRwb2ludCBsb2FkcwoKU2VlIGBgRklORVRVTklORy5tZGBgIGZvciB0aGUgcnVuYm9vay4KIiIiCgpmcm9tIC4gaW1wb3J0IGNvbmZpZyAgIyBub3FhOiBGNDAxCgpfX2FsbF9fID0gWyJjb25maWciLCAiZGF0YSIsICJmZWF0dXJlcyIsICJoZWFkIiwgInNlcXVlbmNlcyIsICJ0cmFpbiJdCg==",
"finetune/config.py": "IiIiU2hhcmVkIGNvbmZpZ3VyYXRpb24gZm9yIHRoZSBFdm8yIHZhcmlhbnQtZWZmZWN0IGZpbmUtdHVuaW5nIHBpcGVsaW5lLgoKRXZlcnl0aGluZyB0aGF0IGJvdGggdGhlIHRyYWluaW5nIGpvYnMgYW5kIHRoZSBpbmZlcmVuY2UgZW5kcG9pbnQgbmVlZCB0byBhZ3JlZSBvbgpsaXZlcyBoZXJlLiBJZiBhIGNvbnN0YW50IGFmZmVjdHMgdGhlIHNoYXBlIG9yIG1lYW5pbmcgb2YgYSBmZWF0dXJlIHZlY3RvciwgaXQKYmVsb25ncyBpbiB0aGlzIGZpbGUgc28gdGhhdCB0cmFpbi9zZXJ2ZSBza2V3IGlzIGltcG9zc2libGUgYnkgY29uc3RydWN0aW9uLgoiIiIKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsLCBUdXBsZQoKIyAtLS0gTW9kYWwgcmVzb3VyY2VzIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCkFQUF9OQU1FID0gImV2bzItZmluZXR1bmUiCgpIRl9DQUNIRV9WT0xVTUUgPSAiaGZfY2FjaGUiCkhGX0NBQ0hFX1BBVEggPSAiL3Jvb3QvLmNhY2hlL2h1Z2dpbmdmYWNlIgoKREFUQV9WT0xVTUUgPSAiZXZvMi1maW5ldHVuZS1kYXRhIgpEQVRBX1BBVEggPSAiL2RhdGEiCgpHRU5PTUVTX0RJUiA9IGYie0RBVEFfUEFUSH0vZ2Vub21lcyIKREFUQVNFVFNfRElSID0gZiJ7REFUQV9QQVRIfS9kYXRhc2V0cyIKRkVBVFVSRVNfRElSID0gZiJ7REFUQV9QQVRIfS9mZWF0dXJlcyIKUlVOU19ESVIgPSBmIntEQVRBX1BBVEh9L3J1bnMiCgojIFRoZSBkZXBsb3llZCBlbmRwb2ludCByZWFkcyB0aGUgdHJhaW5lZCBoZWFkIGZyb20gaGVyZS4KQUNUSVZFX1JVTl9ESVIgPSBmIntSVU5TX0RJUn0vYWN0aXZlIgoKIyAtLS0gTW9kZWwgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCk1PREVMX05BTUUgPSAiZXZvMl83YiIKCiMgRG9jdW1lbnRlZCBpbiB0aGUgRXZvMiBSRUFETUUgYXMgdGhlIGVtYmVkZGluZyB0YXAgZm9yIGV2bzJfN2IuIE91dHB1dCBvZiB0aGUKIyBHTFUgTUxQJ3MgZmluYWwgcHJvamVjdGlvbiBpbiBibG9jayAyOCBvZiAzMiwgc28gaXRzIHdpZHRoIGlzIGhpZGRlbl9zaXplLgpFTUJFRERJTkdfTEFZRVIgPSAiYmxvY2tzLjI4Lm1scC5sMyIKRU1CRURESU5HX0RJTSA9IDQwOTYKCiMgLS0tIFJlZmVyZW5jZSBnZW5vbWVzIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgojIENsaW5WYXIgaXMgZGlzdHJpYnV0ZWQgb24gR1JDaDM4OyB0aGUgRmluZGxheSBCUkNBMSBzYXR1cmF0aW9uLW11dGFnZW5lc2lzCiMgdGFibGUgaXMgb24gR1JDaDM3LCBhbmQgZXZvMiB2ZW5kb3JzIGEgR1JDaDM3IGNocjE3IEZBU1RBLCBzbyBlYWNoIGRhdGFzZXQgaXMKIyBzY29yZWQgYWdhaW5zdCBpdHMgb3duIGFzc2VtYmx5IHJhdGhlciB0aGFuIGxpZnRlZCBvdmVyLgojIE1pcnJvcnMgb2YgdGhlIHNhbWUgZmlsZSwgdHJpZWQgaW4gb3JkZXIuIFRoZSBwcmltYXJ5IGhvc3QgaW50ZXJtaXR0ZW50bHkKIyByZWZ1c2VzIGNvbm5lY3Rpb25zIG91dHJpZ2h0IChUTFMgaGFuZHNoYWtlIHJlc2V0KSBmcm9tIHNvbWUgbmV0d29ya3MsIHdoaWNoCiMgbm8gYW1vdW50IG9mIHJldHJ5aW5nIGZpeGVzOyBoZ2Rvd25sb2FkMiBzZXJ2ZXMgYnl0ZS1pZGVudGljYWwgZGF0YSB1bmRlciB0aGUKIyBzYW1lIHBhdGgsIGluY2x1ZGluZyB0aGUgImNociItcHJlZml4ZWQgc2VxdWVuY2UgbmFtZXMgdGhlIHBpcGVsaW5lIGV4cGVjdHMuCkhHMzhfVVJMUyA9ICgKICAgICJodHRwczovL2hnZG93bmxvYWQuc29lLnVjc2MuZWR1L2dvbGRlblBhdGgvaGczOC9iaWdaaXBzL2hnMzguZmEuZ3oiLAogICAgImh0dHBzOi8vaGdkb3dubG9hZDIuc29lLnVjc2MuZWR1L2dvbGRlblBhdGgvaGczOC9iaWdaaXBzL2hnMzguZmEuZ3oiLAopCkhHMzhfVVJMID0gSEczOF9VUkxTWzBdCkhHMzhfRkFTVEEgPSBmIntHRU5PTUVTX0RJUn0vaGczOC5mYSIKCiMgUHJlc2VudCBpbnNpZGUgdGhlIGltYWdlLCBjbG9uZWQgYnkgdGhlIERvY2tlcmZpbGUgc3RlcCBpbiBtYWluLnB5LgpIRzE5X0NIUjE3X0ZBU1RBX0daID0gIi9ldm8yL25vdGVib29rcy9icmNhMS9HUkNoMzcucDEzX2NocjE3LmZuYS5neiIKSEcxOV9DSFIxN19GQVNUQSA9IGYie0dFTk9NRVNfRElSfS9HUkNoMzcucDEzX2NocjE3LmZhIgoKQlJDQTFfRE1TX1hMU1ggPSAiL2V2bzIvbm90ZWJvb2tzL2JyY2ExLzQxNTg2XzIwMThfNDYxX01PRVNNM19FU00ueGxzeCIKCkNMSU5WQVJfVkNGX1VSTCA9ICgKICAgICJodHRwczovL2Z0cC5uY2JpLm5sbS5uaWguZ292L3B1Yi9jbGludmFyL3ZjZl9HUkNoMzgvY2xpbnZhci52Y2YuZ3oiCikKQ0xJTlZBUl9WQ0ZfR1ogPSBmIntEQVRBU0VUU19ESVJ9L2NsaW52YXIudmNmLmd6IgoKIyBCUkNBMSBsb2N1cywgdXNlZCB0byBob2xkIHRoZSBETVMgYmVuY2htYXJrIG91dCBvZiB0aGUgQ2xpblZhciB0cmFpbmluZyBzZXQuCiMgQ29vcmRpbmF0ZXMgYXJlIDEtYmFzZWQgaW5jbHVzaXZlLgpCUkNBMV9MT0NVUyA9IHsKICAgICJoZzM4IjogKCJjaHIxNyIsIDQzXzA0NF8yOTUsIDQzXzE3MF8yNDUpLAogICAgImhnMTkiOiAoImNocjE3IiwgNDFfMTk2XzMxMiwgNDFfMjc3XzUwMCksCn0KCiMgLS0tIEZlYXR1cmUgZXh0cmFjdGlvbiAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgRmVhdHVyZUNvbmZpZzoKICAgICIiIkRlZmluZXMgdGhlIGZlYXR1cmUgdmVjdG9yIHByb2R1Y2VkIGZvciBhIHNpbmdsZSBTTlYuCgogICAgYGB3aW5kb3dfc2l6ZWBgIGRvbWluYXRlcyBHUFUgY29zdCAoU3RyaXBlZEh5ZW5hIGlzIGNsb3NlIHRvIGxpbmVhciBpbgogICAgc2VxdWVuY2UgbGVuZ3RoKSwgc28gaGFsdmluZyBpdCByb3VnaGx5IGhhbHZlcyB0aGUgZXh0cmFjdGlvbiBiaWxsLiBJdAogICAgZGVmYXVsdHMgdG8gODE5MiB0byBtYXRjaCB0aGUgd2luZG93IHRoZSB6ZXJvLXNob3QgZW5kcG9pbnQgYWxyZWFkeSBzY29yZXMuCiAgICAiIiIKCiAgICB3aW5kb3dfc2l6ZTogaW50ID0gODE5MgoKICAgICMgUG9vbGluZyByYWRpaSBpbiBiYXNlcyBhcm91bmQgdGhlIHZhcmlhbnQsIGFwcGxpZWQgdG8gYm90aCB0aGUgZW1iZWRkaW5nCiAgICAjIGFuZCB0aGUgcGVyLXRva2VuIGxvZy1wcm9iYWJpbGl0aWVzLiBgYDBgYCBpcyB0aGUgdmFyaWFudCBwb3NpdGlvbiBhbG9uZTsKICAgICMgYGBOb25lYGAgaXMgdGhlIHdob2xlIHdpbmRvdy4KICAgIHBvb2xfcmFkaWk6IFR1cGxlW09wdGlvbmFsW2ludF0sIC4uLl0gPSAoMCwgMTI4LCBOb25lKQoKICAgICMgUmFkaXVzIHVzZWQgZm9yIHRoZSBzY2FsYXIgImxvY2FsIiBkZWx0YS1saWtlbGlob29kIGZlYXR1cmUuCiAgICBsb2NhbF9yYWRpdXM6IGludCA9IDY0CgogICAgZW1iZWRkaW5nX2xheWVyOiBzdHIgPSBFTUJFRERJTkdfTEFZRVIKICAgIGVtYmVkZGluZ19kaW06IGludCA9IEVNQkVERElOR19ESU0KCiAgICAjIFNjb3JlIHRoZSByZWZlcmVuY2UgYW5kIHZhcmlhbnQgc2VxdWVuY2VzIGFzIGEgc2luZ2xlIGJhdGNoIG9mIDIuIEJldHRlcgogICAgIyBIMTAwIHV0aWxpc2F0aW9uIHRoYW4gdHdvIGJhdGNoLTEgZm9yd2FyZHM7IHNldCBGYWxzZSBpZiB5b3UgaGl0IE9PTS4KICAgIHBhaXJfYmF0Y2g6IGJvb2wgPSBUcnVlCgogICAgQHByb3BlcnR5CiAgICBkZWYgbl9lbWJlZGRpbmdfZmVhdHVyZXMoc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBsZW4oc2VsZi5wb29sX3JhZGlpKSAqIHNlbGYuZW1iZWRkaW5nX2RpbQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIG5fc2NhbGFyX2ZlYXR1cmVzKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gbGVuKFNDQUxBUl9GRUFUVVJFX05BTUVTKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIG5fZmVhdHVyZXMoc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBzZWxmLm5fZW1iZWRkaW5nX2ZlYXR1cmVzICsgc2VsZi5uX3NjYWxhcl9mZWF0dXJlcwoKICAgIGRlZiB0YWcoc2VsZikgLT4gc3RyOgogICAgICAgICIiIlN0YWJsZSBpZGVudGlmaWVyIHNvIGZlYXR1cmUgY2FjaGVzIHdpdGggZGlmZmVyZW50IHNldHRpbmdzIGRvbid0IG1peC4iIiIKICAgICAgICByYWRpaSA9ICItIi5qb2luKCJmdWxsIiBpZiByIGlzIE5vbmUgZWxzZSBzdHIocikgZm9yIHIgaW4gc2VsZi5wb29sX3JhZGlpKQogICAgICAgIHJldHVybiBmInd7c2VsZi53aW5kb3dfc2l6ZX1fbHtzZWxmLmVtYmVkZGluZ19sYXllci5yZXBsYWNlKCcuJywgJ18nKX1fcHtyYWRpaX0iCgoKIyBPcmRlciBtYXR0ZXJzOiB0aGUgaW5mZXJlbmNlIHBhdGggcmVidWlsZHMgdGhlIHZlY3RvciBmcm9tIHRoZXNlIG5hbWVzLgpTQ0FMQVJfRkVBVFVSRV9OQU1FUyA9ICgKICAgICJkZWx0YV9zY29yZV9mdWxsIiwgICAgICAjIG1lYW4gbG9nLXByb2IodmFyaWFudCB3aW5kb3cpIC0gbWVhbiBsb2ctcHJvYihyZWZlcmVuY2Ugd2luZG93KQogICAgImRlbHRhX3Njb3JlX2xvY2FsIiwgICAgICMgc2FtZSwgcmVzdHJpY3RlZCB0byArLy0gbG9jYWxfcmFkaXVzIGFyb3VuZCB0aGUgdmFyaWFudAogICAgImRlbHRhX2xwX2F0X3ZhcmlhbnQiLCAgICMgbG9nLXByb2Igb2YgdGhlIHN1YnN0aXR1dGVkIGJhc2UgbWludXMgdGhhdCBvZiB0aGUgcmVmZXJlbmNlIGJhc2UKICAgICJyZWZfc2NvcmVfZnVsbCIsICAgICAgICAjIHJlZmVyZW5jZSB3aW5kb3cgbWVhbiBsb2ctcHJvYjogYSBjb25zZXJ2YXRpb24gLyBjb250ZXh0IHByb3h5CiAgICAicmVmX2xwX2F0X3ZhcmlhbnQiLCAgICAgIyBob3cgY29uZmlkZW50bHkgdGhlIG1vZGVsIHByZWRpY3RzIHRoZSByZWZlcmVuY2UgYmFzZQogICAgInZhcl9scF9hdF92YXJpYW50IiwKKQoKREVGQVVMVF9GRUFUVVJFX0NPTkZJRyA9IEZlYXR1cmVDb25maWcoKQoKIyAtLS0gVHJhaW5pbmcgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCgpAZGF0YWNsYXNzCmNsYXNzIFRyYWluQ29uZmlnOgogICAgaGVhZDogc3RyID0gIm1scCIgICAgICAgICAgICAgICMgIm1scCIgb3IgImxpbmVhciIKICAgIGhpZGRlbl9zaXplczogVHVwbGVbaW50LCAuLi5dID0gKDI1NiwgNjQpCiAgICBkcm9wb3V0OiBmbG9hdCA9IDAuMwogICAgbHI6IGZsb2F0ID0gMWUtMwogICAgd2VpZ2h0X2RlY2F5OiBmbG9hdCA9IDFlLTIKICAgIGJhdGNoX3NpemU6IGludCA9IDI1NgogICAgbWF4X2Vwb2NoczogaW50ID0gMTAwCiAgICBwYXRpZW5jZTogaW50ID0gMTAgICAgICAgICAgICAgIyBlYXJseS1zdG9wcGluZyBwYXRpZW5jZSBvbiB2YWxpZGF0aW9uIEFVUk9DCiAgICBzZWVkOiBpbnQgPSAwCgogICAgIyBDaHJvbW9zb21lcyBoZWxkIG91dCBvZiB0cmFpbmluZy4gS2VwdCBkaXNqb2ludCBzbyB0aGF0IHZhcmlhbnRzIGluCiAgICAjIGxpbmthZ2Ugd2l0aCBlYWNoIG90aGVyIGNhbm5vdCBzdHJhZGRsZSB0aGUgc3BsaXQuCiAgICB2YWxfY2hyb21vc29tZXM6IFR1cGxlW3N0ciwgLi4uXSA9ICgiY2hyOCIsICJjaHIxOCIpCiAgICB0ZXN0X2Nocm9tb3NvbWVzOiBUdXBsZVtzdHIsIC4uLl0gPSAoImNocjIiLCAiY2hyMTYiKQoKICAgICMgRHJvcCB0aGUgQlJDQTEgbG9jdXMgZnJvbSBDbGluVmFyIHRyYWluaW5nIGRhdGEgc28gdGhlIERNUyBiZW5jaG1hcmsKICAgICMgc3RheXMgYW4gaG9uZXN0IG91dC1vZi1kaXN0cmlidXRpb24gY2hlY2suCiAgICBleGNsdWRlX2JyY2ExX2Zyb21fdHJhaW5pbmc6IGJvb2wgPSBUcnVlCgogICAgY2xhc3Nfd2VpZ2h0aW5nOiBib29sID0gVHJ1ZSAgICMgQ2xpblZhciBpcyBwYXRob2dlbmljLWhlYXZ5IGF0IDIrIHN0YXJzCgoKREVGQVVMVF9UUkFJTl9DT05GSUcgPSBUcmFpbkNvbmZpZygpCgojIC0tLSBEYXRhc2V0IGNvbnN0cnVjdGlvbiAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKCkBkYXRhY2xhc3MKY2xhc3MgQ2xpblZhckNvbmZpZzoKICAgICIiIkZpbHRlcnMgYXBwbGllZCB3aGVuIHR1cm5pbmcgdGhlIENsaW5WYXIgVkNGIGludG8gYSBsYWJlbGxlZCBTTlYgdGFibGUuIiIiCgogICAgbWluX3Jldmlld19zdGFyczogaW50ID0gMiAgICAgICMgIm11bHRpcGxlIHN1Ym1pdHRlcnMsIG5vIGNvbmZsaWN0cyIgb3IgYmV0dGVyCiAgICBtYXhfdmFyaWFudHM6IE9wdGlvbmFsW2ludF0gPSA0MF8wMDAKICAgIGJhbGFuY2VfY2xhc3NlczogYm9vbCA9IFRydWUKICAgIHNlZWQ6IGludCA9IDAKCgpERUZBVUxUX0NMSU5WQVJfQ09ORklHID0gQ2xpblZhckNvbmZpZygpCg==",
"finetune/data.py": "IiIiTGFiZWxsZWQgU05WIGRhdGFzZXRzOiBDbGluVmFyIGZvciB0cmFpbmluZywgQlJDQTEgRE1TIGFzIGEgaGVsZC1vdXQgYmVuY2htYXJrLgoKQm90aCBzb3VyY2VzIGFyZSBub3JtYWxpc2VkIG9udG8gb25lIHNjaGVtYSBzbyBkb3duc3RyZWFtIHN0YWdlcyBuZXZlciBuZWVkIHRvCmtub3cgd2hlcmUgYSB2YXJpYW50IGNhbWUgZnJvbToKCiAgICB2YXJpYW50X2lkLCBzb3VyY2UsIGFzc2VtYmx5LCBjaHJvbSwgcG9zLCByZWYsIGFsdCwgbGFiZWwsCiAgICBnZW5lLCByZXZpZXdfc3RhcnMsIGNsbnNpZywgc3BsaXQKCmBgbGFiZWxgYCBpcyAxIGZvciBwYXRob2dlbmljIC8gbG9zcy1vZi1mdW5jdGlvbiBhbmQgMCBmb3IgYmVuaWduIC8gZnVuY3Rpb25hbC4KYGBwb3NgYCBpcyAxLWJhc2VkLCBtYXRjaGluZyBib3RoIHRoZSBWQ0YgYW5kIHRoZSBVQ1NDIGNvbnZlbnRpb24uCiIiIgoKaW1wb3J0IG9zCmZyb20gdHlwaW5nIGltcG9ydCBEaWN0LCBMaXN0LCBPcHRpb25hbCwgVHVwbGUKCmZyb20gLiBpbXBvcnQgY29uZmlnCgojIC0tLSBDbGluVmFyIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKIyBFeHBsaWNpdCBhbGxvd2xpc3RzIHJhdGhlciB0aGFuIHN1YnN0cmluZyB0ZXN0czogIkNvbmZsaWN0aW5nX2NsYXNzaWZpY2F0aW9ucwojIF9vZl9wYXRob2dlbmljaXR5IiBjb250YWlucyAicGF0aG9nZW5pYyIgYnV0IGNhcnJpZXMgbm8gdXNhYmxlIGxhYmVsLgpQQVRIT0dFTklDX0NMTlNJRyA9IHsKICAgICJwYXRob2dlbmljIiwKICAgICJsaWtlbHlfcGF0aG9nZW5pYyIsCiAgICAicGF0aG9nZW5pYy9saWtlbHlfcGF0aG9nZW5pYyIsCn0KQkVOSUdOX0NMTlNJRyA9IHsKICAgICJiZW5pZ24iLAogICAgImxpa2VseV9iZW5pZ24iLAogICAgImJlbmlnbi9saWtlbHlfYmVuaWduIiwKfQoKUkVWSUVXX1NUQVRVU19TVEFSUyA9IHsKICAgICJwcmFjdGljZV9ndWlkZWxpbmUiOiA0LAogICAgInJldmlld2VkX2J5X2V4cGVydF9wYW5lbCI6IDMsCiAgICAiY3JpdGVyaWFfcHJvdmlkZWQsX211bHRpcGxlX3N1Ym1pdHRlcnMsX25vX2NvbmZsaWN0cyI6IDIsCiAgICAiY3JpdGVyaWFfcHJvdmlkZWQsX3NpbmdsZV9zdWJtaXR0ZXIiOiAxLAogICAgImNyaXRlcmlhX3Byb3ZpZGVkLF9jb25mbGljdGluZ19jbGFzc2lmaWNhdGlvbnMiOiAxLAogICAgImNyaXRlcmlhX3Byb3ZpZGVkLF9jb25mbGljdGluZ19pbnRlcnByZXRhdGlvbnMiOiAxLAp9CgpfQkFTRVMgPSBmcm96ZW5zZXQoIkFDR1QiKQoKCmRlZiBfcGFyc2VfaW5mbyhpbmZvOiBzdHIpIC0+IERpY3Rbc3RyLCBzdHJdOgogICAgZmllbGRzID0ge30KICAgIGZvciBlbnRyeSBpbiBpbmZvLnNwbGl0KCI7Iik6CiAgICAgICAga2V5LCBzZXAsIHZhbHVlID0gZW50cnkucGFydGl0aW9uKCI9IikKICAgICAgICBmaWVsZHNba2V5XSA9IHZhbHVlIGlmIHNlcCBlbHNlICJ0cnVlIgogICAgcmV0dXJuIGZpZWxkcwoKCmRlZiBfbm9ybWFsaXNlX2NsbnNpZyhyYXc6IHN0cikgLT4gc3RyOgogICAgIiIiUmVkdWNlIGEgQ0xOU0lHIHZhbHVlIHRvIGEgY29tcGFyYWJsZSBiYXNlIGNsYXNzaWZpY2F0aW9uLgoKICAgIENsaW5WYXIgYXBwZW5kcyBxdWFsaWZpZXJzIHN1Y2ggYXMgYGB8b3RoZXJgYCBhbmQgYGAsX2xvd19wZW5ldHJhbmNlYGA7CiAgICB0aG9zZSBtb2RpZnkgdGhlIGFzc2VydGlvbiBidXQgbm90IHRoZSBwYXRob2dlbmljL2JlbmlnbiBjYWxsLgogICAgIiIiCiAgICBiYXNlID0gcmF3LnNwbGl0KCJ8IilbMF0uc3RyaXAoKS5sb3dlcigpCiAgICBmb3Igc3VmZml4IGluICgiLF9sb3dfcGVuZXRyYW5jZSIsICIsX2Fzc29jaWF0aW9uIiwgIixfcmlza19mYWN0b3IiKToKICAgICAgICBpZiBiYXNlLmVuZHN3aXRoKHN1ZmZpeCk6CiAgICAgICAgICAgIGJhc2UgPSBiYXNlWzogLWxlbihzdWZmaXgpXQogICAgcmV0dXJuIGJhc2UKCgpkZWYgX2dlbmVfc3ltYm9sKGdlbmVpbmZvOiBzdHIpIC0+IHN0cjoKICAgIGlmIG5vdCBnZW5laW5mbzoKICAgICAgICByZXR1cm4gIiIKICAgIHJldHVybiBnZW5laW5mby5zcGxpdCgifCIpWzBdLnNwbGl0KCI6IilbMF0KCgpkZWYgZG93bmxvYWRfY2xpbnZhcihmb3JjZTogYm9vbCA9IEZhbHNlKSAtPiBzdHI6CiAgICAiIiJGZXRjaCB0aGUgQ2xpblZhciBHUkNoMzggVkNGIG9udG8gdGhlIGRhdGEgdm9sdW1lLiIiIgogICAgaW1wb3J0IHNodXRpbAoKICAgIGltcG9ydCByZXF1ZXN0cwoKICAgIG9zLm1ha2VkaXJzKGNvbmZpZy5EQVRBU0VUU19ESVIsIGV4aXN0X29rPVRydWUpCiAgICBwYXRoID0gY29uZmlnLkNMSU5WQVJfVkNGX0daCgogICAgaWYgb3MucGF0aC5leGlzdHMocGF0aCkgYW5kIG5vdCBmb3JjZToKICAgICAgICBwcmludChmIkNsaW5WYXIgVkNGIGFscmVhZHkgcHJlc2VudCBhdCB7cGF0aH0iKQogICAgICAgIHJldHVybiBwYXRoCgogICAgcHJpbnQoZiJEb3dubG9hZGluZyBDbGluVmFyIGZyb20ge2NvbmZpZy5DTElOVkFSX1ZDRl9VUkx9IC4uLiIpCiAgICB0bXAgPSBwYXRoICsgIi5wYXJ0IgogICAgd2l0aCByZXF1ZXN0cy5nZXQoY29uZmlnLkNMSU5WQVJfVkNGX1VSTCwgc3RyZWFtPVRydWUsIHRpbWVvdXQ9NjAwKSBhcyByZXNwb25zZToKICAgICAgICByZXNwb25zZS5yYWlzZV9mb3Jfc3RhdHVzKCkKICAgICAgICB3aXRoIG9wZW4odG1wLCAid2IiKSBhcyBoYW5kbGU6CiAgICAgICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihyZXNwb25zZS5yYXcsIGhhbmRsZSwgbGVuZ3RoPTggKiAxMDI0ICogMTAyNCkKICAgIG9zLnJlcGxhY2UodG1wLCBwYXRoKQogICAgcHJpbnQoZiJTYXZlZCB7b3MucGF0aC5nZXRzaXplKHBhdGgpIC8gMWU2Oi4wZn0gTUIgdG8ge3BhdGh9IikKICAgIHJldHVybiBwYXRoCgoKZGVmIHBhcnNlX2NsaW52YXJfdmNmKAogICAgdmNmX3BhdGg6IHN0ciwKICAgIG1pbl9yZXZpZXdfc3RhcnM6IGludCA9IDIsCikgLT4gInBkLkRhdGFGcmFtZSI6CiAgICAiIiJUdXJuIHRoZSBDbGluVmFyIFZDRiBpbnRvIGxhYmVsbGVkIHNpbmdsZS1udWNsZW90aWRlIHZhcmlhbnRzLgoKICAgIEtlZXBzIG9ubHkgU05WcyB3aXRoIGFuIHVuYW1iaWd1b3VzIHBhdGhvZ2VuaWMgb3IgYmVuaWduIGFzc2VydGlvbiBiYWNrZWQgYnkKICAgIGF0IGxlYXN0IGBgbWluX3Jldmlld19zdGFyc2BgIHJldmlldyBzdGFycy4gUmVwb3J0cyB3aHkgcmVjb3JkcyB3ZXJlIGRyb3BwZWQsCiAgICBiZWNhdXNlIGEgc2lsZW50IGZpbHRlciBoZXJlIGlzIHRoZSBlYXNpZXN0IHdheSB0byBidWlsZCBhIGJpYXNlZCBkYXRhc2V0LgogICAgIiIiCiAgICBpbXBvcnQgZ3ppcAoKICAgIGltcG9ydCBwYW5kYXMgYXMgcGQKCiAgICByb3dzOiBMaXN0W2RpY3RdID0gW10KICAgIGRyb3BwZWQgPSB7CiAgICAgICAgIm5vdF9zbnYiOiAwLAogICAgICAgICJtdWx0aWFsbGVsaWMiOiAwLAogICAgICAgICJ1bnVzYWJsZV9jbG5zaWciOiAwLAogICAgICAgICJiZWxvd19zdGFyX3RocmVzaG9sZCI6IDAsCiAgICB9CgogICAgd2l0aCBnemlwLm9wZW4odmNmX3BhdGgsICJydCIpIGFzIGhhbmRsZToKICAgICAgICBmb3IgbGluZSBpbiBoYW5kbGU6CiAgICAgICAgICAgIGlmIGxpbmUuc3RhcnRzd2l0aCgiIyIpOgogICAgICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgICAgIHBhcnRzID0gbGluZS5yc3RyaXAoIlxuIikuc3BsaXQoIlx0IikKICAgICAgICAgICAgaWYgbGVuKHBhcnRzKSA8IDg6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBjaHJvbSwgcG9zLCB2aWQsIHJlZiwgYWx0LCBfcXVhbCwgX2ZpbHQsIGluZm8gPSBwYXJ0c1s6OF0KCiAgICAgICAgICAgIGlmICIsIiBpbiBhbHQ6CiAgICAgICAgICAgICAgICBkcm9wcGVkWyJtdWx0aWFsbGVsaWMiXSArPSAxCiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBpZiBsZW4ocmVmKSAhPSAxIG9yIGxlbihhbHQpICE9IDEgb3IgcmVmIG5vdCBpbiBfQkFTRVMgb3IgYWx0IG5vdCBpbiBfQkFTRVM6CiAgICAgICAgICAgICAgICBkcm9wcGVkWyJub3Rfc252Il0gKz0gMQogICAgICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgICAgIGZpZWxkcyA9IF9wYXJzZV9pbmZvKGluZm8pCiAgICAgICAgICAgIGNsbnNpZyA9IF9ub3JtYWxpc2VfY2xuc2lnKGZpZWxkcy5nZXQoIkNMTlNJRyIsICIiKSkKICAgICAgICAgICAgaWYgY2xuc2lnIGluIFBBVEhPR0VOSUNfQ0xOU0lHOgogICAgICAgICAgICAgICAgbGFiZWwgPSAxCiAgICAgICAgICAgIGVsaWYgY2xuc2lnIGluIEJFTklHTl9DTE5TSUc6CiAgICAgICAgICAgICAgICBsYWJlbCA9IDAKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIGRyb3BwZWRbInVudXNhYmxlX2NsbnNpZyJdICs9IDEKICAgICAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgICAgICBzdGFycyA9IFJFVklFV19TVEFUVVNfU1RBUlMuZ2V0KGZpZWxkcy5nZXQoIkNMTlJFVlNUQVQiLCAiIiksIDApCiAgICAgICAgICAgIGlmIHN0YXJzIDwgbWluX3Jldmlld19zdGFyczoKICAgICAgICAgICAgICAgIGRyb3BwZWRbImJlbG93X3N0YXJfdGhyZXNob2xkIl0gKz0gMQogICAgICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICJ2YXJpYW50X2lkIjogZiJjbGludmFyOnt2aWR9IiwKICAgICAgICAgICAgICAgICAgICAic291cmNlIjogImNsaW52YXIiLAogICAgICAgICAgICAgICAgICAgICJhc3NlbWJseSI6ICJoZzM4IiwKICAgICAgICAgICAgICAgICAgICAjIENsaW5WYXIgd3JpdGVzIGJhcmUgY29udGlnIG5hbWVzOyBub3JtYWxpc2UgdG8gVUNTQyBzdHlsZS4KICAgICAgICAgICAgICAgICAgICAiY2hyb20iOiBjaHJvbSBpZiBjaHJvbS5zdGFydHN3aXRoKCJjaHIiKSBlbHNlIGYiY2hye2Nocm9tfSIsCiAgICAgICAgICAgICAgICAgICAgInBvcyI6IGludChwb3MpLAogICAgICAgICAgICAgICAgICAgICJyZWYiOiByZWYsCiAgICAgICAgICAgICAgICAgICAgImFsdCI6IGFsdCwKICAgICAgICAgICAgICAgICAgICAibGFiZWwiOiBsYWJlbCwKICAgICAgICAgICAgICAgICAgICAiZ2VuZSI6IF9nZW5lX3N5bWJvbChmaWVsZHMuZ2V0KCJHRU5FSU5GTyIsICIiKSksCiAgICAgICAgICAgICAgICAgICAgInJldmlld19zdGFycyI6IHN0YXJzLAogICAgICAgICAgICAgICAgICAgICJjbG5zaWciOiBjbG5zaWcsCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkKCiAgICBmcmFtZSA9IHBkLkRhdGFGcmFtZShyb3dzKQogICAgcHJpbnQoZiJDbGluVmFyOiBrZXB0IHtsZW4oZnJhbWUpfSBTTlZzIGF0ID49e21pbl9yZXZpZXdfc3RhcnN9IHJldmlldyBzdGFycyIpCiAgICBwcmludChmIiAgZHJvcHBlZDoge2Ryb3BwZWR9IikKICAgIGlmIGxlbihmcmFtZSk6CiAgICAgICAgY291bnRzID0gZnJhbWVbImxhYmVsIl0udmFsdWVfY291bnRzKCkudG9fZGljdCgpCiAgICAgICAgcHJpbnQoZiIgIGJlbmlnbj17Y291bnRzLmdldCgwLCAwKX0gcGF0aG9nZW5pYz17Y291bnRzLmdldCgxLCAwKX0iKQogICAgcmV0dXJuIGZyYW1lCgoKZGVmIGJ1aWxkX2NsaW52YXJfZGF0YXNldCgKICAgIGNsaW52YXJfY29uZmlnOiBPcHRpb25hbFtjb25maWcuQ2xpblZhckNvbmZpZ10gPSBOb25lLAogICAgZm9yY2VfZG93bmxvYWQ6IGJvb2wgPSBGYWxzZSwKKSAtPiAicGQuRGF0YUZyYW1lIjoKICAgICIiIkRvd25sb2FkLCBwYXJzZSwgZmlsdGVyIGFuZCBzdWJzYW1wbGUgQ2xpblZhciBpbnRvIHRoZSBzaGFyZWQgc2NoZW1hLiIiIgogICAgY2ZnID0gY2xpbnZhcl9jb25maWcgb3IgY29uZmlnLkRFRkFVTFRfQ0xJTlZBUl9DT05GSUcKICAgIGZyYW1lID0gcGFyc2VfY2xpbnZhcl92Y2YoCiAgICAgICAgZG93bmxvYWRfY2xpbnZhcihmb3JjZT1mb3JjZV9kb3dubG9hZCksCiAgICAgICAgbWluX3Jldmlld19zdGFycz1jZmcubWluX3Jldmlld19zdGFycywKICAgICkKICAgIGlmIGZyYW1lLmVtcHR5OgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTm8gdXNhYmxlIENsaW5WYXIgcmVjb3JkcyBhZnRlciBmaWx0ZXJpbmciKQoKICAgIGlmIGNmZy5iYWxhbmNlX2NsYXNzZXM6CiAgICAgICAgc21hbGxlc3QgPSBpbnQoZnJhbWVbImxhYmVsIl0udmFsdWVfY291bnRzKCkubWluKCkpCiAgICAgICAgZnJhbWUgPSBfc2FtcGxlX3Blcl9jbGFzcyhmcmFtZSwgc21hbGxlc3QsIGNmZy5zZWVkKQogICAgICAgIHByaW50KGYiQmFsYW5jZWQgdG8ge3NtYWxsZXN0fSBwZXIgY2xhc3MgKHtsZW4oZnJhbWUpfSB0b3RhbCkiKQoKICAgIGlmIGNmZy5tYXhfdmFyaWFudHMgaXMgbm90IE5vbmUgYW5kIGxlbihmcmFtZSkgPiBjZmcubWF4X3ZhcmlhbnRzOgogICAgICAgICMgU2FtcGxlIHdpdGhpbiBjbGFzcyBzbyB0aGUgY2FwIGRvZXMgbm90IHJlaW50cm9kdWNlIGltYmFsYW5jZS4KICAgICAgICBwZXJfY2xhc3MgPSBjZmcubWF4X3ZhcmlhbnRzIC8vIGludChmcmFtZVsibGFiZWwiXS5udW5pcXVlKCkpCiAgICAgICAgZnJhbWUgPSBfc2FtcGxlX3Blcl9jbGFzcyhmcmFtZSwgcGVyX2NsYXNzLCBjZmcuc2VlZCkKICAgICAgICBwcmludChmIkNhcHBlZCB0byB7bGVuKGZyYW1lKX0gdmFyaWFudHMgKG1heF92YXJpYW50cz17Y2ZnLm1heF92YXJpYW50c30pIikKCiAgICByZXR1cm4gZnJhbWUuc2FtcGxlKGZyYWM9MS4wLCByYW5kb21fc3RhdGU9Y2ZnLnNlZWQpLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCgpkZWYgX3NhbXBsZV9wZXJfY2xhc3MoZnJhbWU6ICJwZC5EYXRhRnJhbWUiLCBuOiBpbnQsIHNlZWQ6IGludCkgLT4gInBkLkRhdGFGcmFtZSI6CiAgICAiIiJUYWtlIHVwIHRvIGBgbmBgIHJvd3Mgb2YgZWFjaCBsYWJlbCwgcHJlc2VydmluZyB0aGUgc2hhcmVkIHNjaGVtYS4iIiIKICAgIGltcG9ydCBwYW5kYXMgYXMgcGQKCiAgICBwYXJ0cyA9IFsKICAgICAgICBncm91cC5zYW1wbGUobj1taW4obGVuKGdyb3VwKSwgbiksIHJhbmRvbV9zdGF0ZT1zZWVkKQogICAgICAgIGZvciBfLCBncm91cCBpbiBmcmFtZS5ncm91cGJ5KCJsYWJlbCIpCiAgICBdCiAgICByZXR1cm4gcGQuY29uY2F0KHBhcnRzLCBpZ25vcmVfaW5kZXg9VHJ1ZSkKCgojIC0tLSBCUkNBMSBzYXR1cmF0aW9uIG11dGFnZW5lc2lzIChGaW5kbGF5IGV0IGFsLiAyMDE4KSAtLS0tLS0tLS0tLS0tLS0tLS0tLQoKCmRlZiBidWlsZF9icmNhMV9kYXRhc2V0KHhsc3hfcGF0aDogT3B0aW9uYWxbc3RyXSA9IE5vbmUpIC0+ICJwZC5EYXRhRnJhbWUiOgogICAgIiIiTG9hZCB0aGUgQlJDQTEgU0dFIHRhYmxlIHVzZWQgYnkgdGhlIEV2bzIgcGFwZXIncyB6ZXJvLXNob3QgYmVuY2htYXJrLgoKICAgIENvb3JkaW5hdGVzIGFyZSBHUkNoMzcsIHdoaWNoIGlzIHdoeSB0aGlzIGRhdGFzZXQgaXMgc2NvcmVkIGFnYWluc3QgdGhlCiAgICB2ZW5kb3JlZCBHUkNoMzcgY2hyMTcgRkFTVEEgaW5zdGVhZCBvZiBoZzM4LgogICAgIiIiCiAgICBpbXBvcnQgcGFuZGFzIGFzIHBkCgogICAgcGF0aCA9IHhsc3hfcGF0aCBvciBjb25maWcuQlJDQTFfRE1TX1hMU1gKICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhwYXRoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICAgICAgZiJ7cGF0aH0gbm90IGZvdW5kLiBJdCBzaGlwcyB3aXRoIHRoZSBldm8yIHJlcG8gY2xvbmVkIGludG8gdGhlIGltYWdlLiIKICAgICAgICApCgogICAgZnJhbWUgPSBwZC5yZWFkX2V4Y2VsKHBhdGgsIGhlYWRlcj0yKQogICAgZnJhbWUgPSBmcmFtZVsKICAgICAgICBbCiAgICAgICAgICAgICJjaHJvbW9zb21lIiwKICAgICAgICAgICAgInBvc2l0aW9uIChoZzE5KSIsCiAgICAgICAgICAgICJyZWZlcmVuY2UiLAogICAgICAgICAgICAiYWx0IiwKICAgICAgICAgICAgImZ1bmN0aW9uLnNjb3JlLm1lYW4iLAogICAgICAgICAgICAiZnVuYy5jbGFzcyIsCiAgICAgICAgXQogICAgXS5yZW5hbWUoCiAgICAgICAgY29sdW1ucz17CiAgICAgICAgICAgICJjaHJvbW9zb21lIjogImNocm9tIiwKICAgICAgICAgICAgInBvc2l0aW9uIChoZzE5KSI6ICJwb3MiLAogICAgICAgICAgICAicmVmZXJlbmNlIjogInJlZiIsCiAgICAgICAgICAgICJhbHQiOiAiYWx0IiwKICAgICAgICAgICAgImZ1bmN0aW9uLnNjb3JlLm1lYW4iOiAiZnVuY3Rpb25fc2NvcmUiLAogICAgICAgICAgICAiZnVuYy5jbGFzcyI6ICJmdW5jX2NsYXNzIiwKICAgICAgICB9CiAgICApCgogICAgIyBDb2xsYXBzZSB0byB0aGUgdHdvLWNsYXNzIHByb2JsZW0gdXNlZCBpbiB0aGUgcGFwZXI6IGludGVybWVkaWF0ZSB2YXJpYW50cwogICAgIyBhcmUgZ3JvdXBlZCB3aXRoIGZ1bmN0aW9uYWwgb25lcy4KICAgIGZyYW1lWyJmdW5jX2NsYXNzIl0gPSBmcmFtZVsiZnVuY19jbGFzcyJdLnJlcGxhY2UoWyJGVU5DIiwgIklOVCJdLCAiRlVOQy9JTlQiKQoKICAgIGZyYW1lID0gZnJhbWVbZnJhbWVbInJlZiJdLmlzaW4oX0JBU0VTKSAmIGZyYW1lWyJhbHQiXS5pc2luKF9CQVNFUyldLmNvcHkoKQogICAgZnJhbWVbImNocm9tIl0gPSBmcmFtZVsiY2hyb20iXS5hc3R5cGUoc3RyKS5hcHBseSgKICAgICAgICBsYW1iZGEgYzogYyBpZiBjLnN0YXJ0c3dpdGgoImNociIpIGVsc2UgZiJjaHJ7Y30iCiAgICApCiAgICBmcmFtZVsicG9zIl0gPSBmcmFtZVsicG9zIl0uYXN0eXBlKGludCkKICAgIGZyYW1lWyJsYWJlbCJdID0gKGZyYW1lWyJmdW5jX2NsYXNzIl0gPT0gIkxPRiIpLmFzdHlwZShpbnQpCiAgICBmcmFtZVsic291cmNlIl0gPSAiYnJjYTFfZG1zIgogICAgZnJhbWVbImFzc2VtYmx5Il0gPSAiaGcxOSIKICAgIGZyYW1lWyJnZW5lIl0gPSAiQlJDQTEiCiAgICBmcmFtZVsicmV2aWV3X3N0YXJzIl0gPSAtMQogICAgZnJhbWVbImNsbnNpZyJdID0gZnJhbWVbImZ1bmNfY2xhc3MiXQogICAgZnJhbWVbInZhcmlhbnRfaWQiXSA9ICgKICAgICAgICAiYnJjYTE6IiArIGZyYW1lWyJjaHJvbSJdICsgIjoiICsgZnJhbWVbInBvcyJdLmFzdHlwZShzdHIpCiAgICAgICAgKyAiOiIgKyBmcmFtZVsicmVmIl0gKyAiPiIgKyBmcmFtZVsiYWx0Il0KICAgICkKCiAgICBwcmludChmIkJSQ0ExIERNUzoge2xlbihmcmFtZSl9IFNOVnMiKQogICAgcHJpbnQoZiIgIHtmcmFtZVsnZnVuY19jbGFzcyddLnZhbHVlX2NvdW50cygpLnRvX2RpY3QoKX0iKQogICAgcmV0dXJuIGZyYW1lLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCgojIC0tLSBTcGxpdHMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKCmRlZiBfaW5fYnJjYTFfbG9jdXMoZnJhbWU6ICJwZC5EYXRhRnJhbWUiKSAtPiAicGQuU2VyaWVzIjoKICAgIGltcG9ydCBwYW5kYXMgYXMgcGQKCiAgICBtYXNrID0gcGQuU2VyaWVzKEZhbHNlLCBpbmRleD1mcmFtZS5pbmRleCkKICAgIGZvciBhc3NlbWJseSwgKGNocm9tLCBzdGFydCwgZW5kKSBpbiBjb25maWcuQlJDQTFfTE9DVVMuaXRlbXMoKToKICAgICAgICBtYXNrIHw9ICgKICAgICAgICAgICAgKGZyYW1lWyJhc3NlbWJseSJdID09IGFzc2VtYmx5KQogICAgICAgICAgICAmIChmcmFtZVsiY2hyb20iXSA9PSBjaHJvbSkKICAgICAgICAgICAgJiAoZnJhbWVbInBvcyJdID49IHN0YXJ0KQogICAgICAgICAgICAmIChmcmFtZVsicG9zIl0gPD0gZW5kKQogICAgICAgICkKICAgIHJldHVybiBtYXNrCgoKZGVmIGFzc2lnbl9zcGxpdHMoCiAgICBjbGludmFyOiAicGQuRGF0YUZyYW1lIiwKICAgIGJyY2ExOiBPcHRpb25hbFsicGQuRGF0YUZyYW1lIl0gPSBOb25lLAogICAgdHJhaW5fY29uZmlnOiBPcHRpb25hbFtjb25maWcuVHJhaW5Db25maWddID0gTm9uZSwKKSAtPiAicGQuRGF0YUZyYW1lIjoKICAgICIiIkNvbWJpbmUgdGhlIHNvdXJjZXMgYW5kIGFzc2lnbiB0cmFpbiAvIHZhbCAvIHRlc3QgLyBiZW5jaG1hcmsgc3BsaXRzLgoKICAgIFNwbGl0cyBhcmUgY2hyb21vc29tZS1kaXNqb2ludCByYXRoZXIgdGhhbiByYW5kb206IG5lYXJieSB2YXJpYW50cyBzaGFyZQogICAgc2VxdWVuY2UgY29udGV4dCB3aXRoaW4gYW4gOCBrYiB3aW5kb3csIHNvIGEgcmFuZG9tIHNwbGl0IHdvdWxkIGxlYWsKICAgIG5lYXItZHVwbGljYXRlIHdpbmRvd3MgYWNyb3NzIHRoZSBib3VuZGFyeSBhbmQgaW5mbGF0ZSB2YWxpZGF0aW9uIHNjb3Jlcy4KICAgICIiIgogICAgaW1wb3J0IHBhbmRhcyBhcyBwZAoKICAgIGNmZyA9IHRyYWluX2NvbmZpZyBvciBjb25maWcuREVGQVVMVF9UUkFJTl9DT05GSUcKICAgIGZyYW1lID0gY2xpbnZhci5jb3B5KCkKCiAgICBpZiBjZmcuZXhjbHVkZV9icmNhMV9mcm9tX3RyYWluaW5nOgogICAgICAgIGluX2xvY3VzID0gX2luX2JyY2ExX2xvY3VzKGZyYW1lKQogICAgICAgIGlmIGluX2xvY3VzLmFueSgpOgogICAgICAgICAgICBwcmludCgKICAgICAgICAgICAgICAgIGYiRXhjbHVkaW5nIHtpbnQoaW5fbG9jdXMuc3VtKCkpfSBDbGluVmFyIHZhcmlhbnRzIGluIHRoZSBCUkNBMSAiCiAgICAgICAgICAgICAgICAibG9jdXMgdG8ga2VlcCB0aGUgRE1TIGJlbmNobWFyayBvdXQgb2YgZGlzdHJpYnV0aW9uIgogICAgICAgICAgICApCiAgICAgICAgICAgIGZyYW1lID0gZnJhbWVbfmluX2xvY3VzXS5jb3B5KCkKCiAgICB2YWwgPSBzZXQoY2ZnLnZhbF9jaHJvbW9zb21lcykKICAgIHRlc3QgPSBzZXQoY2ZnLnRlc3RfY2hyb21vc29tZXMpCiAgICBvdmVybGFwID0gdmFsICYgdGVzdAogICAgaWYgb3ZlcmxhcDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVmFsaWRhdGlvbiBhbmQgdGVzdCBjaHJvbW9zb21lcyBvdmVybGFwOiB7c29ydGVkKG92ZXJsYXApfSIpCgogICAgZnJhbWVbInNwbGl0Il0gPSAidHJhaW4iCiAgICBmcmFtZS5sb2NbZnJhbWVbImNocm9tIl0uaXNpbih2YWwpLCAic3BsaXQiXSA9ICJ2YWwiCiAgICBmcmFtZS5sb2NbZnJhbWVbImNocm9tIl0uaXNpbih0ZXN0KSwgInNwbGl0Il0gPSAidGVzdCIKCiAgICBmcmFtZXMgPSBbZnJhbWVdCiAgICBpZiBicmNhMSBpcyBub3QgTm9uZSBhbmQgbGVuKGJyY2ExKToKICAgICAgICBiZW5jaG1hcmsgPSBicmNhMS5jb3B5KCkKICAgICAgICBiZW5jaG1hcmtbInNwbGl0Il0gPSAiYmVuY2htYXJrIgogICAgICAgIGZyYW1lcy5hcHBlbmQoYmVuY2htYXJrKQoKICAgIGNvbWJpbmVkID0gcGQuY29uY2F0KGZyYW1lcywgaWdub3JlX2luZGV4PVRydWUsIHNvcnQ9RmFsc2UpCgogICAgcHJpbnQoIlNwbGl0IHNpemVzOiIpCiAgICBmb3Igc3BsaXQsIGdyb3VwIGluIGNvbWJpbmVkLmdyb3VwYnkoInNwbGl0Iik6CiAgICAgICAgcG9zaXRpdmVzID0gaW50KGdyb3VwWyJsYWJlbCJdLnN1bSgpKQogICAgICAgIHByaW50KAogICAgICAgICAgICBmIiAge3NwbGl0OjwxMH0gbj17bGVuKGdyb3VwKTo8N30gIgogICAgICAgICAgICBmInBvc2l0aXZlPXtwb3NpdGl2ZXN9ICh7cG9zaXRpdmVzIC8gbWF4KGxlbihncm91cCksIDEpOi4xJX0pIgogICAgICAgICkKCiAgICBpZiBub3QgKGNvbWJpbmVkWyJzcGxpdCJdID09ICJ2YWwiKS5hbnkoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICBmIlZhbGlkYXRpb24gc3BsaXQgaXMgZW1wdHkuIENocm9tb3NvbWVzIHtzb3J0ZWQodmFsKX0gbWF0Y2hlZCBubyAiCiAgICAgICAgICAgICJ2YXJpYW50czsgcGljayBjaHJvbW9zb21lcyBwcmVzZW50IGluIHRoZSBDbGluVmFyIHRhYmxlLiIKICAgICAgICApCiAgICByZXR1cm4gY29tYmluZWQKCgojIC0tLSBQZXJzaXN0ZW5jZSAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKQ09MVU1OUyA9IFsKICAgICJ2YXJpYW50X2lkIiwKICAgICJzb3VyY2UiLAogICAgImFzc2VtYmx5IiwKICAgICJjaHJvbSIsCiAgICAicG9zIiwKICAgICJyZWYiLAogICAgImFsdCIsCiAgICAibGFiZWwiLAogICAgImdlbmUiLAogICAgInJldmlld19zdGFycyIsCiAgICAiY2xuc2lnIiwKICAgICJzcGxpdCIsCl0KCgpkZWYgZGF0YXNldF9wYXRoKG5hbWU6IHN0ciA9ICJ2YXJpYW50cyIpIC0+IHN0cjoKICAgIHJldHVybiBmIntjb25maWcuREFUQVNFVFNfRElSfS97bmFtZX0ucGFycXVldCIKCgpkZWYgc2F2ZV9kYXRhc2V0KGZyYW1lOiAicGQuRGF0YUZyYW1lIiwgbmFtZTogc3RyID0gInZhcmlhbnRzIikgLT4gc3RyOgogICAgb3MubWFrZWRpcnMoY29uZmlnLkRBVEFTRVRTX0RJUiwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBhdGggPSBkYXRhc2V0X3BhdGgobmFtZSkKICAgIGtlZXAgPSBbYyBmb3IgYyBpbiBDT0xVTU5TIGlmIGMgaW4gZnJhbWUuY29sdW1uc10KICAgIGV4dHJhID0gW2MgZm9yIGMgaW4gKCJmdW5jdGlvbl9zY29yZSIsICJmdW5jX2NsYXNzIikgaWYgYyBpbiBmcmFtZS5jb2x1bW5zXQogICAgZnJhbWVba2VlcCArIGV4dHJhXS50b19wYXJxdWV0KHBhdGgsIGluZGV4PUZhbHNlKQogICAgcHJpbnQoZiJXcm90ZSB7bGVuKGZyYW1lKX0gdmFyaWFudHMgdG8ge3BhdGh9IikKICAgIHJldHVybiBwYXRoCgoKZGVmIGxvYWRfZGF0YXNldChuYW1lOiBzdHIgPSAidmFyaWFudHMiKSAtPiAicGQuRGF0YUZyYW1lIjoKICAgIGltcG9ydCBwYW5kYXMgYXMgcGQKCiAgICBwYXRoID0gZGF0YXNldF9wYXRoKG5hbWUpCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMocGF0aCk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoCiAgICAgICAgICAgIGYie3BhdGh9IG5vdCBmb3VuZC4gUnVuIHRoZSBgYnVpbGQtZGF0YXNldHNgIHN0YWdlIGZpcnN0LiIKICAgICAgICApCiAgICByZXR1cm4gcGQucmVhZF9wYXJxdWV0KHBhdGgpCg==",
"finetune/sequences.py": "IiIiUmVmZXJlbmNlLWdlbm9tZSBhY2Nlc3MgZm9yIGJ1aWxkaW5nIHZhcmlhbnQgd2luZG93cy4KClRyYWluaW5nIG5lZWRzIHRlbnMgb2YgdGhvdXNhbmRzIG9mIDhrYiB3aW5kb3dzLCB3aGljaCBpcyBmYXIgdG9vIG1hbnkgcm91bmQKdHJpcHMgZm9yIHRoZSBVQ1NDIFJFU1QgQVBJLCBzbyBidWxrIHdvcmsgcmVhZHMgZnJvbSBhIGxvY2FsIEZBU1RBIG9uIHRoZSBNb2RhbAp2b2x1bWUuIFRoZSBSRVNUIHBhdGggaXMga2VwdCBmb3Igb25lLW9mZiBsb29rdXBzIG9uIHRoZSBpbmZlcmVuY2UgZW5kcG9pbnQuCiIiIgoKZnJvbSB0eXBpbmcgaW1wb3J0IERpY3QsIE9wdGlvbmFsLCBUdXBsZQoKZnJvbSAuIGltcG9ydCBjb25maWcKCgpjbGFzcyBSZWZlcmVuY2VHZW5vbWU6CiAgICAiIiJSYW5kb20gYWNjZXNzIHRvIGEgcmVmZXJlbmNlIGFzc2VtYmx5LCBpbmRleGVkIHdpdGggcHlmYWlkeC4KCiAgICBDaHJvbW9zb21lIG5hbWVzIGFyZSBub3JtYWxpc2VkIHRvIHRoZSBVQ1NDIGBgY2hyTmBgIGNvbnZlbnRpb24gb24gbG9va3VwLAogICAgYmVjYXVzZSBDbGluVmFyJ3MgVkNGIHVzZXMgYmFyZSBgYE5gYCB3aGlsZSB0aGUgVUNTQyBGQVNUQXMgdXNlIGBgY2hyTmBgLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGZhc3RhX3BhdGg6IHN0ciwgc29sZV9jb250aWc6IE9wdGlvbmFsW3N0cl0gPSBOb25lKToKICAgICAgICAiIiJPcGVuIGBgZmFzdGFfcGF0aGBgLgoKICAgICAgICBgYHNvbGVfY29udGlnYGAgZGVjbGFyZXMgdGhhdCB0aGlzIEZBU1RBIGhvbGRzIGV4YWN0bHkgb25lIHNlcXVlbmNlIGFuZAogICAgICAgIG5hbWVzIHdoaWNoIGNocm9tb3NvbWUgaXQgaXMuIE5lZWRlZCBmb3IgdGhlIHZlbmRvcmVkIEdSQ2gzNyBjaHIxNwogICAgICAgIGZpbGUsIHdob3NlIHNlcXVlbmNlIGlzIGhlYWRlZCB3aXRoIGFuIE5DQkkgYWNjZXNzaW9uCiAgICAgICAgKGBgTkNfMDAwMDE3LjEwYGApIHJhdGhlciB0aGFuIGBgY2hyMTdgYCwgc28gbm8gYW1vdW50IG9mIGNoci1wcmVmaXgKICAgICAgICBub3JtYWxpc2F0aW9uIHdvdWxkIG1hdGNoIGl0LiBOYW1pbmcgdGhlIGNocm9tb3NvbWUgZXhwbGljaXRseSwgcmF0aGVyCiAgICAgICAgdGhhbiByZXNvbHZpbmcgYW55dGhpbmcgYXQgYWxsIHRvICJ0aGUgb25seSBjb250aWcgcHJlc2VudCIsIGtlZXBzIGEKICAgICAgICBsb29rdXAgZm9yIHNvbWUgKm90aGVyKiBjaHJvbW9zb21lIGFuIGVycm9yIGluc3RlYWQgb2Ygc2lsZW50bHkKICAgICAgICByZXR1cm5pbmcgY2hyMTcgc2VxdWVuY2UuCiAgICAgICAgIiIiCiAgICAgICAgZnJvbSBweWZhaWR4IGltcG9ydCBGYXN0YQoKICAgICAgICBzZWxmLmZhc3RhX3BhdGggPSBmYXN0YV9wYXRoCiAgICAgICAgc2VsZi5fZmFzdGEgPSBGYXN0YShmYXN0YV9wYXRoLCBzZXF1ZW5jZV9hbHdheXNfdXBwZXI9VHJ1ZSwgYXNfcmF3PVRydWUpCiAgICAgICAga2V5cyA9IGxpc3Qoc2VsZi5fZmFzdGEua2V5cygpKQogICAgICAgIHNlbGYuX25hbWVfbWFwID0gc2VsZi5fYnVpbGRfbmFtZV9tYXAoa2V5cykKCiAgICAgICAgaWYgc29sZV9jb250aWcgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGlmIGxlbihrZXlzKSAhPSAxOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgICAgICAgICBmIntmYXN0YV9wYXRofSB3YXMgZGVjbGFyZWQgc2luZ2xlLWNvbnRpZyAoe3NvbGVfY29udGlnIXJ9KSAiCiAgICAgICAgICAgICAgICAgICAgZiJidXQgaG9sZHMge2xlbihrZXlzKX0gc2VxdWVuY2VzIgogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBiYXJlID0gKAogICAgICAgICAgICAgICAgc29sZV9jb250aWdbMzpdIGlmIHNvbGVfY29udGlnLnN0YXJ0c3dpdGgoImNociIpIGVsc2Ugc29sZV9jb250aWcKICAgICAgICAgICAgKQogICAgICAgICAgICBzZWxmLl9uYW1lX21hcC5zZXRkZWZhdWx0KGJhcmUsIGtleXNbMF0pCiAgICAgICAgICAgIHNlbGYuX25hbWVfbWFwLnNldGRlZmF1bHQoZiJjaHJ7YmFyZX0iLCBrZXlzWzBdKQoKICAgIEBzdGF0aWNtZXRob2QKICAgIGRlZiBfYnVpbGRfbmFtZV9tYXAoa2V5cykgLT4gRGljdFtzdHIsIHN0cl06CiAgICAgICAgIiIiTWFwIGJvdGggYGAxYGAgYW5kIGBgY2hyMWBgIHNwZWxsaW5ncyBvbnRvIHdoYXRldmVyIHRoZSBGQVNUQSB1c2VzLiIiIgogICAgICAgIG1hcHBpbmcgPSB7fQogICAgICAgIGZvciBrZXkgaW4ga2V5czoKICAgICAgICAgICAgbWFwcGluZ1trZXldID0ga2V5CiAgICAgICAgICAgIGJhcmUgPSBrZXlbMzpdIGlmIGtleS5zdGFydHN3aXRoKCJjaHIiKSBlbHNlIGtleQogICAgICAgICAgICBtYXBwaW5nLnNldGRlZmF1bHQoYmFyZSwga2V5KQogICAgICAgICAgICBtYXBwaW5nLnNldGRlZmF1bHQoZiJjaHJ7YmFyZX0iLCBrZXkpCiAgICAgICAgICAgICMgQ2xpblZhciB3cml0ZXMgdGhlIG1pdG9jaG9uZHJpYWwgY29udGlnIGFzIE1ULCBVQ1NDIGFzIGNock0uCiAgICAgICAgICAgIGlmIGJhcmUgaW4gKCJNIiwgIk1UIik6CiAgICAgICAgICAgICAgICBtYXBwaW5nLnNldGRlZmF1bHQoIk1UIiwga2V5KQogICAgICAgICAgICAgICAgbWFwcGluZy5zZXRkZWZhdWx0KCJjaHJNVCIsIGtleSkKICAgICAgICByZXR1cm4gbWFwcGluZwoKICAgIGRlZiBjbG9zZShzZWxmKSAtPiBOb25lOgogICAgICAgICIiIlJlbGVhc2UgdGhlIHVuZGVybHlpbmcgRkFTVEEgaGFuZGxlLgoKICAgICAgICBNb2RhbCByZXVzZXMgYSBjb250YWluZXIgYWNyb3NzIG1hcCBpbnB1dHMsIGFuZCBgYFZvbHVtZS5yZWxvYWQoKWBgCiAgICAgICAgcmVmdXNlcyB0byBydW4gd2hpbGUgYW55IGZpbGUgb24gdGhlIHZvbHVtZSBpcyBzdGlsbCBvcGVuLiBweWZhaWR4IGhvbGRzCiAgICAgICAgdGhlIEZBU1RBIG9wZW4gZm9yIHRoZSBsaWZlIG9mIHRoZSBvYmplY3QsIHNvIGEgc2hhcmQgdGhhdCBsZWF2ZXMgaXRzCiAgICAgICAgZ2Vub21lcyBvcGVuIG1ha2VzIHRoZSAqbmV4dCogc2hhcmQgb24gdGhhdCBjb250YWluZXIgZmFpbCBiZWZvcmUgaXQKICAgICAgICBzdGFydHMuCiAgICAgICAgIiIiCiAgICAgICAgZmFzdGEgPSBnZXRhdHRyKHNlbGYsICJfZmFzdGEiLCBOb25lKQogICAgICAgIGlmIGZhc3RhIGlzIG5vdCBOb25lOgogICAgICAgICAgICBmYXN0YS5jbG9zZSgpCiAgICAgICAgICAgIHNlbGYuX2Zhc3RhID0gTm9uZQoKICAgIGRlZiByZXNvbHZlKHNlbGYsIGNocm9tb3NvbWU6IHN0cikgLT4gT3B0aW9uYWxbc3RyXToKICAgICAgICByZXR1cm4gc2VsZi5fbmFtZV9tYXAuZ2V0KGNocm9tb3NvbWUpCgogICAgZGVmIGxlbmd0aChzZWxmLCBjaHJvbW9zb21lOiBzdHIpIC0+IGludDoKICAgICAgICBrZXkgPSBzZWxmLnJlc29sdmUoY2hyb21vc29tZSkKICAgICAgICBpZiBrZXkgaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgS2V5RXJyb3IoZiJDaHJvbW9zb21lIHtjaHJvbW9zb21lIXJ9IG5vdCBpbiB7c2VsZi5mYXN0YV9wYXRofSIpCiAgICAgICAgcmV0dXJuIGxlbihzZWxmLl9mYXN0YVtrZXldKQoKICAgIGRlZiB3aW5kb3coCiAgICAgICAgc2VsZiwKICAgICAgICBjaHJvbW9zb21lOiBzdHIsCiAgICAgICAgcG9zaXRpb246IGludCwKICAgICAgICB3aW5kb3dfc2l6ZTogaW50LAogICAgKSAtPiBUdXBsZVtzdHIsIGludF06CiAgICAgICAgIiIiUmV0dXJuIGEgd2luZG93IGNlbnRyZWQgb24gMS1iYXNlZCBgYHBvc2l0aW9uYGAuCgogICAgICAgIFJldHVybnMgYGAoc2VxdWVuY2UsIHN0YXJ0KWBgIHdoZXJlIGBgc3RhcnRgYCBpcyB0aGUgMC1iYXNlZCBvZmZzZXQgb2YKICAgICAgICB0aGUgZmlyc3QgYmFzZSwgc28gdGhlIHZhcmlhbnQgc2l0cyBhdCBgYHBvc2l0aW9uIC0gMSAtIHN0YXJ0YGAuCiAgICAgICAgIiIiCiAgICAgICAga2V5ID0gc2VsZi5yZXNvbHZlKGNocm9tb3NvbWUpCiAgICAgICAgaWYga2V5IGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIEtleUVycm9yKGYiQ2hyb21vc29tZSB7Y2hyb21vc29tZSFyfSBub3QgaW4ge3NlbGYuZmFzdGFfcGF0aH0iKQoKICAgICAgICBjb250aWcgPSBzZWxmLl9mYXN0YVtrZXldCiAgICAgICAgY29udGlnX2xlbiA9IGxlbihjb250aWcpCiAgICAgICAgcCA9IHBvc2l0aW9uIC0gMQogICAgICAgIGlmIHAgPCAwIG9yIHAgPj0gY29udGlnX2xlbjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgICAgIGYiUG9zaXRpb24ge3Bvc2l0aW9ufSBpcyBvdXRzaWRlIHtjaHJvbW9zb21lfSAobGVuZ3RoIHtjb250aWdfbGVufSkiCiAgICAgICAgICAgICkKCiAgICAgICAgaGFsZiA9IHdpbmRvd19zaXplIC8vIDIKICAgICAgICBzdGFydCA9IG1heCgwLCBwIC0gaGFsZikKICAgICAgICBlbmQgPSBtaW4oY29udGlnX2xlbiwgcCArIGhhbGYpCiAgICAgICAgcmV0dXJuIHN0cihjb250aWdbc3RhcnQ6ZW5kXSksIHN0YXJ0CgoKZGVmIGZldGNoX3dpbmRvd191Y3NjKAogICAgcG9zaXRpb246IGludCwKICAgIGdlbm9tZTogc3RyLAogICAgY2hyb21vc29tZTogc3RyLAogICAgd2luZG93X3NpemU6IGludCA9IDgxOTIsCikgLT4gVHVwbGVbc3RyLCBpbnRdOgogICAgIiIiRmV0Y2ggYSBzaW5nbGUgd2luZG93IGZyb20gdGhlIFVDU0MgUkVTVCBBUEkuCgogICAgVXNlZCBieSB0aGUgaW5mZXJlbmNlIGVuZHBvaW50LCB3aGVyZSBvbmUgSFRUUCBjYWxsIHBlciByZXF1ZXN0IGlzIGZpbmUuCiAgICBSZXR1cm5zIGBgKHNlcXVlbmNlLCBzdGFydClgYCB3aXRoIGEgMC1iYXNlZCBgYHN0YXJ0YGAsIG1hdGNoaW5nCiAgICA6bWV0aDpgUmVmZXJlbmNlR2Vub21lLndpbmRvd2AuCiAgICAiIiIKICAgIGltcG9ydCByZXF1ZXN0cwoKICAgIGhhbGYgPSB3aW5kb3dfc2l6ZSAvLyAyCiAgICBzdGFydCA9IG1heCgwLCBwb3NpdGlvbiAtIDEgLSBoYWxmKQogICAgZW5kID0gcG9zaXRpb24gLSAxICsgaGFsZgoKICAgIGFwaV91cmwgPSAoCiAgICAgICAgZiJodHRwczovL2FwaS5nZW5vbWUudWNzYy5lZHUvZ2V0RGF0YS9zZXF1ZW5jZSIKICAgICAgICBmIj9nZW5vbWU9e2dlbm9tZX07Y2hyb209e2Nocm9tb3NvbWV9O3N0YXJ0PXtzdGFydH07ZW5kPXtlbmR9IgogICAgKQogICAgcmVzcG9uc2UgPSByZXF1ZXN0cy5nZXQoYXBpX3VybCwgdGltZW91dD02MCkKICAgIGlmIHJlc3BvbnNlLnN0YXR1c19jb2RlICE9IDIwMDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgIGYiVUNTQyBBUEkgcmV0dXJuZWQge3Jlc3BvbnNlLnN0YXR1c19jb2RlfSBmb3Ige2Nocm9tb3NvbWV9OntzdGFydH0te2VuZH0iCiAgICAgICAgKQoKICAgIHBheWxvYWQgPSByZXNwb25zZS5qc29uKCkKICAgIGlmICJkbmEiIG5vdCBpbiBwYXlsb2FkOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlVDU0MgQVBJIGVycm9yOiB7cGF5bG9hZC5nZXQoJ2Vycm9yJywgJ3Vua25vd24gZXJyb3InKX0iKQoKICAgIHNlcXVlbmNlID0gcGF5bG9hZFsiZG5hIl0udXBwZXIoKQogICAgaWYgbGVuKHNlcXVlbmNlKSAhPSBlbmQgLSBzdGFydDoKICAgICAgICBwcmludCgKICAgICAgICAgICAgZiJXYXJuaW5nOiBVQ1NDIHJldHVybmVkIHtsZW4oc2VxdWVuY2UpfSBiYXNlcywgZXhwZWN0ZWQge2VuZCAtIHN0YXJ0fSIKICAgICAgICApCiAgICByZXR1cm4gc2VxdWVuY2UsIHN0YXJ0CgoKZGVmIGJ1aWxkX3ZhcmlhbnRfd2luZG93KAogICAgcmVmX3dpbmRvdzogc3RyLAogICAgcmVsYXRpdmVfcG9zaXRpb246IGludCwKICAgIGFsdGVybmF0aXZlOiBzdHIsCiAgICBleHBlY3RlZF9yZWZlcmVuY2U6IE9wdGlvbmFsW3N0cl0gPSBOb25lLAopIC0+IFR1cGxlW3N0ciwgc3RyXToKICAgICIiIlN1YnN0aXR1dGUgYSBzaW5nbGUgYmFzZSwgcmV0dXJuaW5nIGBgKHZhcmlhbnRfd2luZG93LCByZWZlcmVuY2VfYmFzZSlgYC4KCiAgICBJZiBgYGV4cGVjdGVkX3JlZmVyZW5jZWBgIGlzIGdpdmVuIGl0IGlzIGNoZWNrZWQgYWdhaW5zdCB0aGUgYXNzZW1ibHkuIEEKICAgIG1pc21hdGNoIG1lYW5zIHRoZSBjb29yZGluYXRlLCBhc3NlbWJseSBvciBzdHJhbmQgaXMgd3JvbmcsIGFuZCBzaWxlbnRseQogICAgc2NvcmluZyBpdCB3b3VsZCBwb2lzb24gdGhlIHRyYWluaW5nIHNldCwgc28gaXQgcmFpc2VzLgogICAgIiIiCiAgICBpZiBub3QgMCA8PSByZWxhdGl2ZV9wb3NpdGlvbiA8IGxlbihyZWZfd2luZG93KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICBmIlJlbGF0aXZlIHBvc2l0aW9uIHtyZWxhdGl2ZV9wb3NpdGlvbn0gb3V0c2lkZSB3aW5kb3cgb2YgIgogICAgICAgICAgICBmImxlbmd0aCB7bGVuKHJlZl93aW5kb3cpfSIKICAgICAgICApCgogICAgcmVmZXJlbmNlX2Jhc2UgPSByZWZfd2luZG93W3JlbGF0aXZlX3Bvc2l0aW9uXQogICAgaWYgZXhwZWN0ZWRfcmVmZXJlbmNlIGlzIG5vdCBOb25lIGFuZCByZWZlcmVuY2VfYmFzZSAhPSBleHBlY3RlZF9yZWZlcmVuY2UudXBwZXIoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICBmIlJlZmVyZW5jZSBtaXNtYXRjaDogYXNzZW1ibHkgaGFzIHtyZWZlcmVuY2VfYmFzZSFyfSwgIgogICAgICAgICAgICBmInJlY29yZCBjbGFpbXMge2V4cGVjdGVkX3JlZmVyZW5jZSFyfSIKICAgICAgICApCgogICAgdmFyaWFudF93aW5kb3cgPSAoCiAgICAgICAgcmVmX3dpbmRvd1s6cmVsYXRpdmVfcG9zaXRpb25dCiAgICAgICAgKyBhbHRlcm5hdGl2ZS51cHBlcigpCiAgICAgICAgKyByZWZfd2luZG93W3JlbGF0aXZlX3Bvc2l0aW9uICsgMTpdCiAgICApCiAgICByZXR1cm4gdmFyaWFudF93aW5kb3csIHJlZmVyZW5jZV9iYXNlCgoKZGVmIGVuc3VyZV9oZzM4KGZvcmNlOiBib29sID0gRmFsc2UpIC0+IHN0cjoKICAgICIiIkRvd25sb2FkIGFuZCBpbmRleCBoZzM4IG9udG8gdGhlIGRhdGEgdm9sdW1lLiBSZXR1cm5zIHRoZSBGQVNUQSBwYXRoLiIiIgogICAgaW1wb3J0IGd6aXAKICAgIGltcG9ydCBvcwogICAgaW1wb3J0IHNodXRpbAogICAgaW1wb3J0IHRpbWUKCiAgICBpbXBvcnQgcmVxdWVzdHMKCiAgICBvcy5tYWtlZGlycyhjb25maWcuR0VOT01FU19ESVIsIGV4aXN0X29rPVRydWUpCiAgICBmYXN0YSA9IGNvbmZpZy5IRzM4X0ZBU1RBCgogICAgaWYgb3MucGF0aC5leGlzdHMoZmFzdGEpIGFuZCBub3QgZm9yY2U6CiAgICAgICAgcHJpbnQoZiJoZzM4IGFscmVhZHkgcHJlc2VudCBhdCB7ZmFzdGF9IikKICAgIGVsc2U6CiAgICAgICAgdG1wX2d6ID0gZmFzdGEgKyAiLmd6LnBhcnQiCiAgICAgICAgIyBUd28gZmFpbHVyZSBtb2RlcyB0byBzdXJ2aXZlOiBhIG1pZC10cmFuc2ZlciByZXNldCwgd2hlcmUgcmVzdGFydGluZwogICAgICAgICMgd291bGQgdGhyb3cgYXdheSB1cCB0byA5NTAgTUIsIGFuZCBhIGhvc3QgcmVmdXNpbmcgY29ubmVjdGlvbnMKICAgICAgICAjIG91dHJpZ2h0LCB3aGVyZSBvbmx5IGEgZGlmZmVyZW50IG1pcnJvciBoZWxwcy4gU28gcmV0cmllcyBib3RoIHJlc3VtZQogICAgICAgICMgZnJvbSB3aGF0ZXZlciBhbHJlYWR5IGxhbmRlZCBhbmQgcm90YXRlIHRocm91Z2ggdGhlIG1pcnJvciBsaXN0LgogICAgICAgIHVybHMgPSBjb25maWcuSEczOF9VUkxTCiAgICAgICAgYXR0ZW1wdHMgPSAzICogbGVuKHVybHMpCiAgICAgICAgZm9yIGF0dGVtcHQgaW4gcmFuZ2UoMSwgYXR0ZW1wdHMgKyAxKToKICAgICAgICAgICAgdXJsID0gdXJsc1soYXR0ZW1wdCAtIDEpICUgbGVuKHVybHMpXQogICAgICAgICAgICBkb25lID0gb3MucGF0aC5nZXRzaXplKHRtcF9neikgaWYgb3MucGF0aC5leGlzdHModG1wX2d6KSBlbHNlIDAKICAgICAgICAgICAgaGVhZGVycyA9IHsiUmFuZ2UiOiBmImJ5dGVzPXtkb25lfS0ifSBpZiBkb25lIGVsc2Uge30KICAgICAgICAgICAgcHJpbnQoCiAgICAgICAgICAgICAgICBmIkRvd25sb2FkaW5nIGhnMzggZnJvbSB7dXJsfSAofjk1MCBNQikiCiAgICAgICAgICAgICAgICArIChmIiwgcmVzdW1pbmcgYXQge2RvbmUgLyAxZTY6LjBmfSBNQiIgaWYgZG9uZSBlbHNlICIiKQogICAgICAgICAgICAgICAgKyAiIC4uLiIKICAgICAgICAgICAgKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICB3aXRoIHJlcXVlc3RzLmdldCgKICAgICAgICAgICAgICAgICAgICB1cmwsIHN0cmVhbT1UcnVlLCB0aW1lb3V0PTYwMCwgaGVhZGVycz1oZWFkZXJzCiAgICAgICAgICAgICAgICApIGFzIHJlc3BvbnNlOgogICAgICAgICAgICAgICAgICAgIHJlc3BvbnNlLnJhaXNlX2Zvcl9zdGF0dXMoKQogICAgICAgICAgICAgICAgICAgICMgMjA2IG1lYW5zIHRoZSByYW5nZSB3YXMgaG9ub3VyZWQ7IGEgc2VydmVyIHRoYXQgaWdub3JlcyBpdAogICAgICAgICAgICAgICAgICAgICMgcmVwbGllcyAyMDAgYW5kIHJlc3RhcnRzIGF0IGJ5dGUgMCwgd2hlcmUgYXBwZW5kaW5nIHdvdWxkCiAgICAgICAgICAgICAgICAgICAgIyBzaWxlbnRseSBjb3JydXB0IHRoZSBmaWxlLgogICAgICAgICAgICAgICAgICAgIG1vZGUgPSAiYWIiIGlmIHJlc3BvbnNlLnN0YXR1c19jb2RlID09IDIwNiBlbHNlICJ3YiIKICAgICAgICAgICAgICAgICAgICB3aXRoIG9wZW4odG1wX2d6LCBtb2RlKSBhcyBoYW5kbGU6CiAgICAgICAgICAgICAgICAgICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaigKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJlc3BvbnNlLnJhdywgaGFuZGxlLCBsZW5ndGg9OCAqIDEwMjQgKiAxMDI0CiAgICAgICAgICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIGV4Y2VwdCByZXF1ZXN0cy5leGNlcHRpb25zLlJlcXVlc3RFeGNlcHRpb24gYXMgZXhjOgogICAgICAgICAgICAgICAgaWYgYXR0ZW1wdCA9PSBhdHRlbXB0czoKICAgICAgICAgICAgICAgICAgICByYWlzZQogICAgICAgICAgICAgICAgcHJpbnQoZiIgIGF0dGVtcHQge2F0dGVtcHR9L3thdHRlbXB0c30gZmFpbGVkICh7ZXhjfSk7IHJldHJ5aW5nIC4uLiIpCiAgICAgICAgICAgICAgICB0aW1lLnNsZWVwKDUgKiBhdHRlbXB0KQoKICAgICAgICBwcmludCgiRGVjb21wcmVzc2luZyAofjMuMSBHQikgLi4uIikKICAgICAgICB0bXBfZmEgPSBmYXN0YSArICIucGFydCIKICAgICAgICB3aXRoIGd6aXAub3Blbih0bXBfZ3osICJyYiIpIGFzIHNyYywgb3Blbih0bXBfZmEsICJ3YiIpIGFzIGRzdDoKICAgICAgICAgICAgc2h1dGlsLmNvcHlmaWxlb2JqKHNyYywgZHN0LCBsZW5ndGg9OCAqIDEwMjQgKiAxMDI0KQogICAgICAgIG9zLnJlcGxhY2UodG1wX2ZhLCBmYXN0YSkKICAgICAgICBvcy5yZW1vdmUodG1wX2d6KQoKICAgIF9lbnN1cmVfaW5kZXgoZmFzdGEpCiAgICByZXR1cm4gZmFzdGEKCgpkZWYgZW5zdXJlX2hnMTlfY2hyMTcoZm9yY2U6IGJvb2wgPSBGYWxzZSkgLT4gc3RyOgogICAgIiIiRGVjb21wcmVzcyBhbmQgaW5kZXggdGhlIHZlbmRvcmVkIEdSQ2gzNyBjaHIxNyBGQVNUQS4gUmV0dXJucyBpdHMgcGF0aC4iIiIKICAgIGltcG9ydCBnemlwCiAgICBpbXBvcnQgb3MKICAgIGltcG9ydCBzaHV0aWwKCiAgICBvcy5tYWtlZGlycyhjb25maWcuR0VOT01FU19ESVIsIGV4aXN0X29rPVRydWUpCiAgICBmYXN0YSA9IGNvbmZpZy5IRzE5X0NIUjE3X0ZBU1RBCgogICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKGZhc3RhKSBvciBmb3JjZToKICAgICAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoY29uZmlnLkhHMTlfQ0hSMTdfRkFTVEFfR1opOgogICAgICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICAgICAgICAgIGYie2NvbmZpZy5IRzE5X0NIUjE3X0ZBU1RBX0dafSBub3QgZm91bmQuIEl0IHNoaXBzIHdpdGggdGhlIGV2bzIgIgogICAgICAgICAgICAgICAgInJlcG8gdGhhdCB0aGUgaW1hZ2UgY2xvbmVzOyBjaGVjayB0aGUgaW1hZ2UgYnVpbGQuIgogICAgICAgICAgICApCiAgICAgICAgcHJpbnQoZiJEZWNvbXByZXNzaW5nIHtjb25maWcuSEcxOV9DSFIxN19GQVNUQV9HWn0gLi4uIikKICAgICAgICB0bXAgPSBmYXN0YSArICIucGFydCIKICAgICAgICB3aXRoIGd6aXAub3Blbihjb25maWcuSEcxOV9DSFIxN19GQVNUQV9HWiwgInJiIikgYXMgc3JjLCBvcGVuKHRtcCwgIndiIikgYXMgZHN0OgogICAgICAgICAgICBzaHV0aWwuY29weWZpbGVvYmooc3JjLCBkc3QsIGxlbmd0aD04ICogMTAyNCAqIDEwMjQpCiAgICAgICAgb3MucmVwbGFjZSh0bXAsIGZhc3RhKQoKICAgIF9lbnN1cmVfaW5kZXgoZmFzdGEpCiAgICByZXR1cm4gZmFzdGEKCgpkZWYgX2Vuc3VyZV9pbmRleChmYXN0YTogc3RyKSAtPiBOb25lOgogICAgaW1wb3J0IG9zCgogICAgZnJvbSBweWZhaWR4IGltcG9ydCBGYWlkeAoKICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhmYXN0YSArICIuZmFpIik6CiAgICAgICAgcHJpbnQoZiJCdWlsZGluZyAuZmFpIGluZGV4IGZvciB7ZmFzdGF9IC4uLiIpCiAgICAgICAgRmFpZHgoZmFzdGEpCiAgICBwcmludChmIlJlYWR5OiB7ZmFzdGF9IikKCgpkZWYgb3Blbl9nZW5vbWUoYXNzZW1ibHk6IHN0cikgLT4gUmVmZXJlbmNlR2Vub21lOgogICAgIiIiT3BlbiB0aGUgbG9jYWwgRkFTVEEgZm9yIGBgYXNzZW1ibHlgYCwgcHJlcGFyaW5nIGl0IGlmIG5lY2Vzc2FyeS4iIiIKICAgIGlmIGFzc2VtYmx5ID09ICJoZzM4IjoKICAgICAgICByZXR1cm4gUmVmZXJlbmNlR2Vub21lKGVuc3VyZV9oZzM4KCkpCiAgICBpZiBhc3NlbWJseSA9PSAiaGcxOSI6CiAgICAgICAgIyBPbmx5IGNocjE3IGlzIHZlbmRvcmVkOyB0aGUgQlJDQTEgYmVuY2htYXJrIG5ldmVyIGxlYXZlcyB0aGF0IGxvY3VzLgogICAgICAgIHJldHVybiBSZWZlcmVuY2VHZW5vbWUoZW5zdXJlX2hnMTlfY2hyMTcoKSwgc29sZV9jb250aWc9ImNocjE3IikKICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgZiJVbnN1cHBvcnRlZCBhc3NlbWJseSB7YXNzZW1ibHkhcn0uIEV4cGVjdGVkICdoZzM4JyBvciAnaGcxOScuIgogICAgKQo=",
"finetune/features.py": "IiIiRXZvMiBmZWF0dXJlIGV4dHJhY3Rpb24gZm9yIHZhcmlhbnQgZWZmZWN0IHByZWRpY3Rpb24uCgpUaGUgZXhwZW5zaXZlIHBhcnQgb2YgdGhlIHBpcGVsaW5lLiBGb3IgZWFjaCBTTlYgd2UgcnVuIHRoZSByZWZlcmVuY2UgYW5kIHRoZQptdXRhdGVkIHdpbmRvdyB0aHJvdWdoIEV2bzIgb25jZSBhbmQga2VlcCB0d28gdGhpbmdzIGZyb20gdGhlIHNhbWUgZm9yd2FyZCBwYXNzOgoKKiBwZXItdG9rZW4gbG9nLXByb2JhYmlsaXRpZXMsIHdoaWNoIGdpdmUgdGhlIHplcm8tc2hvdCBkZWx0YS1saWtlbGlob29kIHNjb3JlCiogaGlkZGVuIHN0YXRlcyBmcm9tIGBgYmxvY2tzLjI4Lm1scC5sM2BgLCBwb29sZWQgYXJvdW5kIHRoZSB2YXJpYW50Cgo6ZnVuYzpgZXh0cmFjdF92YXJpYW50X2ZlYXR1cmVzYCBpcyBzaGFyZWQgYnkgdGhlIHRyYWluaW5nIGpvYiBhbmQgdGhlIGluZmVyZW5jZQplbmRwb2ludCwgc28gdGhlIHZlY3RvciBhIHRyYWluZWQgaGVhZCBzZWVzIGF0IHNlcnZpbmcgdGltZSBpcyBidWlsdCBieSBleGFjdGx5CnRoZSBjb2RlIHRoYXQgcHJvZHVjZWQgaXRzIHRyYWluaW5nIGRhdGEuCiIiIgoKaW1wb3J0IG9zCmZyb20gdHlwaW5nIGltcG9ydCBEaWN0LCBMaXN0LCBPcHRpb25hbCwgU2VxdWVuY2UsIFR1cGxlCgpmcm9tIC4gaW1wb3J0IGNvbmZpZwpmcm9tIC5jb25maWcgaW1wb3J0IFNDQUxBUl9GRUFUVVJFX05BTUVTLCBGZWF0dXJlQ29uZmlnCgoKZGVmIHRva2VuX2xvZ3Byb2JzKGxvZ2l0cywgaW5wdXRfaWRzKToKICAgICIiIkxvZy1wcm9iYWJpbGl0eSB0aGUgbW9kZWwgYXNzaWduZWQgdG8gZWFjaCBvYnNlcnZlZCB0b2tlbi4KCiAgICBSZXR1cm5zIGEgYGAoYmF0Y2gsIGxlbmd0aClgYCBhcnJheSB3aGVyZSBlbGVtZW50IGBgaWBgIGlzCiAgICBgYGxvZyBQKGlucHV0X2lkc1tpXSB8IGlucHV0X2lkc1s8aV0pYGAsIGkuZS4gKippbmRleGVkIGJ5IHNlcXVlbmNlCiAgICBwb3NpdGlvbiwgbm90IHNoaWZ0ZWQqKi4gUG9zaXRpb24gMCBpcyBOYU4gYmVjYXVzZSBub3RoaW5nIHByZWNlZGVzIGl0LgoKICAgIEtlZXBpbmcgdGhlIGFycmF5IGFsaWduZWQgdG8gc2VxdWVuY2UgY29vcmRpbmF0ZXMgaXMgd2hhdCBsZXRzIHRoZSBwb29saW5nCiAgICBjb2RlIGJlbG93IHNsaWNlIGJ5IGdlbm9taWMgb2Zmc2V0IHdpdGhvdXQgYW4gb2ZmLWJ5LW9uZS4KICAgICIiIgogICAgaW1wb3J0IG51bXB5IGFzIG5wCiAgICBpbXBvcnQgdG9yY2gKCiAgICBsb2dwcm9icyA9IHRvcmNoLmxvZ19zb2Z0bWF4KGxvZ2l0cy5mbG9hdCgpLCBkaW09LTEpCiAgICAjIHByZWRpY3RlZFs6LCBpXSBzY29yZXMgaW5wdXRfaWRzWzosIGkgKyAxXQogICAgZ2F0aGVyZWQgPSB0b3JjaC5nYXRoZXIoCiAgICAgICAgbG9ncHJvYnNbOiwgOi0xXSwgMiwgaW5wdXRfaWRzWzosIDE6XS51bnNxdWVlemUoLTEpCiAgICApLnNxdWVlemUoLTEpCgogICAgYmF0Y2gsIGxlbmd0aCA9IGlucHV0X2lkcy5zaGFwZQogICAgb3V0ID0gbnAuZnVsbCgoYmF0Y2gsIGxlbmd0aCksIG5wLm5hbiwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIG91dFs6LCAxOl0gPSBnYXRoZXJlZC5mbG9hdCgpLmNwdSgpLm51bXB5KCkKICAgIHJldHVybiBvdXQKCgpkZWYgX3Bvb2xfc2xpY2UobGVuZ3RoOiBpbnQsIGNlbnRyZTogaW50LCByYWRpdXM6IE9wdGlvbmFsW2ludF0pIC0+IHNsaWNlOgogICAgaWYgcmFkaXVzIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIHNsaWNlKDAsIGxlbmd0aCkKICAgIHJldHVybiBzbGljZShtYXgoMCwgY2VudHJlIC0gcmFkaXVzKSwgbWluKGxlbmd0aCwgY2VudHJlICsgcmFkaXVzICsgMSkpCgoKZGVmIF9tZWFuX292ZXIodmFsdWVzLCBsZW5ndGg6IGludCwgY2VudHJlOiBpbnQsIHJhZGl1czogT3B0aW9uYWxbaW50XSkgLT4gZmxvYXQ6CiAgICBpbXBvcnQgbnVtcHkgYXMgbnAKCiAgICB3aW5kb3cgPSB2YWx1ZXNbX3Bvb2xfc2xpY2UobGVuZ3RoLCBjZW50cmUsIHJhZGl1cyldCiAgICBpZiBucC5hbGwobnAuaXNuYW4od2luZG93KSk6CiAgICAgICAgcmV0dXJuIDAuMAogICAgcmV0dXJuIGZsb2F0KG5wLm5hbm1lYW4od2luZG93KSkKCgpkZWYgX3Vud3JhcCh2YWx1ZSk6CiAgICAiIiJUYWtlIHRoZSB0ZW5zb3Igb3V0IG9mIGEgYGAodGVuc29yLCAuLi4pYGAgcmV0dXJuIHZhbHVlLgoKICAgIGBgRXZvMi5mb3J3YXJkYGAgcGFzc2VzIHRocm91Z2ggd2hhdGV2ZXIgdm9ydGV4J3MgYGBTdHJpcGVkSHllbmEuZm9yd2FyZGBgCiAgICByZXR1cm5zLCB3aGljaCBpcyBgYChsb2dpdHMsIGluZmVyZW5jZV9wYXJhbXMpYGAgcmF0aGVyIHRoYW4gYSBiYXJlIHRlbnNvci4KICAgIE90aGVyIHZlcnNpb25zIGhhbmQgYmFjayB0aGUgdGVuc29yIGRpcmVjdGx5LCBzbyBhY2NlcHQgYm90aCBpbnN0ZWFkIG9mCiAgICBkZXBlbmRpbmcgb24gd2hpY2ggb25lIGlzIGluc3RhbGxlZC4KICAgICIiIgogICAgd2hpbGUgaXNpbnN0YW5jZSh2YWx1ZSwgKHR1cGxlLCBsaXN0KSk6CiAgICAgICAgdmFsdWUgPSB2YWx1ZVswXQogICAgcmV0dXJuIHZhbHVlCgoKZGVmIF9mb3J3YXJkKG1vZGVsLCBzZXFzOiBTZXF1ZW5jZVtzdHJdLCBmZWF0dXJlX2NvbmZpZzogRmVhdHVyZUNvbmZpZyk6CiAgICAiIiJSdW4gRXZvMiBvbmNlIG92ZXIgYGBzZXFzYGAsIHJldHVybmluZyAobG9ncHJvYnMsIGVtYmVkZGluZ3MpLiIiIgogICAgaW1wb3J0IHRvcmNoCgogICAgZnJvbSBldm8yLnNjb3JpbmcgaW1wb3J0IHByZXBhcmVfYmF0Y2gKCiAgICBpbnB1dF9pZHMsIF8gPSBwcmVwYXJlX2JhdGNoKGxpc3Qoc2VxcyksIG1vZGVsLnRva2VuaXplciwgZGV2aWNlPSJjdWRhOjAiKQoKICAgIHdpdGggdG9yY2guaW5mZXJlbmNlX21vZGUoKToKICAgICAgICBsb2dpdHMsIGVtYmVkZGluZ3MgPSBtb2RlbC5mb3J3YXJkKAogICAgICAgICAgICBpbnB1dF9pZHMsCiAgICAgICAgICAgIHJldHVybl9lbWJlZGRpbmdzPVRydWUsCiAgICAgICAgICAgIGxheWVyX25hbWVzPVtmZWF0dXJlX2NvbmZpZy5lbWJlZGRpbmdfbGF5ZXJdLAogICAgICAgICkKCiAgICBsb2dpdHMgPSBfdW53cmFwKGxvZ2l0cykKICAgIGhpZGRlbiA9IF91bndyYXAoZW1iZWRkaW5nc1tmZWF0dXJlX2NvbmZpZy5lbWJlZGRpbmdfbGF5ZXJdKQogICAgaWYgaGlkZGVuLm5kaW0gPT0gMjogICMgZGVmZW5zaXZlOiBzb21lIGxheWVycyBlbWl0IChsZW5ndGgsIGhpZGRlbikKICAgICAgICBoaWRkZW4gPSBoaWRkZW4udW5zcXVlZXplKDApCgogICAgbHAgPSB0b2tlbl9sb2dwcm9icyhsb2dpdHMsIGlucHV0X2lkcykKICAgIGVtYiA9IGhpZGRlbi5mbG9hdCgpLmNwdSgpLm51bXB5KCkKICAgIGRlbCBsb2dpdHMsIGVtYmVkZGluZ3MsIGhpZGRlbiwgaW5wdXRfaWRzCiAgICByZXR1cm4gbHAsIGVtYgoKCmRlZiBleHRyYWN0X3ZhcmlhbnRfZmVhdHVyZXMoCiAgICBtb2RlbCwKICAgIHJlZl93aW5kb3c6IHN0ciwKICAgIHZhcl93aW5kb3c6IHN0ciwKICAgIHJlbGF0aXZlX3Bvc2l0aW9uOiBpbnQsCiAgICBmZWF0dXJlX2NvbmZpZzogT3B0aW9uYWxbRmVhdHVyZUNvbmZpZ10gPSBOb25lLAopIC0+IERpY3Rbc3RyLCAibnAubmRhcnJheSJdOgogICAgIiIiQnVpbGQgdGhlIGZlYXR1cmUgdmVjdG9yIGZvciBvbmUgU05WLgoKICAgIFJldHVybnMgYGB7ImVtYmVkZGluZyI6IChuX2VtYmVkZGluZ19mZWF0dXJlcywpLCAic2NhbGFyIjogKG5fc2NhbGFyLCksCiAgICAic2NhbGFyX25hbWVzIjogKC4uLil9YGAuIFRoZSBlbWJlZGRpbmcgYmxvY2sgaG9sZHMgYGB2YXIgLSByZWZgYCBkZWx0YXMgYXQKICAgIGVhY2ggY29uZmlndXJlZCBwb29saW5nIHJhZGl1cywgY29uY2F0ZW5hdGVkIGluIGBgcG9vbF9yYWRpaWBgIG9yZGVyLgogICAgIiIiCiAgICBpbXBvcnQgbnVtcHkgYXMgbnAKCiAgICBjZmcgPSBmZWF0dXJlX2NvbmZpZyBvciBjb25maWcuREVGQVVMVF9GRUFUVVJFX0NPTkZJRwoKICAgIGlmIGxlbihyZWZfd2luZG93KSAhPSBsZW4odmFyX3dpbmRvdyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgZiJSZWZlcmVuY2UgYW5kIHZhcmlhbnQgd2luZG93cyBkaWZmZXIgaW4gbGVuZ3RoICIKICAgICAgICAgICAgZiIoe2xlbihyZWZfd2luZG93KX0gdnMge2xlbih2YXJfd2luZG93KX0pOyBleHBlY3RlZCBhbiBTTlYuIgogICAgICAgICkKCiAgICBpZiBjZmcucGFpcl9iYXRjaDoKICAgICAgICBscCwgZW1iID0gX2ZvcndhcmQobW9kZWwsIFtyZWZfd2luZG93LCB2YXJfd2luZG93XSwgY2ZnKQogICAgICAgIHJlZl9scCwgdmFyX2xwID0gbHBbMF0sIGxwWzFdCiAgICAgICAgcmVmX2VtYiwgdmFyX2VtYiA9IGVtYlswXSwgZW1iWzFdCiAgICBlbHNlOgogICAgICAgIHJlZl9scF9iYXRjaCwgcmVmX2VtYl9iYXRjaCA9IF9mb3J3YXJkKG1vZGVsLCBbcmVmX3dpbmRvd10sIGNmZykKICAgICAgICB2YXJfbHBfYmF0Y2gsIHZhcl9lbWJfYmF0Y2ggPSBfZm9yd2FyZChtb2RlbCwgW3Zhcl93aW5kb3ddLCBjZmcpCiAgICAgICAgcmVmX2xwLCB2YXJfbHAgPSByZWZfbHBfYmF0Y2hbMF0sIHZhcl9scF9iYXRjaFswXQogICAgICAgIHJlZl9lbWIsIHZhcl9lbWIgPSByZWZfZW1iX2JhdGNoWzBdLCB2YXJfZW1iX2JhdGNoWzBdCgogICAgbGVuZ3RoID0gbGVuKHJlZl93aW5kb3cpCiAgICBjZW50cmUgPSByZWxhdGl2ZV9wb3NpdGlvbgoKICAgICMgLS0tIHBvb2xlZCBlbWJlZGRpbmcgZGVsdGFzCiAgICAjIEhpZGRlbiBzdGF0ZXMgYXJlIGluZGV4ZWQgYnkgc2VxdWVuY2UgcG9zaXRpb24gd2l0aCBubyBzaGlmdCwgYnV0IGNsYW1wCiAgICAjIHRoZSBjZW50cmUgaW4gY2FzZSB0aGUgbW9kZWwgcmV0dXJucyBhIGRpZmZlcmVudCBsZW5ndGggdGhhbiBpdCB3YXMgZ2l2ZW4uCiAgICBlbWJfbGVuID0gcmVmX2VtYi5zaGFwZVswXQogICAgZW1iX2NlbnRyZSA9IG1pbihjZW50cmUsIGVtYl9sZW4gLSAxKQoKICAgIGJsb2NrcyA9IFtdCiAgICBmb3IgcmFkaXVzIGluIGNmZy5wb29sX3JhZGlpOgogICAgICAgIHdpbmRvdyA9IF9wb29sX3NsaWNlKGVtYl9sZW4sIGVtYl9jZW50cmUsIHJhZGl1cykKICAgICAgICByZWZfcG9vbCA9IHJlZl9lbWJbd2luZG93XS5tZWFuKGF4aXM9MCkKICAgICAgICB2YXJfcG9vbCA9IHZhcl9lbWJbd2luZG93XS5tZWFuKGF4aXM9MCkKICAgICAgICBibG9ja3MuYXBwZW5kKHZhcl9wb29sIC0gcmVmX3Bvb2wpCiAgICBlbWJlZGRpbmcgPSBucC5jb25jYXRlbmF0ZShibG9ja3MpLmFzdHlwZShucC5mbG9hdDMyKQoKICAgICMgLS0tIHNjYWxhciBsaWtlbGlob29kIGZlYXR1cmVzCiAgICByZWZfZnVsbCA9IF9tZWFuX292ZXIocmVmX2xwLCBsZW5ndGgsIGNlbnRyZSwgTm9uZSkKICAgIHZhcl9mdWxsID0gX21lYW5fb3Zlcih2YXJfbHAsIGxlbmd0aCwgY2VudHJlLCBOb25lKQogICAgcmVmX2xvY2FsID0gX21lYW5fb3ZlcihyZWZfbHAsIGxlbmd0aCwgY2VudHJlLCBjZmcubG9jYWxfcmFkaXVzKQogICAgdmFyX2xvY2FsID0gX21lYW5fb3Zlcih2YXJfbHAsIGxlbmd0aCwgY2VudHJlLCBjZmcubG9jYWxfcmFkaXVzKQogICAgcmVmX2F0ID0gZmxvYXQocmVmX2xwW2NlbnRyZV0pIGlmIGNlbnRyZSA+IDAgZWxzZSAwLjAKICAgIHZhcl9hdCA9IGZsb2F0KHZhcl9scFtjZW50cmVdKSBpZiBjZW50cmUgPiAwIGVsc2UgMC4wCgogICAgc2NhbGFycyA9IHsKICAgICAgICAiZGVsdGFfc2NvcmVfZnVsbCI6IHZhcl9mdWxsIC0gcmVmX2Z1bGwsCiAgICAgICAgImRlbHRhX3Njb3JlX2xvY2FsIjogdmFyX2xvY2FsIC0gcmVmX2xvY2FsLAogICAgICAgICJkZWx0YV9scF9hdF92YXJpYW50IjogdmFyX2F0IC0gcmVmX2F0LAogICAgICAgICJyZWZfc2NvcmVfZnVsbCI6IHJlZl9mdWxsLAogICAgICAgICJyZWZfbHBfYXRfdmFyaWFudCI6IHJlZl9hdCwKICAgICAgICAidmFyX2xwX2F0X3ZhcmlhbnQiOiB2YXJfYXQsCiAgICB9CiAgICBzY2FsYXIgPSBucC5hcnJheSgKICAgICAgICBbc2NhbGFyc1tuYW1lXSBmb3IgbmFtZSBpbiBTQ0FMQVJfRkVBVFVSRV9OQU1FU10sIGR0eXBlPW5wLmZsb2F0MzIKICAgICkKCiAgICByZXR1cm4gewogICAgICAgICJlbWJlZGRpbmciOiBlbWJlZGRpbmcsCiAgICAgICAgInNjYWxhciI6IHNjYWxhciwKICAgICAgICAic2NhbGFyX25hbWVzIjogU0NBTEFSX0ZFQVRVUkVfTkFNRVMsCiAgICAgICAgIyBTdXJmYWNlZCBzZXBhcmF0ZWx5IHNvIHRoZSBlbmRwb2ludCBjYW4gc3RpbGwgcmVwb3J0IHRoZSB6ZXJvLXNob3QKICAgICAgICAjIG51bWJlciB0aGUgZnJvbnRlbmQgYWxyZWFkeSByZW5kZXJzLgogICAgICAgICJkZWx0YV9zY29yZSI6IHNjYWxhcnNbImRlbHRhX3Njb3JlX2Z1bGwiXSwKICAgIH0KCgojIC0tLSBTaGFyZGVkIGJhdGNoIGV4dHJhY3Rpb24gLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgoKZGVmIGZlYXR1cmVzX2RpcihmZWF0dXJlX2NvbmZpZzogT3B0aW9uYWxbRmVhdHVyZUNvbmZpZ10gPSBOb25lKSAtPiBzdHI6CiAgICBjZmcgPSBmZWF0dXJlX2NvbmZpZyBvciBjb25maWcuREVGQVVMVF9GRUFUVVJFX0NPTkZJRwogICAgcmV0dXJuIGYie2NvbmZpZy5GRUFUVVJFU19ESVJ9L3tjZmcudGFnKCl9IgoKCmRlZiBzaGFyZF9wYXRoKHNoYXJkX2luZGV4OiBpbnQsIGZlYXR1cmVfY29uZmlnOiBPcHRpb25hbFtGZWF0dXJlQ29uZmlnXSA9IE5vbmUpIC0+IHN0cjoKICAgIHJldHVybiBmIntmZWF0dXJlc19kaXIoZmVhdHVyZV9jb25maWcpfS9zaGFyZF97c2hhcmRfaW5kZXg6MDVkfS5ucHoiCgoKZGVmIHNoYXJkX2JvdW5kcyhuX2l0ZW1zOiBpbnQsIG5fc2hhcmRzOiBpbnQpIC0+IExpc3RbVHVwbGVbaW50LCBpbnRdXToKICAgICIiIlNwbGl0IGBgbl9pdGVtc2BgIGludG8gYGBuX3NoYXJkc2BgIGNvbnRpZ3VvdXMgcmFuZ2VzLiIiIgogICAgaWYgbl9zaGFyZHMgPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm5fc2hhcmRzIG11c3QgYmUgPj0gMSIpCiAgICBzaXplID0gKG5faXRlbXMgKyBuX3NoYXJkcyAtIDEpIC8vIG5fc2hhcmRzCiAgICByZXR1cm4gWwogICAgICAgIChzdGFydCwgbWluKHN0YXJ0ICsgc2l6ZSwgbl9pdGVtcykpCiAgICAgICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsIG5faXRlbXMsIHNpemUpCiAgICBdIG9yIFsoMCwgMCldCgoKZGVmIGV4dHJhY3Rfc2hhcmQoCiAgICBmcmFtZSwKICAgIHNoYXJkX2luZGV4OiBpbnQsCiAgICBmZWF0dXJlX2NvbmZpZzogT3B0aW9uYWxbRmVhdHVyZUNvbmZpZ10gPSBOb25lLAogICAgc2tpcF9leGlzdGluZzogYm9vbCA9IFRydWUsCiAgICBtb2RlbD1Ob25lLAopIC0+IHN0cjoKICAgICIiIkV4dHJhY3QgZmVhdHVyZXMgZm9yIG9uZSBzaGFyZCBvZiB0aGUgdmFyaWFudCB0YWJsZSBhbmQgc2F2ZSBhbiBgYC5ucHpgYC4KCiAgICBSb3dzIHdob3NlIHJlZmVyZW5jZSBiYXNlIGRpc2FncmVlcyB3aXRoIHRoZSBhc3NlbWJseSBhcmUgZHJvcHBlZCBhbmQKICAgIGNvdW50ZWQgcmF0aGVyIHRoYW4gc2NvcmVkLCBzaW5jZSBhIGNvb3JkaW5hdGUgb3IgYXNzZW1ibHkgZXJyb3Igd291bGQKICAgIG90aGVyd2lzZSBlbnRlciB0aGUgdHJhaW5pbmcgc2V0IGFzIGEgbWlzbGFiZWxsZWQgZXhhbXBsZS4KCiAgICBgYG1vZGVsYGAgbGV0cyBhIGNhbGxlciB0aGF0IHJ1bnMgc2V2ZXJhbCBzaGFyZHMgaW4gb25lIHByb2Nlc3MgKHRoZSBLYWdnbGUKICAgIG5vdGVib29rcykgbG9hZCBFdm8yIG9uY2UgYW5kIHBhc3MgaXQgaW47IG9uIE1vZGFsIGVhY2ggc2hhcmQgaXMgYSBmcmVzaAogICAgY29udGFpbmVyIHNvIGl0IGlzIGxlZnQgYGBOb25lYGAgYW5kIGxvYWRlZCBoZXJlLgogICAgIiIiCiAgICBpbXBvcnQgbnVtcHkgYXMgbnAKCiAgICBmcm9tIC5sb2FkZXIgaW1wb3J0IGxvYWRfZXZvMgoKICAgIGNmZyA9IGZlYXR1cmVfY29uZmlnIG9yIGNvbmZpZy5ERUZBVUxUX0ZFQVRVUkVfQ09ORklHCiAgICBvdXRfcGF0aCA9IHNoYXJkX3BhdGgoc2hhcmRfaW5kZXgsIGNmZykKCiAgICBpZiBza2lwX2V4aXN0aW5nIGFuZCBvcy5wYXRoLmV4aXN0cyhvdXRfcGF0aCk6CiAgICAgICAgcHJpbnQoZiJTaGFyZCB7c2hhcmRfaW5kZXh9IGFscmVhZHkgZG9uZSBhdCB7b3V0X3BhdGh9LCBza2lwcGluZyIpCiAgICAgICAgcmV0dXJuIG91dF9wYXRoCgogICAgb3MubWFrZWRpcnMoZmVhdHVyZXNfZGlyKGNmZyksIGV4aXN0X29rPVRydWUpCgogICAgaWYgbW9kZWwgaXMgTm9uZToKICAgICAgICBwcmludChmIkxvYWRpbmcge2NvbmZpZy5NT0RFTF9OQU1FfSAuLi4iKQogICAgICAgIG1vZGVsID0gbG9hZF9ldm8yKGNvbmZpZy5NT0RFTF9OQU1FKQogICAgICAgIHByaW50KCJNb2RlbCBsb2FkZWQiKQoKICAgIGdlbm9tZXMgPSB7fQogICAgdmFyaWFudF9pZHMsIGVtYmVkZGluZ3MsIHNjYWxhcnMsIGxhYmVscyA9IFtdLCBbXSwgW10sIFtdCiAgICBza2lwcGVkID0geyJyZWZlcmVuY2VfbWlzbWF0Y2giOiAwLCAib3V0X29mX2JvdW5kcyI6IDAsICJlcnJvciI6IDB9CgogICAgdHJ5OgogICAgICAgIF9zY29yZV9yb3dzKAogICAgICAgICAgICBmcmFtZSwgY2ZnLCBtb2RlbCwgZ2Vub21lcywKICAgICAgICAgICAgdmFyaWFudF9pZHMsIGVtYmVkZGluZ3MsIHNjYWxhcnMsIGxhYmVscywgc2tpcHBlZCwgc2hhcmRfaW5kZXgsCiAgICAgICAgKQogICAgZmluYWxseToKICAgICAgICAjIExlYXZpbmcgdGhlc2Ugb3BlbiB3b3VsZCBicmVhayB0aGUgdm9sdW1lIHJlbG9hZCBhdCB0aGUgc3RhcnQgb2YgdGhlCiAgICAgICAgIyBuZXh0IHNoYXJkIHNjaGVkdWxlZCBvbnRvIHRoaXMgY29udGFpbmVyLgogICAgICAgIGZvciBnZW5vbWUgaW4gZ2Vub21lcy52YWx1ZXMoKToKICAgICAgICAgICAgZ2Vub21lLmNsb3NlKCkKCiAgICBpZiBub3QgdmFyaWFudF9pZHM6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICBmIlNoYXJkIHtzaGFyZF9pbmRleH0gcHJvZHVjZWQgbm8gZmVhdHVyZXMuIFNraXBwZWQ6IHtza2lwcGVkfSIKICAgICAgICApCgogICAgIyBXcml0dGVuIHRocm91Z2ggYW4gb3BlbiBmaWxlIGhhbmRsZSwgbm90IGEgcGF0aDogZ2l2ZW4gYSBwYXRoIHRoYXQgZG9lcwogICAgIyBub3QgYWxyZWFkeSBlbmQgaW4gYGAubnB6YGAsIG51bXB5IHNpbGVudGx5IGFwcGVuZHMgdGhlIHN1ZmZpeCwgc28KICAgICMgYGBzaGFyZF8wLm5wei5wYXJ0YGAgbGFuZHMgYXMgYGBzaGFyZF8wLm5wei5wYXJ0Lm5wemBgIGFuZCB0aGUgcmVuYW1lCiAgICAjIGJlbG93IHRoZW4gZmFpbHMgb24gYSBmaWxlIHRoYXQgd2FzIG5ldmVyIGNyZWF0ZWQg4oCUIGFmdGVyIHRoZSBzaGFyZCdzCiAgICAjIGVudGlyZSBHUFUgY29zdCBoYXMgYmVlbiBwYWlkLiBQYXNzaW5nIGEgaGFuZGxlIGRpc2FibGVzIHRoYXQgcmV3cml0aW5nLgogICAgdG1wX3BhdGggPSBvdXRfcGF0aCArICIucGFydCIKICAgIHdpdGggb3Blbih0bXBfcGF0aCwgIndiIikgYXMgaGFuZGxlOgogICAgICAgIG5wLnNhdmV6X2NvbXByZXNzZWQoCiAgICAgICAgICAgIGhhbmRsZSwKICAgICAgICAgICAgdmFyaWFudF9pZD1ucC5hcnJheSh2YXJpYW50X2lkcyksCiAgICAgICAgICAgIGVtYmVkZGluZz1ucC5zdGFjayhlbWJlZGRpbmdzKSwKICAgICAgICAgICAgc2NhbGFyPW5wLnN0YWNrKHNjYWxhcnMpLAogICAgICAgICAgICBsYWJlbD1ucC5hcnJheShsYWJlbHMsIGR0eXBlPW5wLmludDY0KSwKICAgICAgICApCiAgICBvcy5yZXBsYWNlKHRtcF9wYXRoLCBvdXRfcGF0aCkKCiAgICBwcmludCgKICAgICAgICBmIlNoYXJkIHtzaGFyZF9pbmRleH06IHdyb3RlIHtsZW4odmFyaWFudF9pZHMpfSByb3dzIHRvIHtvdXRfcGF0aH0gIgogICAgICAgIGYiKHNraXBwZWQge3NraXBwZWR9KSIKICAgICkKICAgIHJldHVybiBvdXRfcGF0aAoKCmRlZiBfc2NvcmVfcm93cygKICAgIGZyYW1lLCBjZmcsIG1vZGVsLCBnZW5vbWVzLAogICAgdmFyaWFudF9pZHMsIGVtYmVkZGluZ3MsIHNjYWxhcnMsIGxhYmVscywgc2tpcHBlZCwgc2hhcmRfaW5kZXgsCik6CiAgICAiIiJTY29yZSBldmVyeSByb3cgb2YgYGBmcmFtZWBgIGluIHBsYWNlIGludG8gdGhlIGFjY3VtdWxhdG9yIGxpc3RzLiIiIgogICAgaW1wb3J0IG51bXB5IGFzIG5wCiAgICBpbXBvcnQgdG9yY2gKCiAgICBmcm9tIC5zZXF1ZW5jZXMgaW1wb3J0IGJ1aWxkX3ZhcmlhbnRfd2luZG93LCBvcGVuX2dlbm9tZQoKICAgIGZvciBjb3VudGVyLCAoXywgcm93KSBpbiBlbnVtZXJhdGUoZnJhbWUuaXRlcnJvd3MoKSwgc3RhcnQ9MSk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBhc3NlbWJseSA9IHJvd1siYXNzZW1ibHkiXQogICAgICAgICAgICBpZiBhc3NlbWJseSBub3QgaW4gZ2Vub21lczoKICAgICAgICAgICAgICAgIGdlbm9tZXNbYXNzZW1ibHldID0gb3Blbl9nZW5vbWUoYXNzZW1ibHkpCiAgICAgICAgICAgIGdlbm9tZSA9IGdlbm9tZXNbYXNzZW1ibHldCgogICAgICAgICAgICByZWZfd2luZG93LCBzdGFydCA9IGdlbm9tZS53aW5kb3coCiAgICAgICAgICAgICAgICByb3dbImNocm9tIl0sIGludChyb3dbInBvcyJdKSwgY2ZnLndpbmRvd19zaXplCiAgICAgICAgICAgICkKICAgICAgICAgICAgcmVsYXRpdmUgPSBpbnQocm93WyJwb3MiXSkgLSAxIC0gc3RhcnQKICAgICAgICAgICAgdmFyX3dpbmRvdywgXyA9IGJ1aWxkX3ZhcmlhbnRfd2luZG93KAogICAgICAgICAgICAgICAgcmVmX3dpbmRvdywgcmVsYXRpdmUsIHJvd1siYWx0Il0sIGV4cGVjdGVkX3JlZmVyZW5jZT1yb3dbInJlZiJdCiAgICAgICAgICAgICkKCiAgICAgICAgICAgIHJlc3VsdCA9IGV4dHJhY3RfdmFyaWFudF9mZWF0dXJlcygKICAgICAgICAgICAgICAgIG1vZGVsLCByZWZfd2luZG93LCB2YXJfd2luZG93LCByZWxhdGl2ZSwgY2ZnCiAgICAgICAgICAgICkKICAgICAgICAgICAgdmFyaWFudF9pZHMuYXBwZW5kKHJvd1sidmFyaWFudF9pZCJdKQogICAgICAgICAgICBlbWJlZGRpbmdzLmFwcGVuZChyZXN1bHRbImVtYmVkZGluZyJdLmFzdHlwZShucC5mbG9hdDE2KSkKICAgICAgICAgICAgc2NhbGFycy5hcHBlbmQocmVzdWx0WyJzY2FsYXIiXSkKICAgICAgICAgICAgbGFiZWxzLmFwcGVuZChpbnQocm93WyJsYWJlbCJdKSkKCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3IgYXMgZXhjOgogICAgICAgICAgICBrZXkgPSAoCiAgICAgICAgICAgICAgICAicmVmZXJlbmNlX21pc21hdGNoIgogICAgICAgICAgICAgICAgaWYgIlJlZmVyZW5jZSBtaXNtYXRjaCIgaW4gc3RyKGV4YykKICAgICAgICAgICAgICAgIGVsc2UgIm91dF9vZl9ib3VuZHMiCiAgICAgICAgICAgICkKICAgICAgICAgICAgc2tpcHBlZFtrZXldICs9IDEKICAgICAgICBleGNlcHQgdG9yY2guY3VkYS5PdXRPZk1lbW9yeUVycm9yIGFzIGV4YzoKICAgICAgICAgICAgIyBBbiBPT00gbGVhdmVzIHRoZSBmYWlsZWQgYWxsb2NhdGlvbidzIG1lbW9yeSByZXNlcnZlZCwgc28gd2l0aG91dAogICAgICAgICAgICAjIGRyb3BwaW5nIHRoZSBjYWNoZSBoZXJlIG9uZSBiYWQgcm93IGNhc2NhZGVzIGludG8gZXZlcnkgcm93IGFmdGVyCiAgICAgICAgICAgICMgaXQgZmFpbGluZyB0b28uIFJldHJpZWQgb25jZSwgc2luY2Ugd2l0aCB0aGUgY2FjaGUgY2xlYXJlZCB0aGUKICAgICAgICAgICAgIyBzYW1lIHdpbmRvdyB1c3VhbGx5IGZpdHMuCiAgICAgICAgICAgIHNraXBwZWRbImVycm9yIl0gKz0gMQogICAgICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkKICAgICAgICAgICAgaWYgc2tpcHBlZFsiZXJyb3IiXSA8PSA1OgogICAgICAgICAgICAgICAgcHJpbnQoZiIgIE9PTSBvbiB7cm93LmdldCgndmFyaWFudF9pZCcpfSwgY2FjaGUgY2xlYXJlZDoge2V4Y30iKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOiAgIyBub3FhOiBCTEUwMDEgLSBvbmUgYmFkIHJvdyBtdXN0IG5vdCBraWxsIGEgc2hhcmQKICAgICAgICAgICAgc2tpcHBlZFsiZXJyb3IiXSArPSAxCiAgICAgICAgICAgIGlmIHNraXBwZWRbImVycm9yIl0gPD0gNToKICAgICAgICAgICAgICAgIHByaW50KGYiICBlcnJvciBvbiB7cm93LmdldCgndmFyaWFudF9pZCcpfToge2V4Y30iKQoKICAgICAgICBpZiBjb3VudGVyICUgMTAwID09IDA6CiAgICAgICAgICAgIHByaW50KGYiICBzaGFyZCB7c2hhcmRfaW5kZXh9OiB7Y291bnRlcn0ve2xlbihmcmFtZSl9IHByb2Nlc3NlZCIpCgoKZGVmIG1lcmdlX3NoYXJkcyhmZWF0dXJlX2NvbmZpZzogT3B0aW9uYWxbRmVhdHVyZUNvbmZpZ10gPSBOb25lKSAtPiBzdHI6CiAgICAiIiJDb25jYXRlbmF0ZSBldmVyeSBzaGFyZCBpbnRvIGEgc2luZ2xlIGBgZmVhdHVyZXMubnB6YGAuIiIiCiAgICBpbXBvcnQgZ2xvYgogICAgaW1wb3J0IHJlCgogICAgaW1wb3J0IG51bXB5IGFzIG5wCgogICAgY2ZnID0gZmVhdHVyZV9jb25maWcgb3IgY29uZmlnLkRFRkFVTFRfRkVBVFVSRV9DT05GSUcKICAgICMgRGVsaWJlcmF0ZWx5IHN0cmljdDogYHNoYXJkXyoubnB6YCB3b3VsZCBhbHNvIG1hdGNoIGxlZnRvdmVycyBsaWtlCiAgICAjIGBzaGFyZF8wMDAwMC5ucHoucGFydC5ucHpgLCB3aG9zZSByb3dzIGJlbG9uZyB0byB3aGF0ZXZlciBzcGxpdCB3YXMgYmVpbmcKICAgICMgZXh0cmFjdGVkIHdoZW4gdGhleSB3ZXJlIHdyaXR0ZW4uIFNpbGVudGx5IGNvbmNhdGVuYXRpbmcgdGhvc2UgaW50byB0aGUKICAgICMgdHJhaW5pbmcgc2V0IGlzIGZhciB3b3JzZSB0aGFuIGZhaWxpbmcgdG8gZmluZCB0aGVtLgogICAgcGF0dGVybiA9IHJlLmNvbXBpbGUociJec2hhcmRfXGR7NX1cLm5weiQiKQogICAgcGF0aHMgPSBzb3J0ZWQoCiAgICAgICAgcCBmb3IgcCBpbiBnbG9iLmdsb2IoZiJ7ZmVhdHVyZXNfZGlyKGNmZyl9L3NoYXJkXyoubnB6IikKICAgICAgICBpZiBwYXR0ZXJuLm1hdGNoKG9zLnBhdGguYmFzZW5hbWUocCkpCiAgICApCiAgICBpZiBub3QgcGF0aHM6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoCiAgICAgICAgICAgIGYiTm8gc2hhcmRzIHVuZGVyIHtmZWF0dXJlc19kaXIoY2ZnKX0uIFJ1biB0aGUgZXh0cmFjdCBzdGFnZSBmaXJzdC4iCiAgICAgICAgKQoKICAgIHZhcmlhbnRfaWRzLCBlbWJlZGRpbmdzLCBzY2FsYXJzLCBsYWJlbHMgPSBbXSwgW10sIFtdLCBbXQogICAgZm9yIHBhdGggaW4gcGF0aHM6CiAgICAgICAgd2l0aCBucC5sb2FkKHBhdGgsIGFsbG93X3BpY2tsZT1GYWxzZSkgYXMgZGF0YToKICAgICAgICAgICAgdmFyaWFudF9pZHMuYXBwZW5kKGRhdGFbInZhcmlhbnRfaWQiXSkKICAgICAgICAgICAgZW1iZWRkaW5ncy5hcHBlbmQoZGF0YVsiZW1iZWRkaW5nIl0pCiAgICAgICAgICAgIHNjYWxhcnMuYXBwZW5kKGRhdGFbInNjYWxhciJdKQogICAgICAgICAgICBsYWJlbHMuYXBwZW5kKGRhdGFbImxhYmVsIl0pCgogICAgbWVyZ2VkID0gZiJ7ZmVhdHVyZXNfZGlyKGNmZyl9L2ZlYXR1cmVzLm5weiIKICAgIG5wLnNhdmV6KAogICAgICAgIG1lcmdlZCwKICAgICAgICB2YXJpYW50X2lkPW5wLmNvbmNhdGVuYXRlKHZhcmlhbnRfaWRzKSwKICAgICAgICBlbWJlZGRpbmc9bnAuY29uY2F0ZW5hdGUoZW1iZWRkaW5ncyksCiAgICAgICAgc2NhbGFyPW5wLmNvbmNhdGVuYXRlKHNjYWxhcnMpLAogICAgICAgIGxhYmVsPW5wLmNvbmNhdGVuYXRlKGxhYmVscyksCiAgICApCiAgICB0b3RhbCA9IHN1bShsZW4oeCkgZm9yIHggaW4gbGFiZWxzKQogICAgcHJpbnQoZiJNZXJnZWQge2xlbihwYXRocyl9IHNoYXJkcyAoe3RvdGFsfSB2YXJpYW50cykgaW50byB7bWVyZ2VkfSIpCiAgICByZXR1cm4gbWVyZ2VkCgoKZGVmIGxvYWRfZmVhdHVyZXMoZmVhdHVyZV9jb25maWc6IE9wdGlvbmFsW0ZlYXR1cmVDb25maWddID0gTm9uZSk6CiAgICAiIiJMb2FkIG1lcmdlZCBmZWF0dXJlcyBhcyBgYCh2YXJpYW50X2lkLCBYLCB5KWBgIHdpdGggWCBmbG9hdDMyLiIiIgogICAgaW1wb3J0IG51bXB5IGFzIG5wCgogICAgY2ZnID0gZmVhdHVyZV9jb25maWcgb3IgY29uZmlnLkRFRkFVTFRfRkVBVFVSRV9DT05GSUcKICAgIG1lcmdlZCA9IGYie2ZlYXR1cmVzX2RpcihjZmcpfS9mZWF0dXJlcy5ucHoiCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMobWVyZ2VkKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICAgICAgZiJ7bWVyZ2VkfSBub3QgZm91bmQuIFJ1biB0aGUgZXh0cmFjdCBhbmQgbWVyZ2Ugc3RhZ2VzIGZpcnN0LiIKICAgICAgICApCgogICAgd2l0aCBucC5sb2FkKG1lcmdlZCwgYWxsb3dfcGlja2xlPUZhbHNlKSBhcyBkYXRhOgogICAgICAgIHZhcmlhbnRfaWQgPSBkYXRhWyJ2YXJpYW50X2lkIl0KICAgICAgICBYID0gbnAuY29uY2F0ZW5hdGUoCiAgICAgICAgICAgIFtkYXRhWyJlbWJlZGRpbmciXS5hc3R5cGUobnAuZmxvYXQzMiksIGRhdGFbInNjYWxhciJdLmFzdHlwZShucC5mbG9hdDMyKV0sCiAgICAgICAgICAgIGF4aXM9MSwKICAgICAgICApCiAgICAgICAgeSA9IGRhdGFbImxhYmVsIl0KICAgIHJldHVybiB2YXJpYW50X2lkLCBYLCB5Cg==",
"finetune/head.py": "IiIiVGhlIHRyYWluZWQgY2xhc3NpZmllciBoZWFkIGFuZCBpdHMgc2VyaWFsaXNlZCBhcnRpZmFjdC4KClRoZSBoZWFkIGlzIGRlbGliZXJhdGVseSBzbWFsbDogRXZvMiBzdGF5cyBmcm96ZW4gYW5kIG9ubHkgdGhpcyBzaXRzIG9uIHRvcCBvZgppdHMgZW1iZWRkaW5ncy4gRXZlcnl0aGluZyB0aGUgaW5mZXJlbmNlIGVuZHBvaW50IG5lZWRzIHRvIHJlcHJvZHVjZSBhCnByZWRpY3Rpb24g4oCUIGFyY2hpdGVjdHVyZSwgZmVhdHVyZSBzdGFuZGFyZGlzYXRpb24sIGRlY2lzaW9uIHRocmVzaG9sZCBhbmQgdGhlCmZlYXR1cmUgY29uZmlnIHRoZSB2ZWN0b3Igd2FzIGJ1aWx0IHdpdGgg4oCUIHRyYXZlbHMgaW4gYSBzaW5nbGUgYGBoZWFkLnB0YGAgc28gYQpjaGVja3BvaW50IGNhbiBuZXZlciBiZSBwYWlyZWQgd2l0aCB0aGUgd3JvbmcgcHJlcHJvY2Vzc2luZy4KIiIiCgppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSB0eXBpbmcgaW1wb3J0IERpY3QsIE9wdGlvbmFsLCBTZXF1ZW5jZQoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgoKZnJvbSAuIGltcG9ydCBjb25maWcKZnJvbSAuY29uZmlnIGltcG9ydCBGZWF0dXJlQ29uZmlnCgpBUlRJRkFDVF9OQU1FID0gImhlYWQucHQiCk1FVFJJQ1NfTkFNRSA9ICJtZXRyaWNzLmpzb24iCgoKY2xhc3MgVmFyaWFudEhlYWQobm4uTW9kdWxlKToKICAgICIiIkxpbmVhciBvciBzbWFsbCBNTFAgY2xhc3NpZmllciBvdmVyIHN0YW5kYXJkaXNlZCBFdm8yIGZlYXR1cmVzLiIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIG5fZmVhdHVyZXM6IGludCwKICAgICAgICBoZWFkOiBzdHIgPSAibWxwIiwKICAgICAgICBoaWRkZW5fc2l6ZXM6IFNlcXVlbmNlW2ludF0gPSAoMjU2LCA2NCksCiAgICAgICAgZHJvcG91dDogZmxvYXQgPSAwLjMsCiAgICApOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYubl9mZWF0dXJlcyA9IG5fZmVhdHVyZXMKICAgICAgICBzZWxmLmhlYWQgPSBoZWFkCgogICAgICAgIGlmIGhlYWQgPT0gImxpbmVhciI6CiAgICAgICAgICAgIHNlbGYubmV0ID0gbm4uTGluZWFyKG5fZmVhdHVyZXMsIDEpCiAgICAgICAgZWxpZiBoZWFkID09ICJtbHAiOgogICAgICAgICAgICBsYXllcnMgPSBbXQogICAgICAgICAgICBpbl9kaW0gPSBuX2ZlYXR1cmVzCiAgICAgICAgICAgIGZvciB3aWR0aCBpbiBoaWRkZW5fc2l6ZXM6CiAgICAgICAgICAgICAgICBsYXllcnMgKz0gWwogICAgICAgICAgICAgICAgICAgIG5uLkxpbmVhcihpbl9kaW0sIHdpZHRoKSwKICAgICAgICAgICAgICAgICAgICBubi5MYXllck5vcm0od2lkdGgpLAogICAgICAgICAgICAgICAgICAgIG5uLkdFTFUoKSwKICAgICAgICAgICAgICAgICAgICBubi5Ecm9wb3V0KGRyb3BvdXQpLAogICAgICAgICAgICAgICAgXQogICAgICAgICAgICAgICAgaW5fZGltID0gd2lkdGgKICAgICAgICAgICAgbGF5ZXJzLmFwcGVuZChubi5MaW5lYXIoaW5fZGltLCAxKSkKICAgICAgICAgICAgc2VsZi5uZXQgPSBubi5TZXF1ZW50aWFsKCpsYXllcnMpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVua25vd24gaGVhZCB7aGVhZCFyfTsgZXhwZWN0ZWQgJ2xpbmVhcicgb3IgJ21scCciKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgICIiIlJldHVybiByYXcgbG9naXRzIG9mIHNoYXBlIGBgKGJhdGNoLClgYC4iIiIKICAgICAgICByZXR1cm4gc2VsZi5uZXQoeCkuc3F1ZWV6ZSgtMSkKCgpjbGFzcyBTdGFuZGFyZGl6ZXI6CiAgICAiIiJQZXItZmVhdHVyZSBtZWFuL3N0ZCBub3JtYWxpc2F0aW9uLCBmaXR0ZWQgb24gdGhlIHRyYWluaW5nIHNwbGl0IG9ubHkuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1lYW4sIHN0ZCk6CiAgICAgICAgaW1wb3J0IG51bXB5IGFzIG5wCgogICAgICAgIHNlbGYubWVhbiA9IG5wLmFzYXJyYXkobWVhbiwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICAjIEEgY29uc3RhbnQgZmVhdHVyZSB3b3VsZCBkaXZpZGUgYnkgemVybzsgbGVhdmUgaXQgYXQgemVybyBpbnN0ZWFkLgogICAgICAgIHN0ZCA9IG5wLmFzYXJyYXkoc3RkLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgIHNlbGYuc3RkID0gbnAud2hlcmUoc3RkIDwgMWUtNiwgMS4wLCBzdGQpLmFzdHlwZShucC5mbG9hdDMyKQoKICAgIEBjbGFzc21ldGhvZAogICAgZGVmIGZpdChjbHMsIFgpIC0+ICJTdGFuZGFyZGl6ZXIiOgogICAgICAgIHJldHVybiBjbHMoWC5tZWFuKGF4aXM9MCksIFguc3RkKGF4aXM9MCkpCgogICAgZGVmIHRyYW5zZm9ybShzZWxmLCBYKToKICAgICAgICBpbXBvcnQgbnVtcHkgYXMgbnAKCiAgICAgICAgcmV0dXJuICgobnAuYXNhcnJheShYLCBkdHlwZT1ucC5mbG9hdDMyKSAtIHNlbGYubWVhbikgLyBzZWxmLnN0ZCkuYXN0eXBlKAogICAgICAgICAgICBucC5mbG9hdDMyCiAgICAgICAgKQoKCmNsYXNzIFRyYWluZWRIZWFkOgogICAgIiIiQSBoZWFkIHBsdXMgZXZlcnl0aGluZyBuZWVkZWQgdG8gYXBwbHkgaXQgdG8gYSBmcmVzaCB2YXJpYW50LiIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIG1vZGVsOiBWYXJpYW50SGVhZCwKICAgICAgICBzdGFuZGFyZGl6ZXI6IFN0YW5kYXJkaXplciwKICAgICAgICB0aHJlc2hvbGQ6IGZsb2F0LAogICAgICAgIGZlYXR1cmVfY29uZmlnOiBGZWF0dXJlQ29uZmlnLAogICAgICAgIG1ldHJpY3M6IE9wdGlvbmFsW0RpY3RdID0gTm9uZSwKICAgICk6CiAgICAgICAgc2VsZi5tb2RlbCA9IG1vZGVsCiAgICAgICAgc2VsZi5zdGFuZGFyZGl6ZXIgPSBzdGFuZGFyZGl6ZXIKICAgICAgICAjIERlY2lzaW9uIHRocmVzaG9sZCBvbiB0aGUgKnByb2JhYmlsaXR5KiwgY2hvc2VuIG9uIHRoZSB2YWxpZGF0aW9uIHNwbGl0LgogICAgICAgIHNlbGYudGhyZXNob2xkID0gZmxvYXQodGhyZXNob2xkKQogICAgICAgIHNlbGYuZmVhdHVyZV9jb25maWcgPSBmZWF0dXJlX2NvbmZpZwogICAgICAgIHNlbGYubWV0cmljcyA9IG1ldHJpY3Mgb3Ige30KCiAgICAjIC0tIGFwcGxpY2F0aW9uIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKICAgIGRlZiBwcmVkaWN0X3Byb2JhKHNlbGYsIFgpIC0+ICJucC5uZGFycmF5IjoKICAgICAgICBpbXBvcnQgbnVtcHkgYXMgbnAKCiAgICAgICAgZGV2aWNlID0gbmV4dChzZWxmLm1vZGVsLnBhcmFtZXRlcnMoKSkuZGV2aWNlCiAgICAgICAgdGVuc29yID0gdG9yY2guZnJvbV9udW1weShzZWxmLnN0YW5kYXJkaXplci50cmFuc2Zvcm0oWCkpLnRvKGRldmljZSkKICAgICAgICBzZWxmLm1vZGVsLmV2YWwoKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBsb2dpdHMgPSBzZWxmLm1vZGVsKHRlbnNvcikKICAgICAgICAgICAgcHJvYnMgPSB0b3JjaC5zaWdtb2lkKGxvZ2l0cykuY3B1KCkubnVtcHkoKQogICAgICAgIHJldHVybiBucC5hdGxlYXN0XzFkKHByb2JzKQoKICAgIGRlZiBwcmVkaWN0X29uZShzZWxmLCBlbWJlZGRpbmcsIHNjYWxhcikgLT4gRGljdFtzdHIsIGZsb2F0XToKICAgICAgICAiIiJDbGFzc2lmeSBhIHNpbmdsZSB2YXJpYW50IGZyb20gOmZ1bmM6YGZlYXR1cmVzLmV4dHJhY3RfdmFyaWFudF9mZWF0dXJlc2AuCgogICAgICAgIGBgY2xhc3NpZmljYXRpb25fY29uZmlkZW5jZWBgIGlzIHRoZSBjYWxpYnJhdGVkIHByb2JhYmlsaXR5IG9mIHdoaWNoZXZlcgogICAgICAgIGNsYXNzIHdhcyBwcmVkaWN0ZWQsIHNvIGl0IGlzIGEgZ2VudWluZSBwcm9iYWJpbGl0eSByYXRoZXIgdGhhbiB0aGUKICAgICAgICBkaXN0YW5jZS1vdmVyLXNpZ21hIGhldXJpc3RpYyB0aGUgemVyby1zaG90IHBhdGggdXNlcy4KICAgICAgICAiIiIKICAgICAgICBpbXBvcnQgbnVtcHkgYXMgbnAKCiAgICAgICAgeCA9IG5wLmNvbmNhdGVuYXRlKAogICAgICAgICAgICBbbnAuYXNhcnJheShlbWJlZGRpbmcsIGR0eXBlPW5wLmZsb2F0MzIpLCBucC5hc2FycmF5KHNjYWxhciwgZHR5cGU9bnAuZmxvYXQzMildCiAgICAgICAgKS5yZXNoYXBlKDEsIC0xKQogICAgICAgIGlmIHguc2hhcGVbMV0gIT0gc2VsZi5tb2RlbC5uX2ZlYXR1cmVzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJGZWF0dXJlIHZlY3RvciBoYXMge3guc2hhcGVbMV19IGRpbWVuc2lvbnMgYnV0IHRoZSBoZWFkIGV4cGVjdHMgIgogICAgICAgICAgICAgICAgZiJ7c2VsZi5tb2RlbC5uX2ZlYXR1cmVzfS4gVGhlIGhlYWQgd2FzIHRyYWluZWQgd2l0aCBhIGRpZmZlcmVudCAiCiAgICAgICAgICAgICAgICAiRmVhdHVyZUNvbmZpZyB0aGFuIHRoZSBvbmUgdXNlZCB0byBidWlsZCB0aGlzIHZlY3Rvci4iCiAgICAgICAgICAgICkKCiAgICAgICAgcHJvYmFiaWxpdHkgPSBmbG9hdChzZWxmLnByZWRpY3RfcHJvYmEoeClbMF0pCiAgICAgICAgaXNfcGF0aG9nZW5pYyA9IHByb2JhYmlsaXR5ID49IHNlbGYudGhyZXNob2xkCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgInByZWRpY3Rpb24iOiAiTGlrZWx5IHBhdGhvZ2VuaWMiIGlmIGlzX3BhdGhvZ2VuaWMgZWxzZSAiTGlrZWx5IGJlbmlnbiIsCiAgICAgICAgICAgICJwYXRob2dlbmljaXR5X3Byb2JhYmlsaXR5IjogcHJvYmFiaWxpdHksCiAgICAgICAgICAgICJjbGFzc2lmaWNhdGlvbl9jb25maWRlbmNlIjogKAogICAgICAgICAgICAgICAgcHJvYmFiaWxpdHkgaWYgaXNfcGF0aG9nZW5pYyBlbHNlIDEuMCAtIHByb2JhYmlsaXR5CiAgICAgICAgICAgICksCiAgICAgICAgfQoKICAgICMgLS0gcGVyc2lzdGVuY2UgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgogICAgZGVmIHNhdmUoc2VsZiwgcnVuX2Rpcjogc3RyKSAtPiBzdHI6CiAgICAgICAgb3MubWFrZWRpcnMocnVuX2RpciwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBwYXRoID0gb3MucGF0aC5qb2luKHJ1bl9kaXIsIEFSVElGQUNUX05BTUUpCiAgICAgICAgdG9yY2guc2F2ZSgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgInN0YXRlX2RpY3QiOiBzZWxmLm1vZGVsLnN0YXRlX2RpY3QoKSwKICAgICAgICAgICAgICAgICJoZWFkIjogc2VsZi5tb2RlbC5oZWFkLAogICAgICAgICAgICAgICAgIm5fZmVhdHVyZXMiOiBzZWxmLm1vZGVsLm5fZmVhdHVyZXMsCiAgICAgICAgICAgICAgICAiaGlkZGVuX3NpemVzIjogX2luZmVyX2hpZGRlbl9zaXplcyhzZWxmLm1vZGVsKSwKICAgICAgICAgICAgICAgICJtZWFuIjogc2VsZi5zdGFuZGFyZGl6ZXIubWVhbiwKICAgICAgICAgICAgICAgICJzdGQiOiBzZWxmLnN0YW5kYXJkaXplci5zdGQsCiAgICAgICAgICAgICAgICAidGhyZXNob2xkIjogc2VsZi50aHJlc2hvbGQsCiAgICAgICAgICAgICAgICAiZmVhdHVyZV9jb25maWciOiBfZmVhdHVyZV9jb25maWdfdG9fZGljdChzZWxmLmZlYXR1cmVfY29uZmlnKSwKICAgICAgICAgICAgICAgICJtZXRyaWNzIjogc2VsZi5tZXRyaWNzLAogICAgICAgICAgICB9LAogICAgICAgICAgICBwYXRoLAogICAgICAgICkKICAgICAgICB3aXRoIG9wZW4ob3MucGF0aC5qb2luKHJ1bl9kaXIsIE1FVFJJQ1NfTkFNRSksICJ3IikgYXMgaGFuZGxlOgogICAgICAgICAgICBqc29uLmR1bXAoc2VsZi5tZXRyaWNzLCBoYW5kbGUsIGluZGVudD0yLCBkZWZhdWx0PWZsb2F0KQogICAgICAgIHByaW50KGYiU2F2ZWQgaGVhZCB0byB7cGF0aH0iKQogICAgICAgIHJldHVybiBwYXRoCgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgbG9hZChjbHMsIHJ1bl9kaXI6IHN0ciwgZGV2aWNlOiBzdHIgPSAiY3B1IikgLT4gIlRyYWluZWRIZWFkIjoKICAgICAgICBwYXRoID0gb3MucGF0aC5qb2luKHJ1bl9kaXIsIEFSVElGQUNUX05BTUUpCiAgICAgICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKHBhdGgpOgogICAgICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIk5vIHRyYWluZWQgaGVhZCBhdCB7cGF0aH0iKQoKICAgICAgICBibG9iID0gdG9yY2gubG9hZChwYXRoLCBtYXBfbG9jYXRpb249ZGV2aWNlLCB3ZWlnaHRzX29ubHk9RmFsc2UpCiAgICAgICAgbW9kZWwgPSBWYXJpYW50SGVhZCgKICAgICAgICAgICAgbl9mZWF0dXJlcz1ibG9iWyJuX2ZlYXR1cmVzIl0sCiAgICAgICAgICAgIGhlYWQ9YmxvYlsiaGVhZCJdLAogICAgICAgICAgICBoaWRkZW5fc2l6ZXM9YmxvYi5nZXQoImhpZGRlbl9zaXplcyIsICgyNTYsIDY0KSksCiAgICAgICAgICAgIGRyb3BvdXQ9MC4wLCAgIyBpbmZlcmVuY2U6IGRyb3BvdXQgb2ZmCiAgICAgICAgKQogICAgICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChibG9iWyJzdGF0ZV9kaWN0Il0pCiAgICAgICAgbW9kZWwudG8oZGV2aWNlKS5ldmFsKCkKCiAgICAgICAgcmV0dXJuIGNscygKICAgICAgICAgICAgbW9kZWw9bW9kZWwsCiAgICAgICAgICAgIHN0YW5kYXJkaXplcj1TdGFuZGFyZGl6ZXIoYmxvYlsibWVhbiJdLCBibG9iWyJzdGQiXSksCiAgICAgICAgICAgIHRocmVzaG9sZD1ibG9iWyJ0aHJlc2hvbGQiXSwKICAgICAgICAgICAgZmVhdHVyZV9jb25maWc9X2ZlYXR1cmVfY29uZmlnX2Zyb21fZGljdChibG9iWyJmZWF0dXJlX2NvbmZpZyJdKSwKICAgICAgICAgICAgbWV0cmljcz1ibG9iLmdldCgibWV0cmljcyIsIHt9KSwKICAgICAgICApCgoKZGVmIF9pbmZlcl9oaWRkZW5fc2l6ZXMobW9kZWw6IFZhcmlhbnRIZWFkKToKICAgIGlmIG1vZGVsLmhlYWQgPT0gImxpbmVhciI6CiAgICAgICAgcmV0dXJuICgpCiAgICByZXR1cm4gdHVwbGUoCiAgICAgICAgbGF5ZXIub3V0X2ZlYXR1cmVzCiAgICAgICAgZm9yIGxheWVyIGluIG1vZGVsLm5ldAogICAgICAgIGlmIGlzaW5zdGFuY2UobGF5ZXIsIG5uLkxpbmVhcikKICAgIClbOi0xXQoKCmRlZiBfZmVhdHVyZV9jb25maWdfdG9fZGljdChjZmc6IEZlYXR1cmVDb25maWcpIC0+IERpY3Q6CiAgICByZXR1cm4gewogICAgICAgICJ3aW5kb3dfc2l6ZSI6IGNmZy53aW5kb3dfc2l6ZSwKICAgICAgICAjIEpTT04vdG9yY2ggcm91bmQtdHJpcHMgdHVybiBOb25lIGludG8gbnVsbCBjbGVhbmx5LCB0dXBsZXMgaW50byBsaXN0cy4KICAgICAgICAicG9vbF9yYWRpaSI6IGxpc3QoY2ZnLnBvb2xfcmFkaWkpLAogICAgICAgICJsb2NhbF9yYWRpdXMiOiBjZmcubG9jYWxfcmFkaXVzLAogICAgICAgICJlbWJlZGRpbmdfbGF5ZXIiOiBjZmcuZW1iZWRkaW5nX2xheWVyLAogICAgICAgICJlbWJlZGRpbmdfZGltIjogY2ZnLmVtYmVkZGluZ19kaW0sCiAgICAgICAgInBhaXJfYmF0Y2giOiBjZmcucGFpcl9iYXRjaCwKICAgIH0KCgpkZWYgX2ZlYXR1cmVfY29uZmlnX2Zyb21fZGljdChibG9iOiBEaWN0KSAtPiBGZWF0dXJlQ29uZmlnOgogICAgcmV0dXJuIEZlYXR1cmVDb25maWcoCiAgICAgICAgd2luZG93X3NpemU9YmxvYlsid2luZG93X3NpemUiXSwKICAgICAgICBwb29sX3JhZGlpPXR1cGxlKGJsb2JbInBvb2xfcmFkaWkiXSksCiAgICAgICAgbG9jYWxfcmFkaXVzPWJsb2JbImxvY2FsX3JhZGl1cyJdLAogICAgICAgIGVtYmVkZGluZ19sYXllcj1ibG9iWyJlbWJlZGRpbmdfbGF5ZXIiXSwKICAgICAgICBlbWJlZGRpbmdfZGltPWJsb2JbImVtYmVkZGluZ19kaW0iXSwKICAgICAgICBwYWlyX2JhdGNoPWJsb2IuZ2V0KCJwYWlyX2JhdGNoIiwgVHJ1ZSksCiAgICApCgoKZGVmIGxvYWRfYWN0aXZlX2hlYWQoZGV2aWNlOiBzdHIgPSAiY3B1IikgLT4gT3B0aW9uYWxbVHJhaW5lZEhlYWRdOgogICAgIiIiTG9hZCB0aGUgaGVhZCB0aGUgZW5kcG9pbnQgc2hvdWxkIHNlcnZlLCBvciBgYE5vbmVgYCBpZiBub25lIGlzIHB1Ymxpc2hlZC4KCiAgICBSZXR1cm5pbmcgYGBOb25lYGAgcmF0aGVyIHRoYW4gcmFpc2luZyBpcyBkZWxpYmVyYXRlOiB0aGUgZW5kcG9pbnQgZmFsbHMKICAgIGJhY2sgdG8gemVyby1zaG90IHNjb3Jpbmcgc28gaXQga2VlcHMgd29ya2luZyBiZWZvcmUgdGhlIGZpcnN0IHRyYWluaW5nIHJ1bi4KICAgICIiIgogICAgdHJ5OgogICAgICAgIHJldHVybiBUcmFpbmVkSGVhZC5sb2FkKGNvbmZpZy5BQ1RJVkVfUlVOX0RJUiwgZGV2aWNlPWRldmljZSkKICAgIGV4Y2VwdCBGaWxlTm90Rm91bmRFcnJvcjoKICAgICAgICByZXR1cm4gTm9uZQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6ICAjIG5vcWE6IEJMRTAwMSAtIG5ldmVyIHRha2UgdGhlIGVuZHBvaW50IGRvd24KICAgICAgICBwcmludChmIkNvdWxkIG5vdCBsb2FkIHRyYWluZWQgaGVhZCBmcm9tIHtjb25maWcuQUNUSVZFX1JVTl9ESVJ9OiB7ZXhjfSIpCiAgICAgICAgcmV0dXJuIE5vbmUK",
"finetune/train.py": "IiIiVHJhaW4gdGhlIGNsYXNzaWZpZXIgaGVhZCBvbiBjYWNoZWQgRXZvMiBmZWF0dXJlcyBhbmQgZXZhbHVhdGUgaXQuCgpFdmVyeSBydW4gcmVwb3J0cyB0aGUgemVyby1zaG90IGRlbHRhLWxpa2VsaWhvb2QgQVVST0MgYWxvbmdzaWRlIHRoZSB0cmFpbmVkCmhlYWQncywgb24gdGhlIHNhbWUgcm93cy4gV2l0aG91dCB0aGF0IGJhc2VsaW5lIHRoZXJlIGlzIG5vIHdheSB0byB0ZWxsIHdoZXRoZXIKdGhlIGhlYWQgbGVhcm5lZCBhbnl0aGluZyBiZXlvbmQgd2hhdCB0aHJlc2hvbGRpbmcgdGhlIHJhdyBzY29yZSBhbHJlYWR5IGdhdmUuCiIiIgoKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzaHV0aWwKaW1wb3J0IHRpbWUKZnJvbSB0eXBpbmcgaW1wb3J0IERpY3QsIE9wdGlvbmFsLCBUdXBsZQoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KCmZyb20gLiBpbXBvcnQgY29uZmlnLCBkYXRhLCBmZWF0dXJlcwpmcm9tIC5jb25maWcgaW1wb3J0IFNDQUxBUl9GRUFUVVJFX05BTUVTLCBGZWF0dXJlQ29uZmlnLCBUcmFpbkNvbmZpZwpmcm9tIC5oZWFkIGltcG9ydCBTdGFuZGFyZGl6ZXIsIFRyYWluZWRIZWFkLCBWYXJpYW50SGVhZAoKCmRlZiBfemVyb19zaG90X2NvbHVtbihmZWF0dXJlX2NvbmZpZzogRmVhdHVyZUNvbmZpZykgLT4gaW50OgogICAgIiIiSW5kZXggb2YgYGBkZWx0YV9zY29yZV9mdWxsYGAgd2l0aGluIHRoZSBhc3NlbWJsZWQgZmVhdHVyZSBtYXRyaXguIiIiCiAgICByZXR1cm4gZmVhdHVyZV9jb25maWcubl9lbWJlZGRpbmdfZmVhdHVyZXMgKyBTQ0FMQVJfRkVBVFVSRV9OQU1FUy5pbmRleCgKICAgICAgICAiZGVsdGFfc2NvcmVfZnVsbCIKICAgICkKCgpkZWYgYXNzZW1ibGVfc3BsaXRzKAogICAgZmVhdHVyZV9jb25maWc6IE9wdGlvbmFsW0ZlYXR1cmVDb25maWddID0gTm9uZSwKICAgIGRhdGFzZXRfbmFtZTogc3RyID0gInZhcmlhbnRzIiwKKSAtPiBEaWN0W3N0ciwgVHVwbGVbIm5wLm5kYXJyYXkiLCAibnAubmRhcnJheSJdXToKICAgICIiIkpvaW4gY2FjaGVkIGZlYXR1cmVzIGFnYWluc3QgdGhlIHZhcmlhbnQgdGFibGUgYW5kIGdyb3VwIHRoZW0gYnkgc3BsaXQuIiIiCiAgICBjZmcgPSBmZWF0dXJlX2NvbmZpZyBvciBjb25maWcuREVGQVVMVF9GRUFUVVJFX0NPTkZJRwoKICAgIHZhcmlhbnRfaWRzLCBYLCB5ID0gZmVhdHVyZXMubG9hZF9mZWF0dXJlcyhjZmcpCiAgICB0YWJsZSA9IGRhdGEubG9hZF9kYXRhc2V0KGRhdGFzZXRfbmFtZSkuc2V0X2luZGV4KCJ2YXJpYW50X2lkIikKCiAgICAjIEEgZHVwbGljYXRlZCBpZCB3b3VsZCBtYWtlIHRoZSAubG9jIGxvb2t1cCBiZWxvdyByZXR1cm4gbW9yZSByb3dzIHRoYW4gaXQKICAgICMgd2FzIGFza2VkIGZvciwgc2lsZW50bHkgbWlzYWxpZ25pbmcgZmVhdHVyZXMgYWdhaW5zdCBzcGxpdHMuCiAgICBpZiBub3QgdGFibGUuaW5kZXguaXNfdW5pcXVlOgogICAgICAgIGR1cGxpY2F0ZWQgPSB0YWJsZS5pbmRleFt0YWJsZS5pbmRleC5kdXBsaWNhdGVkKCldLnVuaXF1ZSgpWzo1XS50b2xpc3QoKQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgZiJ7ZGF0YXNldF9uYW1lfS5wYXJxdWV0IGhhcyBkdXBsaWNhdGUgdmFyaWFudF9pZHMsIGUuZy4ge2R1cGxpY2F0ZWR9LiAiCiAgICAgICAgICAgICJSZWJ1aWxkIHRoZSBkYXRhc2V0IGJlZm9yZSB0cmFpbmluZy4iCiAgICAgICAgKQoKICAgIGtub3duID0gdGFibGUuaW5kZXgKICAgIG1hc2sgPSBucC5pc2luKHZhcmlhbnRfaWRzLCBrbm93bikKICAgIGlmIG5vdCBtYXNrLmFsbCgpOgogICAgICAgIHByaW50KAogICAgICAgICAgICBmIldhcm5pbmc6IHtpbnQoKH5tYXNrKS5zdW0oKSl9IGZlYXR1cmUgcm93cyBoYXZlIG5vIG1hdGNoaW5nIHJvdyBpbiAiCiAgICAgICAgICAgIGYie2RhdGFzZXRfbmFtZX0ucGFycXVldCBhbmQgd2VyZSBkcm9wcGVkIgogICAgICAgICkKICAgIHZhcmlhbnRfaWRzLCBYLCB5ID0gdmFyaWFudF9pZHNbbWFza10sIFhbbWFza10sIHlbbWFza10KCiAgICBzcGxpdHMgPSB0YWJsZS5sb2NbdmFyaWFudF9pZHMsICJzcGxpdCJdLnRvX251bXB5KCkKCiAgICBncm91cGVkID0ge30KICAgIGZvciBuYW1lIGluICgidHJhaW4iLCAidmFsIiwgInRlc3QiLCAiYmVuY2htYXJrIik6CiAgICAgICAgaWR4ID0gbnAuZmxhdG5vbnplcm8oc3BsaXRzID09IG5hbWUpCiAgICAgICAgaWYgbGVuKGlkeCk6CiAgICAgICAgICAgIGdyb3VwZWRbbmFtZV0gPSAoWFtpZHhdLCB5W2lkeF0pCiAgICAgICAgICAgIHBvc2l0aXZlcyA9IGludCh5W2lkeF0uc3VtKCkpCiAgICAgICAgICAgIHByaW50KAogICAgICAgICAgICAgICAgZiIgIHtuYW1lOjwxMH0gbj17bGVuKGlkeCk6PDd9IHBvc2l0aXZlPXtwb3NpdGl2ZXN9ICIKICAgICAgICAgICAgICAgIGYiKHtwb3NpdGl2ZXMgLyBsZW4oaWR4KTouMSV9KSIKICAgICAgICAgICAgKQoKICAgIGZvciByZXF1aXJlZCBpbiAoInRyYWluIiwgInZhbCIpOgogICAgICAgIGlmIHJlcXVpcmVkIG5vdCBpbiBncm91cGVkOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgICAgICBmIlNwbGl0IHtyZXF1aXJlZCFyfSBpcyBlbXB0eS4gQ2hlY2sgdGhlIGNocm9tb3NvbWUgYXNzaWdubWVudHMgIgogICAgICAgICAgICAgICAgImluIFRyYWluQ29uZmlnIGFnYWluc3QgdGhlIHZhcmlhbnRzIHlvdSBleHRyYWN0ZWQuIgogICAgICAgICAgICApCiAgICAgICAgIyBFYXJseSBzdG9wcGluZyBhbmQgdGhyZXNob2xkIHNlbGVjdGlvbiBib3RoIG5lZWQgdHdvIGNsYXNzZXM7IGEKICAgICAgICAjIHNpbmdsZS1jbGFzcyBzcGxpdCB3b3VsZCBnaXZlIGEgTmFOIEFVUk9DIGFuZCBubyB1c2FibGUgdGhyZXNob2xkLgogICAgICAgIGlmIGxlbihucC51bmlxdWUoZ3JvdXBlZFtyZXF1aXJlZF1bMV0pKSA8IDI6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgICAgIGYiU3BsaXQge3JlcXVpcmVkIXJ9IGNvbnRhaW5zIG9ubHkgb25lIGNsYXNzLiBQaWNrIGRpZmZlcmVudCAiCiAgICAgICAgICAgICAgICAidmFsL3Rlc3QgY2hyb21vc29tZXMgaW4gVHJhaW5Db25maWcuIgogICAgICAgICAgICApCiAgICByZXR1cm4gZ3JvdXBlZAoKCmRlZiBfbWV0cmljcyh5X3RydWUsIHNjb3JlcywgdGhyZXNob2xkOiBPcHRpb25hbFtmbG9hdF0gPSBOb25lKSAtPiBEaWN0W3N0ciwgZmxvYXRdOgogICAgZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0ICgKICAgICAgICBhY2N1cmFjeV9zY29yZSwKICAgICAgICBhdmVyYWdlX3ByZWNpc2lvbl9zY29yZSwKICAgICAgICByb2NfYXVjX3Njb3JlLAogICAgKQoKICAgICMgQSBzcGxpdCB3aXRoIGEgc2luZ2xlIGNsYXNzIG1ha2VzIEFVUk9DIHVuZGVmaW5lZC4KICAgIGlmIGxlbihucC51bmlxdWUoeV90cnVlKSkgPCAyOgogICAgICAgIHJldHVybiB7ImF1cm9jIjogZmxvYXQoIm5hbiIpLCAiYXVwcmMiOiBmbG9hdCgibmFuIiksICJuIjogaW50KGxlbih5X3RydWUpKX0KCiAgICBvdXQgPSB7CiAgICAgICAgImF1cm9jIjogZmxvYXQocm9jX2F1Y19zY29yZSh5X3RydWUsIHNjb3JlcykpLAogICAgICAgICJhdXByYyI6IGZsb2F0KGF2ZXJhZ2VfcHJlY2lzaW9uX3Njb3JlKHlfdHJ1ZSwgc2NvcmVzKSksCiAgICAgICAgIm4iOiBpbnQobGVuKHlfdHJ1ZSkpLAogICAgfQogICAgaWYgdGhyZXNob2xkIGlzIG5vdCBOb25lOgogICAgICAgIHByZWRpY3RlZCA9IChzY29yZXMgPj0gdGhyZXNob2xkKS5hc3R5cGUoaW50KQogICAgICAgIG91dFsiYWNjdXJhY3kiXSA9IGZsb2F0KGFjY3VyYWN5X3Njb3JlKHlfdHJ1ZSwgcHJlZGljdGVkKSkKICAgICAgICBvdXRbInRocmVzaG9sZCJdID0gZmxvYXQodGhyZXNob2xkKQogICAgcmV0dXJuIG91dAoKCmRlZiBfeW91ZGVuX3RocmVzaG9sZCh5X3RydWUsIHNjb3JlcykgLT4gZmxvYXQ6CiAgICAiIiJUaHJlc2hvbGQgbWF4aW1pc2luZyBUUFIgLSBGUFIsIHRoZSBjcml0ZXJpb24gdGhlIHplcm8tc2hvdCBwYXRoIHVzZWQuIiIiCiAgICBmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgcm9jX2N1cnZlCgogICAgZnByLCB0cHIsIHRocmVzaG9sZHMgPSByb2NfY3VydmUoeV90cnVlLCBzY29yZXMpCiAgICByZXR1cm4gZmxvYXQodGhyZXNob2xkc1tucC5hcmdtYXgodHByIC0gZnByKV0pCgoKZGVmIHRyYWluX2hlYWQoCiAgICBmZWF0dXJlX2NvbmZpZzogT3B0aW9uYWxbRmVhdHVyZUNvbmZpZ10gPSBOb25lLAogICAgdHJhaW5fY29uZmlnOiBPcHRpb25hbFtUcmFpbkNvbmZpZ10gPSBOb25lLAogICAgZGF0YXNldF9uYW1lOiBzdHIgPSAidmFyaWFudHMiLAogICAgcnVuX25hbWU6IE9wdGlvbmFsW3N0cl0gPSBOb25lLAopIC0+IFRyYWluZWRIZWFkOgogICAgZmVhdHVyZV9jZmcgPSBmZWF0dXJlX2NvbmZpZyBvciBjb25maWcuREVGQVVMVF9GRUFUVVJFX0NPTkZJRwogICAgY2ZnID0gdHJhaW5fY29uZmlnIG9yIGNvbmZpZy5ERUZBVUxUX1RSQUlOX0NPTkZJRwoKICAgIHRvcmNoLm1hbnVhbF9zZWVkKGNmZy5zZWVkKQogICAgbnAucmFuZG9tLnNlZWQoY2ZnLnNlZWQpCgogICAgcHJpbnQoIkFzc2VtYmxpbmcgc3BsaXRzIC4uLiIpCiAgICBzcGxpdHMgPSBhc3NlbWJsZV9zcGxpdHMoZmVhdHVyZV9jZmcsIGRhdGFzZXRfbmFtZSkKICAgIFhfdHJhaW4sIHlfdHJhaW4gPSBzcGxpdHNbInRyYWluIl0KICAgIFhfdmFsLCB5X3ZhbCA9IHNwbGl0c1sidmFsIl0KCiAgICBzdGFuZGFyZGl6ZXIgPSBTdGFuZGFyZGl6ZXIuZml0KFhfdHJhaW4pCiAgICBkZXZpY2UgPSAiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiCiAgICBwcmludChmIlRyYWluaW5nIG9uIHtkZXZpY2V9IHdpdGgge1hfdHJhaW4uc2hhcGVbMV19IGZlYXR1cmVzIikKCiAgICBkZWYgdG9fdGVuc29ycyhYLCB5KToKICAgICAgICByZXR1cm4gKAogICAgICAgICAgICB0b3JjaC5mcm9tX251bXB5KHN0YW5kYXJkaXplci50cmFuc2Zvcm0oWCkpLnRvKGRldmljZSksCiAgICAgICAgICAgIHRvcmNoLmZyb21fbnVtcHkoeS5hc3R5cGUobnAuZmxvYXQzMikpLnRvKGRldmljZSksCiAgICAgICAgKQoKICAgIFh0LCB5dCA9IHRvX3RlbnNvcnMoWF90cmFpbiwgeV90cmFpbikKICAgIFh2LCB5diA9IHRvX3RlbnNvcnMoWF92YWwsIHlfdmFsKQoKICAgIG1vZGVsID0gVmFyaWFudEhlYWQoCiAgICAgICAgbl9mZWF0dXJlcz1YX3RyYWluLnNoYXBlWzFdLAogICAgICAgIGhlYWQ9Y2ZnLmhlYWQsCiAgICAgICAgaGlkZGVuX3NpemVzPWNmZy5oaWRkZW5fc2l6ZXMsCiAgICAgICAgZHJvcG91dD1jZmcuZHJvcG91dCwKICAgICkudG8oZGV2aWNlKQoKICAgIHBvc193ZWlnaHQgPSBOb25lCiAgICBpZiBjZmcuY2xhc3Nfd2VpZ2h0aW5nOgogICAgICAgIG5fcG9zID0gZmxvYXQoeV90cmFpbi5zdW0oKSkKICAgICAgICBuX25lZyA9IGZsb2F0KGxlbih5X3RyYWluKSAtIG5fcG9zKQogICAgICAgIGlmIG5fcG9zID4gMCBhbmQgbl9uZWcgPiAwOgogICAgICAgICAgICBwb3Nfd2VpZ2h0ID0gdG9yY2gudGVuc29yKFtuX25lZyAvIG5fcG9zXSwgZGV2aWNlPWRldmljZSkKICAgICAgICAgICAgcHJpbnQoZiJDbGFzcyB3ZWlnaHRpbmc6IHBvc193ZWlnaHQ9e3Bvc193ZWlnaHQuaXRlbSgpOi4zZn0iKQoKICAgIGNyaXRlcmlvbiA9IG5uLkJDRVdpdGhMb2dpdHNMb3NzKHBvc193ZWlnaHQ9cG9zX3dlaWdodCkKICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW1XKAogICAgICAgIG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9Y2ZnLmxyLCB3ZWlnaHRfZGVjYXk9Y2ZnLndlaWdodF9kZWNheQogICAgKQoKICAgIGJlc3RfYXVyb2MsIGJlc3Rfc3RhdGUsIGJlc3RfZXBvY2ggPSAtMS4wLCBOb25lLCAtMQogICAgc3RhcnRlZCA9IHRpbWUudGltZSgpCgogICAgZm9yIGVwb2NoIGluIHJhbmdlKGNmZy5tYXhfZXBvY2hzKToKICAgICAgICBtb2RlbC50cmFpbigpCiAgICAgICAgb3JkZXIgPSB0b3JjaC5yYW5kcGVybShsZW4oWHQpLCBkZXZpY2U9ZGV2aWNlKQogICAgICAgIGVwb2NoX2xvc3MgPSAwLjAKICAgICAgICBmb3Igc3RhcnQgaW4gcmFuZ2UoMCwgbGVuKG9yZGVyKSwgY2ZnLmJhdGNoX3NpemUpOgogICAgICAgICAgICBiYXRjaCA9IG9yZGVyW3N0YXJ0OnN0YXJ0ICsgY2ZnLmJhdGNoX3NpemVdCiAgICAgICAgICAgIG9wdGltaXplci56ZXJvX2dyYWQoKQogICAgICAgICAgICBsb3NzID0gY3JpdGVyaW9uKG1vZGVsKFh0W2JhdGNoXSksIHl0W2JhdGNoXSkKICAgICAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgICAgIG9wdGltaXplci5zdGVwKCkKICAgICAgICAgICAgZXBvY2hfbG9zcyArPSBsb3NzLml0ZW0oKSAqIGxlbihiYXRjaCkKCiAgICAgICAgbW9kZWwuZXZhbCgpCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIHZhbF9zY29yZXMgPSB0b3JjaC5zaWdtb2lkKG1vZGVsKFh2KSkuY3B1KCkubnVtcHkoKQogICAgICAgIHZhbF9hdXJvYyA9IF9tZXRyaWNzKHlfdmFsLCB2YWxfc2NvcmVzKVsiYXVyb2MiXQoKICAgICAgICBpZiB2YWxfYXVyb2MgPiBiZXN0X2F1cm9jOgogICAgICAgICAgICBiZXN0X2F1cm9jID0gdmFsX2F1cm9jCiAgICAgICAgICAgIGJlc3Rfc3RhdGUgPSB7azogdi5kZXRhY2goKS5jbG9uZSgpIGZvciBrLCB2IGluIG1vZGVsLnN0YXRlX2RpY3QoKS5pdGVtcygpfQogICAgICAgICAgICBiZXN0X2Vwb2NoID0gZXBvY2gKCiAgICAgICAgaWYgZXBvY2ggJSA1ID09IDAgb3IgZXBvY2ggPT0gY2ZnLm1heF9lcG9jaHMgLSAxOgogICAgICAgICAgICBwcmludCgKICAgICAgICAgICAgICAgIGYiICBlcG9jaCB7ZXBvY2g6PjN9IGxvc3M9e2Vwb2NoX2xvc3MgLyBsZW4ob3JkZXIpOi40Zn0gIgogICAgICAgICAgICAgICAgZiJ2YWxfYXVyb2M9e3ZhbF9hdXJvYzouNGZ9IChiZXN0IHtiZXN0X2F1cm9jOi40Zn0gQCB7YmVzdF9lcG9jaH0pIgogICAgICAgICAgICApCgogICAgICAgIGlmIGVwb2NoIC0gYmVzdF9lcG9jaCA+PSBjZmcucGF0aWVuY2U6CiAgICAgICAgICAgIHByaW50KGYiICBlYXJseSBzdG9wIGF0IGVwb2NoIHtlcG9jaH0gKG5vIGdhaW4gZm9yIHtjZmcucGF0aWVuY2V9IGVwb2NocykiKQogICAgICAgICAgICBicmVhawoKICAgIGlmIGJlc3Rfc3RhdGUgaXMgbm90IE5vbmU6CiAgICAgICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KGJlc3Rfc3RhdGUpCiAgICBwcmludChmIlRyYWluZWQgaW4ge3RpbWUudGltZSgpIC0gc3RhcnRlZDouMWZ9czsgYmVzdCB2YWwgQVVST0Mge2Jlc3RfYXVyb2M6LjRmfSIpCgogICAgIyBUaHJlc2hvbGQgaXMgZml0dGVkIG9uIHZhbGlkYXRpb24sIG5ldmVyIG9uIHRlc3QuCiAgICBtb2RlbC5ldmFsKCkKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIHZhbF9zY29yZXMgPSB0b3JjaC5zaWdtb2lkKG1vZGVsKFh2KSkuY3B1KCkubnVtcHkoKQogICAgdGhyZXNob2xkID0gX3lvdWRlbl90aHJlc2hvbGQoeV92YWwsIHZhbF9zY29yZXMpCiAgICBwcmludChmIkRlY2lzaW9uIHRocmVzaG9sZCAoWW91ZGVuJ3MgSiBvbiB2YWwpOiB7dGhyZXNob2xkOi40Zn0iKQoKICAgIHRyYWluZWQgPSBUcmFpbmVkSGVhZCgKICAgICAgICBtb2RlbD1tb2RlbCwKICAgICAgICBzdGFuZGFyZGl6ZXI9c3RhbmRhcmRpemVyLAogICAgICAgIHRocmVzaG9sZD10aHJlc2hvbGQsCiAgICAgICAgZmVhdHVyZV9jb25maWc9ZmVhdHVyZV9jZmcsCiAgICApCgogICAgIyAtLS0gZXZhbHVhdGlvbiwgdHJhaW5lZCBoZWFkIHZzIHRoZSB6ZXJvLXNob3Qgc2NvcmUgaXQgcmVwbGFjZXMKICAgIHplcm9fc2hvdF9jb2wgPSBfemVyb19zaG90X2NvbHVtbihmZWF0dXJlX2NmZykKICAgIHJlcG9ydDogRGljdFtzdHIsIERpY3RdID0ge30KICAgIGZvciBuYW1lLCAoWCwgeSkgaW4gc3BsaXRzLml0ZW1zKCk6CiAgICAgICAgcHJvYnMgPSB0cmFpbmVkLnByZWRpY3RfcHJvYmEoWCkKICAgICAgICAjIFBhdGhvZ2VuaWMgdmFyaWFudHMgaGF2ZSBhICptb3JlIG5lZ2F0aXZlKiBkZWx0YSBsaWtlbGlob29kLCBzbyB0aGUKICAgICAgICAjIGJhc2VsaW5lIHNjb3JlIGlzIG5lZ2F0ZWQgdG8gcG9pbnQgdGhlIHNhbWUgd2F5IGFzIHRoZSBoZWFkJ3Mgb3V0cHV0LgogICAgICAgIGJhc2VsaW5lID0gLVhbOiwgemVyb19zaG90X2NvbF0KICAgICAgICByZXBvcnRbbmFtZV0gPSB7CiAgICAgICAgICAgICJoZWFkIjogX21ldHJpY3MoeSwgcHJvYnMsIHRocmVzaG9sZCksCiAgICAgICAgICAgICJ6ZXJvX3Nob3QiOiBfbWV0cmljcyh5LCBiYXNlbGluZSksCiAgICAgICAgfQogICAgICAgIHByaW50KAogICAgICAgICAgICBmIntuYW1lOjwxMH0gaGVhZCBBVVJPQz17cmVwb3J0W25hbWVdWydoZWFkJ11bJ2F1cm9jJ106LjRmfSAgIgogICAgICAgICAgICBmInplcm8tc2hvdCBBVVJPQz17cmVwb3J0W25hbWVdWyd6ZXJvX3Nob3QnXVsnYXVyb2MnXTouNGZ9IgogICAgICAgICkKCiAgICB0cmFpbmVkLm1ldHJpY3MgPSB7CiAgICAgICAgInNwbGl0cyI6IHJlcG9ydCwKICAgICAgICAiYmVzdF92YWxfYXVyb2MiOiBiZXN0X2F1cm9jLAogICAgICAgICJiZXN0X2Vwb2NoIjogYmVzdF9lcG9jaCwKICAgICAgICAibl9mZWF0dXJlcyI6IGludChYX3RyYWluLnNoYXBlWzFdKSwKICAgICAgICAidHJhaW5fY29uZmlnIjogdmFycyhjZmcpLAogICAgICAgICJmZWF0dXJlX3RhZyI6IGZlYXR1cmVfY2ZnLnRhZygpLAogICAgfQoKICAgIHJ1biA9IHJ1bl9uYW1lIG9yIHRpbWUuc3RyZnRpbWUoInJ1bi0lWSVtJWQtJUglTSVTIikKICAgIHJ1bl9kaXIgPSBmIntjb25maWcuUlVOU19ESVJ9L3tydW59IgogICAgdHJhaW5lZC5zYXZlKHJ1bl9kaXIpCiAgICBwcmludChmIlJ1biBkaXJlY3Rvcnk6IHtydW5fZGlyfSIpCiAgICByZXR1cm4gdHJhaW5lZAoKCmRlZiBwdWJsaXNoKHJ1bl9uYW1lOiBzdHIpIC0+IHN0cjoKICAgICIiIlBvaW50IHRoZSBpbmZlcmVuY2UgZW5kcG9pbnQgYXQgYSBmaW5pc2hlZCBydW4uCgogICAgQ29waWVzIHJhdGhlciB0aGFuIHN5bWxpbmtzIHNvIHRoZSBlbmRwb2ludCBrZWVwcyBzZXJ2aW5nIGEgY29oZXJlbnQKICAgIGNoZWNrcG9pbnQgZXZlbiBpZiB0aGUgc291cmNlIHJ1biBkaXJlY3RvcnkgaXMgbGF0ZXIgZGVsZXRlZC4KICAgICIiIgogICAgc291cmNlID0gZiJ7Y29uZmlnLlJVTlNfRElSfS97cnVuX25hbWV9IgogICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKG9zLnBhdGguam9pbihzb3VyY2UsICJoZWFkLnB0IikpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiTm8gaGVhZC5wdCBpbiB7c291cmNlfSIpCgogICAgdGFyZ2V0ID0gY29uZmlnLkFDVElWRV9SVU5fRElSCiAgICBvcy5tYWtlZGlycyhvcy5wYXRoLmRpcm5hbWUodGFyZ2V0KSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGlmIG9zLnBhdGguZXhpc3RzKHRhcmdldCk6CiAgICAgICAgc2h1dGlsLnJtdHJlZSh0YXJnZXQpCiAgICBzaHV0aWwuY29weXRyZWUoc291cmNlLCB0YXJnZXQpCgogICAgd2l0aCBvcGVuKG9zLnBhdGguam9pbih0YXJnZXQsICJQVUJMSVNIRURfRlJPTSIpLCAidyIpIGFzIGhhbmRsZToKICAgICAgICBoYW5kbGUud3JpdGUoZiJ7cnVuX25hbWV9XG57dGltZS5zdHJmdGltZSgnJVktJW0tJWQgJUg6JU06JVMnKX1cbiIpCgogICAgcHJpbnQoZiJQdWJsaXNoZWQge3J1bl9uYW1lfSB0byB7dGFyZ2V0fTsgdGhlIGVuZHBvaW50IHdpbGwgc2VydmUgaXQgb24gbmV4dCBzdGFydCIpCiAgICByZXR1cm4gdGFyZ2V0CgoKZGVmIHN1bW1hcmlzZV9ydW5zKCkgLT4gTm9uZToKICAgICIiIlByaW50IHRoZSBtZXRyaWNzIG9mIGV2ZXJ5IGNvbXBsZXRlZCBydW4uIiIiCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoY29uZmlnLlJVTlNfRElSKToKICAgICAgICBwcmludCgiTm8gcnVucyB5ZXQiKQogICAgICAgIHJldHVybgoKICAgIGZvciBydW4gaW4gc29ydGVkKG9zLmxpc3RkaXIoY29uZmlnLlJVTlNfRElSKSk6CiAgICAgICAgbWV0cmljc19wYXRoID0gZiJ7Y29uZmlnLlJVTlNfRElSfS97cnVufS9tZXRyaWNzLmpzb24iCiAgICAgICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKG1ldHJpY3NfcGF0aCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgd2l0aCBvcGVuKG1ldHJpY3NfcGF0aCkgYXMgaGFuZGxlOgogICAgICAgICAgICBtZXRyaWNzID0ganNvbi5sb2FkKGhhbmRsZSkKICAgICAgICBwcmludChmIlxue3J1bn0iKQogICAgICAgIGZvciBzcGxpdCwgZW50cnkgaW4gbWV0cmljcy5nZXQoInNwbGl0cyIsIHt9KS5pdGVtcygpOgogICAgICAgICAgICBwcmludCgKICAgICAgICAgICAgICAgIGYiICB7c3BsaXQ6PDEwfSBoZWFkPXtlbnRyeVsnaGVhZCddWydhdXJvYyddOi40Zn0gIgogICAgICAgICAgICAgICAgZiJ6ZXJvLXNob3Q9e2VudHJ5Wyd6ZXJvX3Nob3QnXVsnYXVyb2MnXTouNGZ9IG49e2VudHJ5WydoZWFkJ11bJ24nXX0iCiAgICAgICAgICAgICkK",
"finetune/loader.py": "IiIiQ29uc3RydWN0aW5nIEV2bzIgdW5kZXIgUHlUb3JjaCdzIGBgd2VpZ2h0c19vbmx5YGAgdW5waWNrbGVyLgoKTG9hZGluZyB0aGUgcHVibGlzaGVkIGV2bzIgY2hlY2twb2ludCB0cmlwcyBgYHdlaWdodHNfb25seT1UcnVlYGAgaW4gdHdvCmluZGVwZW5kZW50IHBsYWNlcywgbmVpdGhlciBvZiB0aGVtIG91cnM6CgoqIGBgdm9ydGV4Lm1vZGVsLnV0aWxzLmxvYWRfY2hlY2twb2ludGBgIHBhc3NlcyBgYHdlaWdodHNfb25seT1UcnVlYGAKICBleHBsaWNpdGx5LCBzbyB0aGlzIGlzIG5vdCBzb21ldGhpbmcgYSB0b3JjaCB2ZXJzaW9uIHBpbiBjYW4gYXZvaWQuCiogYGB0cmFuc2Zvcm1lcl9lbmdpbmVgYCdzIGBgc2V0X2V4dHJhX3N0YXRlYGAgY2FsbHMgYGB0b3JjaC5sb2FkYGAgb24gZWFjaAogIG1vZHVsZSdzIHNlcmlhbGlzZWQgZXh0cmEgc3RhdGUgZHVyaW5nIGBgbG9hZF9zdGF0ZV9kaWN0YGAuCgpUaGUgY2hlY2twb2ludCBsZWdpdGltYXRlbHkgY29udGFpbnMgb2JqZWN0cyBvdXRzaWRlIHRoZSBkZWZhdWx0IGFsbG93bGlzdAooYGBfY29kZWNzLmVuY29kZWBgLCBgYHRyYW5zZm9ybWVyX2VuZ2luZS5jb21tb24ucmVjaXBlLl9PdmVycmlkZUxpbmVhclByZWNpc2lvbmBgLAphbmQgaG93ZXZlciBtYW55IG1vcmUgc2l0IGJlaGluZCB0aG9zZSksIGFuZCB3aGljaCBvbmUgc3VyZmFjZXMgZmlyc3Qgc2hpZnRzCndpdGggdGhlIHRvcmNoIHZlcnNpb24sIHNvIGFsbG93bGlzdGluZyB0aGVtIG9uZSBhdCBhIHRpbWUgaXMgYSB0cmVhZG1pbGwuCgpgYHdlaWdodHNfb25seWBgIGV4aXN0cyB0byBzdG9wIGFuICp1bnRydXN0ZWQqIHBpY2tsZSBmcm9tIGV4ZWN1dGluZyBjb2RlIG9uCmxvYWQuIFRoZXNlIHdlaWdodHMgY29tZSBmcm9tIEFyYyBJbnN0aXR1dGUncyBvZmZpY2lhbCBIdWdnaW5nIEZhY2UgcmVwbywgd2hpY2gKaXMgdGhlIHNhbWUgYXJ0aWZhY3QgdGhlIHJlc3Qgb2YgdGhlIHBpcGVsaW5lIGlzIGJ1aWx0IGFyb3VuZCBhbmQgaXMgYWxyZWFkeQp0cnVzdGVkIHRvIHJ1biBhcyBtb2RlbCBjb2RlIOKAlCBzbyB0aGUgdHJ1c3QgY29uZGl0aW9uIGlzIG1ldCBhbmQgdGhlIGNoZWNrIGlzCnJlbGF4ZWQgZm9yIHRoZSBkdXJhdGlvbiBvZiB0aGUgbG9hZCBvbmx5LgoiIiIKCmltcG9ydCBjb250ZXh0bGliCmltcG9ydCBpbXBvcnRsaWIKCiMgQnVtcCBvbiBldmVyeSBjaGFuZ2UgdG8gdGhpcyBmaWxlLiBQcmludGVkIGJ5IGxvYWRfZXZvMiBzbyBhIHJ1bidzIGxvZyBtYWtlcwojIGNsZWFyIHdoaWNoIHZlcnNpb24gb2YgbG9hZGVyLnB5IGlzIGFjdHVhbGx5IGVtYmVkZGVkIGluIHRoZSBub3RlYm9vayAtLSBLYWdnbGUKIyBoYXMgcmVwZWF0ZWRseSBydW4gYSBzdGFsZSBjb3B5IGFmdGVyIGEgcmUtdXBsb2FkLgpMT0FERVJfUkVWSVNJT04gPSA3CgoKQGNvbnRleHRsaWIuY29udGV4dG1hbmFnZXIKZGVmIHRydXN0ZWRfdG9yY2hfbG9hZCgpOgogICAgIiIiRm9yY2UgYGB3ZWlnaHRzX29ubHk9RmFsc2VgYCBvbiBldmVyeSBgYHRvcmNoLmxvYWRgYCBpbnNpZGUgdGhlIGJsb2NrLgoKICAgIFBhdGNoaW5nIHRoZSBtb2R1bGUgYXR0cmlidXRlIGlzIHdoYXQgbWFrZXMgdGhpcyByZWFjaCBgYHRvcmNoLmxvYWRgYCBjYWxscwogICAgaW5zaWRlIHZvcnRleCBhbmQgdHJhbnNmb3JtZXJfZW5naW5lLCB3aGljaCB0YWtlIG5vIGFyZ3VtZW50cyBmcm9tIHVzLiBUaGUKICAgIG9yaWdpbmFsIGlzIGFsd2F5cyByZXN0b3JlZCwgc28gbm90aGluZyBvdXRzaWRlIHRoZSBibG9jayBpcyBhZmZlY3RlZC4KICAgICIiIgogICAgaW1wb3J0IHRvcmNoCgogICAgb3JpZ2luYWwgPSB0b3JjaC5sb2FkCgogICAgZGVmIHBhdGNoZWQoKmFyZ3MsICoqa3dhcmdzKToKICAgICAgICBrd2FyZ3NbIndlaWdodHNfb25seSJdID0gRmFsc2UKICAgICAgICByZXR1cm4gb3JpZ2luYWwoKmFyZ3MsICoqa3dhcmdzKQoKICAgIHRvcmNoLmxvYWQgPSBwYXRjaGVkCiAgICB0cnk6CiAgICAgICAgeWllbGQKICAgIGZpbmFsbHk6CiAgICAgICAgdG9yY2gubG9hZCA9IG9yaWdpbmFsCgoKZGVmIF9ncHVfc3VwcG9ydHNfZnA4KCkgLT4gYm9vbDoKICAgIGltcG9ydCB0b3JjaAoKICAgIGlmIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHJldHVybiBGYWxzZQogICAgcmV0dXJuIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KCkgPj0gKDgsIDkpCgoKZGVmIF9ncHVfc3VwcG9ydHNfYmYxNigpIC0+IGJvb2w6CiAgICBpbXBvcnQgdG9yY2gKCiAgICBpZiBub3QgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICByZXR1cm4gRmFsc2UKICAgIHJldHVybiB0b3JjaC5jdWRhLmdldF9kZXZpY2VfY2FwYWJpbGl0eSgpID49ICg4LCAwKQoKCmRlZiBfY2FzdF90b19mcDE2KGV2bzJfbW9kZWwpIC0+IE5vbmU6CiAgICAiIiJSdW4gdGhlIG1vZGVsIGluIGZsb2F0MTYgYmVjYXVzZSB0aGUgR1BVIGhhcyBubyBiZmxvYXQxNi4KCiAgICBgYGV2bzJfN2JgYCBsb2FkcyBpbiBiZmxvYXQxNi4gT24gYSBwcmUtQW1wZXJlIEdQVSAoY29tcHV0ZSBjYXBhYmlsaXR5IDwgOC4wLAogICAgZS5nLiBhIFQ0IGF0IDcuNSkgYGBwdHhhc2BgIHJlamVjdHMgdGhlIGBgLmJmMTZgYCBQVFggdGhhdCB2b3J0ZXgncyBUcml0b24KICAgIGtlcm5lbHMgKHJvdGFyeSBlbWJlZGRpbmcsIGV0Yy4pIGVtaXQgZm9yIGJmMTYgdGVuc29yczoKCiAgICAgICAgRmVhdHVyZSAnLmJmMTYnIHJlcXVpcmVzIC50YXJnZXQgc21fODAgb3IgaGlnaGVyCgogICAgQ2FzdGluZyB0aGUgd2hvbGUgbW9kdWxlIHRvIGZsb2F0MTYgbWFrZXMgdGhvc2Uga2VybmVscyBjb21waWxlIHRoZSBgYC5mMTZgYAogICAgcGF0aCBpbnN0ZWFkLCB3aGljaCBldmVyeSBDVURBIEdQVSBzaW5jZSBQYXNjYWwgc3VwcG9ydHMuIGZsb2F0MTYgaGFzIGJmMTYncwogICAgcHJlY2lzaW9uIGJ1dCBhIG11Y2ggc21hbGxlciBleHBvbmVudCByYW5nZSwgc28gdGhpcyBjYW4gb3ZlcmZsb3cgdG8gaW5mL25hbgogICAgd2hlcmUgYmYxNiB3b3VsZCBub3QgLS0gYW5vdGhlciByZWFzb24gdGhpcyBpcyBhIGxhc3QtcmVzb3J0IGZhbGxiYWNrLCBvbmx5CiAgICB0YWtlbiBvbiBHUFVzIHRoYXQgY2Fubm90IHJ1biB0aGUgbW9kZWwgYW55IG90aGVyIHdheS4KICAgICIiIgogICAgZXZvMl9tb2RlbC5tb2RlbC5oYWxmKCkKCgpkZWYgX25ldXRlcl9mcDhfYXV0b2Nhc3QoKSAtPiBOb25lOgogICAgIiIiTWFrZSBgYHRlLmZwOF9hdXRvY2FzdGBgIGEgbm8tb3AgY29udGV4dCBtYW5hZ2VyLCBwcm9jZXNzLXdpZGUuCgogICAgYGBldm8yXzdiYGAncyBjaGVja3BvaW50IGNvbmZpZyBzZXRzIGBgdXNlX2ZwOF9pbnB1dF9wcm9qZWN0aW9ucz1UcnVlYGAuIEZQOAogICAgZXhlY3V0aW9uIG5lZWRzIGFuIEFkYS9Ib3BwZXIgR1BVIChjb21wdXRlIGNhcGFiaWxpdHkgPj0gOC45KTsgb24gYSBUNAogICAgKENDIDcuNSkgdGhlIGBgd2l0aCB0ZS5mcDhfYXV0b2Nhc3QoZW5hYmxlZD1UcnVlLCAuLi4pYGAgaW4KICAgIGBgdm9ydGV4Lm1vZGVsLmxheWVyc2BgIGFzc2VydHMgYXQgZm9yd2FyZCB0aW1lLiBUaGUgYm9keSBvZiB0aGF0IGBgd2l0aGBgCiAgICBpcyBgYG91dCA9IHN1cGVyKCkuZm9yd2FyZCh4KWBgIC0tIGFuIG9yZGluYXJ5IGBgbm4uTGluZWFyYGAgY2FsbCAtLSBzbyBpZgogICAgdGhlIGNvbnRleHQgbWFuYWdlciBzaW1wbHkgZG9lcyBub3RoaW5nLCB0aGUgcHJvamVjdGlvbiBydW5zIGluIHRoZSBtb2RlbCdzCiAgICBub3JtYWwgcHJlY2lzaW9uIGFuZCBldmVyeXRoaW5nIGRvd25zdHJlYW0gd29ya3MuCgogICAgUGF0Y2hpbmcgcGVyLW1vZHVsZSBgYHVzZV9mcDhfaW5wdXRfcHJvamVjdGlvbnNgYCBmbGFncyBwcm92ZWQgdW5yZWxpYWJsZQogICAgKHRoZSBmbGFnIGxpdmVzIG9uIGEgY2xvc3VyZS1kZWZpbmVkIGNsYXNzIGFuZCB0aGUgd2FsayBjYW4gbWlzcyBpdCk7CiAgICByZXBsYWNpbmcgdGhlIG9uZSBmdW5jdGlvbiBldmVyeSBGUDggcGF0aCBmdW5uZWxzIHRocm91Z2ggY2Fubm90IGJlIG1pc3NlZC4KICAgIHZvcnRleCBkb2VzIGBgaW1wb3J0IHRyYW5zZm9ybWVyX2VuZ2luZS5weXRvcmNoIGFzIHRlYGAgYW5kIGNhbGxzCiAgICBgYHRlLmZwOF9hdXRvY2FzdGBgIGJ5IGF0dHJpYnV0ZSwgc28gb3ZlcndyaXRpbmcgdGhlIG1vZHVsZSBhdHRyaWJ1dGUgdGFrZXMKICAgIGVmZmVjdCBmb3IgY2FsbHMgbWFkZSBhZnRlciB0aGlzIHJ1bnMuCgogICAgVGhpcyBpcyBhbiBhcHByb3hpbWF0aW9uIG9mIHRoZSBpbnRlbmRlZCBudW1lcmljcyAtLSB0aGUgcHJvamVjdGlvbiB3ZWlnaHRzCiAgICB3ZXJlIGNhbGlicmF0ZWQgZm9yIEZQOCAtLSBub3QgYSBiaXQtZXhhY3Qgc3Vic3RpdHV0ZS4gT25seSBhcHBsaWVkIG9uIEdQVXMKICAgIHRoYXQgY2Fubm90IHJ1biB0aGUgbW9kZWwgYW55IG90aGVyIHdheS4KICAgICIiIgogICAgaW1wb3J0IHRyYW5zZm9ybWVyX2VuZ2luZS5weXRvcmNoIGFzIHRlCgogICAgaWYgZ2V0YXR0cih0ZS5mcDhfYXV0b2Nhc3QsICJfZXZvMl9uZXV0ZXJlZCIsIEZhbHNlKToKICAgICAgICByZXR1cm4KCiAgICBAY29udGV4dGxpYi5jb250ZXh0bWFuYWdlcgogICAgZGVmIF9ub29wX2ZwOF9hdXRvY2FzdCgqYXJncywgKiprd2FyZ3MpOgogICAgICAgIHlpZWxkCgogICAgX25vb3BfZnA4X2F1dG9jYXN0Ll9ldm8yX25ldXRlcmVkID0gVHJ1ZQogICAgdGUuZnA4X2F1dG9jYXN0ID0gX25vb3BfZnA4X2F1dG9jYXN0CgoKZGVmIF9wYXRjaF9mbGFzaF9hdHRlbnRpb25fdG9fc2RwYSgpIC0+IE5vbmU6CiAgICAiIiJSb3V0ZSB2b3J0ZXgncyBhdHRlbnRpb24gdGhyb3VnaCB0b3JjaCBTRFBBIGluc3RlYWQgb2YgRmxhc2hBdHRlbnRpb24tMi4KCiAgICBGbGFzaEF0dGVudGlvbi0yICh0aGUgYGBmbGFzaF9hdHRuYGAgd2hlZWwpIG9ubHkgc3VwcG9ydHMgQW1wZXJlIGFuZCBuZXdlcgogICAgKHNtXzgwKykuIE9uIGEgVDQgKHNtXzc1KSB0aGUga2VybmVsIHJhaXNlcyBgYEZsYXNoQXR0ZW50aW9uIG9ubHkgc3VwcG9ydHMKICAgIEFtcGVyZSBHUFVzIG9yIG5ld2VyYGAgYXQgZm9yd2FyZCB0aW1lLiBUaGUgaW5zdGFsbGVkIHZvcnRleCBmdW5uZWxzIGl0cwogICAgZGVuc2UtYXR0ZW50aW9uIHBhdGggdGhyb3VnaCB0aGUgbW9kdWxlLWxldmVsCiAgICBgYGxvY2FsX2ZsYXNoX2F0dG5fcWt2cGFja2VkX2Z1bmNgYCAoYW5kIGl0cyB2YXJsZW4gc2libGluZyksIHNvIHJlcGxhY2UKICAgIHRob3NlIHdpdGggYSBgYHRvcmNoLm5uLmZ1bmN0aW9uYWwuc2NhbGVkX2RvdF9wcm9kdWN0X2F0dGVudGlvbmBgIGVxdWl2YWxlbnQKICAgIC0tIFNEUEEncyBtYXRoIC8gbWVtLWVmZmljaWVudCBiYWNrZW5kcyBydW4gb24gZXZlcnkgQ1VEQSBHUFUuIFBhdGNoaW5nIHRoZQogICAgZnVuY3Rpb24gcmF0aGVyIHRoYW4gYSBjbGFzcyBuYW1lIHN1cnZpdmVzIHZvcnRleCByZWZhY3RvcnMuCgogICAgRnVsbCwgbm9uLXdpbmRvd2VkIGF0dGVudGlvbiBvbmx5IChldm8yXzdiJ3MgYXR0ZW50aW9uIGxheWVycyBhcmUgZ2xvYmFsKTsKICAgIGBgc29mdGNhcGBgIGFuZCBgYGFsaWJpX3Nsb3Blc2BgIGFyZSBub3Qgc3VwcG9ydGVkIGJ5IFNEUEEgYW5kIGFyZSBhc3NlcnRlZAogICAgdG8gYmUgdW5zZXQgcmF0aGVyIHRoYW4gc2lsZW50bHkgZHJvcHBlZC4KICAgICIiIgogICAgaW1wb3J0IG1hdGgKCiAgICBpbXBvcnQgdG9yY2gKCiAgICBkZWYgX3NkcGFfcWt2cGFja2VkKAogICAgICAgIHFrdiwgZHJvcG91dF9wPTAuMCwgc29mdG1heF9zY2FsZT1Ob25lLCBjYXVzYWw9RmFsc2UsCiAgICAgICAgd2luZG93X3NpemU9KC0xLCAtMSksIHNvZnRjYXA9MC4wLCBhbGliaV9zbG9wZXM9Tm9uZSwKICAgICAgICBkZXRlcm1pbmlzdGljPUZhbHNlLCByZXR1cm5fYXR0bl9wcm9icz1GYWxzZSwgKipfLAogICAgKToKICAgICAgICBhc3NlcnQgbm90IHNvZnRjYXAsICJTRFBBIGF0dGVudGlvbiBmYWxsYmFjayBkb2VzIG5vdCBzdXBwb3J0IHNvZnRjYXAiCiAgICAgICAgYXNzZXJ0IGFsaWJpX3Nsb3BlcyBpcyBOb25lLCAiU0RQQSBhdHRlbnRpb24gZmFsbGJhY2sgZG9lcyBub3Qgc3VwcG9ydCBhbGliaSIKICAgICAgICBpZiB0dXBsZSh3aW5kb3dfc2l6ZSkgbm90IGluICgoLTEsIC0xKSwpOgogICAgICAgICAgICBwcmludChmIiAgISEgYXR0ZW50aW9uIHdpbmRvd19zaXplPXt3aW5kb3dfc2l6ZX0gaWdub3JlZCBieSBTRFBBIGZhbGxiYWNrIikKICAgICAgICBxLCBrLCB2ID0gcWt2LnVuYmluZChkaW09MikgICAgICAgICAgICAgICAjIGVhY2ggKEIsIFMsIEgsIEQpCiAgICAgICAgcSA9IHEudHJhbnNwb3NlKDEsIDIpICAgICAgICAgICAgICAgICAgICAgIyAoQiwgSCwgUywgRCkKICAgICAgICBrID0gay50cmFuc3Bvc2UoMSwgMikKICAgICAgICB2ID0gdi50cmFuc3Bvc2UoMSwgMikKICAgICAgICBkID0gcS5zaGFwZVstMV0KICAgICAgICBzY2FsZSA9IHNvZnRtYXhfc2NhbGUgaWYgc29mdG1heF9zY2FsZSBpcyBub3QgTm9uZSBlbHNlIDEuMCAvIG1hdGguc3FydChkKQogICAgICAgIG91dCA9IHRvcmNoLm5uLmZ1bmN0aW9uYWwuc2NhbGVkX2RvdF9wcm9kdWN0X2F0dGVudGlvbigKICAgICAgICAgICAgcSwgaywgdiwgZHJvcG91dF9wPTAuMCwgaXNfY2F1c2FsPWJvb2woY2F1c2FsKSwgc2NhbGU9c2NhbGUsCiAgICAgICAgKQogICAgICAgIHJldHVybiBvdXQudHJhbnNwb3NlKDEsIDIpLmNvbnRpZ3VvdXMoKSAgICMgKEIsIFMsIEgsIEQpCgogICAgZGVmIF9zZHBhX3Zhcmxlbl9xa3ZwYWNrZWQocWt2LCBjdV9zZXFsZW5zLCBtYXhfc2VxbGVuLCBkcm9wb3V0X3A9MC4wLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc29mdG1heF9zY2FsZT1Ob25lLCBjYXVzYWw9RmFsc2UsICoqa3cpOgogICAgICAgICMgZmVhdHVyZXMuX2ZvcndhcmQgbmV2ZXIgcGFkcywgc28gZXZlcnkgYmF0Y2ggaXMgZml4ZWQtbGVuZ3RoOiBmb2xkIHRoZQogICAgICAgICMgc2luZ2xlIHNlcXVlbmNlIGJhY2sgdG8gKDEsIFMsIDMsIEgsIEQpIGFuZCByZXVzZSB0aGUgZGVuc2UgcGF0aC4KICAgICAgICBpZiBxa3YuZGltKCkgPT0gNDoKICAgICAgICAgICAgcWt2ID0gcWt2LnVuc3F1ZWV6ZSgwKQogICAgICAgIHJldHVybiBfc2RwYV9xa3ZwYWNrZWQocWt2LCBkcm9wb3V0X3AsIHNvZnRtYXhfc2NhbGUsIGNhdXNhbClbMF0KCiAgICBfc2RwYV9xa3ZwYWNrZWQuX2V2bzJfc2RwYSA9IFRydWUKICAgIHBhdGNoZWQgPSAwCiAgICBmb3IgbW9kbmFtZSBpbiAoInZvcnRleC5vcHMuYXR0bl9pbnRlcmZhY2UiLCAidm9ydGV4Lm1vZGVsLmF0dGVudGlvbiIpOgogICAgICAgIHRyeToKICAgICAgICAgICAgbW9kID0gaW1wb3J0bGliLmltcG9ydF9tb2R1bGUobW9kbmFtZSkKICAgICAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgZm9yIGF0dHIsIHJlcGwgaW4gKAogICAgICAgICAgICAoImxvY2FsX2ZsYXNoX2F0dG5fcWt2cGFja2VkX2Z1bmMiLCBfc2RwYV9xa3ZwYWNrZWQpLAogICAgICAgICAgICAoImxvY2FsX2ZsYXNoX2F0dG5fdmFybGVuX3FrdnBhY2tlZF9mdW5jIiwgX3NkcGFfdmFybGVuX3FrdnBhY2tlZCksCiAgICAgICAgKToKICAgICAgICAgICAgaWYgaGFzYXR0cihtb2QsIGF0dHIpIGFuZCBub3QgZ2V0YXR0cihnZXRhdHRyKG1vZCwgYXR0ciksICJfZXZvMl9zZHBhIiwgRmFsc2UpOgogICAgICAgICAgICAgICAgc2V0YXR0cihtb2QsIGF0dHIsIHJlcGwpCiAgICAgICAgICAgICAgICBwYXRjaGVkICs9IDEKICAgIHByaW50KGYiICByb3V0ZWQge3BhdGNoZWR9IHZvcnRleCBhdHRlbnRpb24gZW50cnlwb2ludChzKSB0aHJvdWdoIFNEUEEiKQoKCmRlZiBsb2FkX2V2bzIobW9kZWxfbmFtZTogc3RyLCBhbGxvd19mcDhfZmFsbGJhY2s6IGJvb2wgPSBUcnVlKToKICAgICIiIkJ1aWxkIGFuIGBgRXZvMmBgIHdpdGggdGhlIGNoZWNrcG9pbnQncyBvd24gcGlja2xlIGNvbnRlbnRzIHBlcm1pdHRlZC4KCiAgICBXaGVuIGBgYWxsb3dfZnA4X2ZhbGxiYWNrYGAgYW5kIHRoZSBjdXJyZW50IEdQVSBsYWNrcyBGUDggc3VwcG9ydCwgRlA4CiAgICBhdXRvY2FzdCBpcyBkaXNhYmxlZCBzbyB0aGUgaW5wdXQgcHJvamVjdGlvbnMgcnVuIGluIG5vcm1hbCBwcmVjaXNpb24uCiAgICAiIiIKICAgIGZyb20gZXZvMiBpbXBvcnQgRXZvMgoKICAgIGltcG9ydCB0b3JjaAoKICAgIHByaW50KAogICAgICAgIGYibG9hZF9ldm8yOiBsb2FkZXIucHkgcmV2aXNpb24ge0xPQURFUl9SRVZJU0lPTn07ICIKICAgICAgICBmInt0b3JjaC5jdWRhLmRldmljZV9jb3VudCgpfSBDVURBIGRldmljZShzKSB2aXNpYmxlIgogICAgKQoKICAgIGlmIGFsbG93X2ZwOF9mYWxsYmFjayBhbmQgbm90IF9ncHVfc3VwcG9ydHNfZnA4KCk6CiAgICAgICAgX25ldXRlcl9mcDhfYXV0b2Nhc3QoKQogICAgICAgIHByaW50KAogICAgICAgICAgICAibG9hZF9ldm8yOiBHUFUgbGFja3MgRlA4IHN1cHBvcnQgKGNvbXB1dGUgY2FwYWJpbGl0eSA8IDguOSkgLS0gIgogICAgICAgICAgICAiZGlzYWJsZWQgRlA4IGF1dG9jYXN0OyBwcm9qZWN0aW9uIHJlc3VsdHMgYXJlIGFuIGFwcHJveGltYXRpb24gb2YgIgogICAgICAgICAgICAidGhlIEZQOC1jYWxpYnJhdGVkIG1vZGVsLiIKICAgICAgICApCgogICAgaWYgYWxsb3dfZnA4X2ZhbGxiYWNrIGFuZCBub3QgX2dwdV9zdXBwb3J0c19iZjE2KCk6CiAgICAgICAgX3BhdGNoX2ZsYXNoX2F0dGVudGlvbl90b19zZHBhKCkKICAgICAgICBwcmludCgKICAgICAgICAgICAgImxvYWRfZXZvMjogR1BVIHByZWRhdGVzIEZsYXNoQXR0ZW50aW9uLTIgKGNvbXB1dGUgY2FwYWJpbGl0eSA8IDguMCkgIgogICAgICAgICAgICAiLS0gcm91dGluZyBhdHRlbnRpb24gdGhyb3VnaCB0b3JjaCBTRFBBLiIKICAgICAgICApCgogICAgd2l0aCB0cnVzdGVkX3RvcmNoX2xvYWQoKToKICAgICAgICBtb2RlbCA9IEV2bzIobW9kZWxfbmFtZSkKCiAgICBpZiBhbGxvd19mcDhfZmFsbGJhY2sgYW5kIG5vdCBfZ3B1X3N1cHBvcnRzX2JmMTYoKToKICAgICAgICBfY2FzdF90b19mcDE2KG1vZGVsKQogICAgICAgIHByaW50KAogICAgICAgICAgICAibG9hZF9ldm8yOiBHUFUgbGFja3MgYmZsb2F0MTYgKGNvbXB1dGUgY2FwYWJpbGl0eSA8IDguMCkgLS0gY2FzdCAiCiAgICAgICAgICAgICJtb2RlbCB0byBmbG9hdDE2OyB3YXRjaCBmb3IgaW5mL25hbiBmcm9tIHRoZSByZWR1Y2VkIGV4cG9uZW50IHJhbmdlLiIKICAgICAgICApCgogICAgcmV0dXJuIG1vZGVsCg=="
}

for rel, b64 in FILES.items():
    p = pathlib.Path(SRC, rel)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))
if SRC not in sys.path:
    sys.path.insert(0, SRC)
print("wrote", len(FILES), "modules to", SRC)

### 2 · Config — set `STAGE`, then Run All

In [ ]:
STAGE = "auto"          # datasets | extract | merge | train | publish | auto

WINDOW_SIZE  = 2048     # 8192 on Modal; shrunk to fit a 16 GB T4
MODEL_NAME   = "evo2_7b"
MAX_VARIANTS = 40_000   # ClinVar cap; lower to cut extraction cost
N_SHARDS     = 60       # DO NOT change between extract runs
MIN_STARS    = 2
EXTRACT_HOURS = 8.0     # stop extracting this long into the session (9 h wall)
PUBLISH_RUN  = ""       # for STAGE=publish: the run dir name from `train`

# Worker mode: extract only shards [SHARD_LO, SHARD_HI). Leave both None on the
# main notebook (does all pending). Set a disjoint range in each parallel copy,
# then attach every copy's output to the main notebook and run merge/train.
SHARD_LO = 57
SHARD_HI = 60

import os, time, glob, subprocess
WORK, OUT, DATA, TMP = "/kaggle/working", "/kaggle/working/out", "/kaggle/working/out/data", "/kaggle/working/tmp"

# Kaggle pins its whole preinstalled package set via PIP_CONSTRAINT, which
# silently overrides `pip install torch==2.4.1` back to the image's torch 2.10.
# Clear it so our version pins actually take.
for _v in ("PIP_CONSTRAINT", "UV_CONSTRAINT"):
    os.environ.pop(_v, None)

def sh(cmd, check=True):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=check)

### 3 · Restore state from a previous run

Copies `data/`, `features/`, `runs/` out of any attached input (this notebook's
earlier output, or a dataset) into the working tree. hg38 is symlinked, not
copied.

In [ ]:
for d in (f"{DATA}/genomes", f"{DATA}/datasets", f"{OUT}/features", f"{OUT}/runs", TMP):
    os.makedirs(d, exist_ok=True)

for base in glob.glob("/kaggle/input/*"):
    src_data = None
    for cand in (f"{base}/out/data", f"{base}/data"):
        if os.path.isdir(cand):
            src_data = cand
    if src_data:
        g = f"{src_data}/genomes"
        if os.path.isdir(g):
            for n in os.listdir(g):
                dst = f"{DATA}/genomes/{n}"
                if not os.path.exists(dst):
                    os.symlink(f"{g}/{n}", dst)
        sh(f"cp -n {src_data}/datasets/*.parquet {DATA}/datasets/ 2>/dev/null", check=False)
    for sub in ("features", "runs"):
        for cand in (f"{base}/out/{sub}", f"{base}/{sub}"):
            if os.path.isdir(cand):
                sh(f"cp -rn {cand}/. {OUT}/{sub}/ 2>/dev/null", check=False)

sh(f"ls -R {OUT} | head -40", check=False)

### 4 · Point `finetune.config` at Kaggle paths

In [ ]:
import finetune.config as C
C.DATA_PATH     = DATA
C.GENOMES_DIR   = f"{DATA}/genomes"
C.DATASETS_DIR  = f"{DATA}/datasets"
C.FEATURES_DIR  = f"{OUT}/features"
C.RUNS_DIR      = f"{OUT}/runs"
C.ACTIVE_RUN_DIR = f"{C.RUNS_DIR}/active"
C.CLINVAR_VCF_GZ = f"{C.DATASETS_DIR}/clinvar.vcf.gz"
C.HG38_FASTA     = f"{C.GENOMES_DIR}/hg38.fa"
C.HG19_CHR17_FASTA = f"{C.GENOMES_DIR}/GRCh37.p13_chr17.fa"

from finetune.config import FeatureConfig, TrainConfig, ClinVarConfig
FC = FeatureConfig(window_size=WINDOW_SIZE)
print("feature tag:", FC.tag())

need = {"datasets", "extract"} if STAGE == "auto" else {STAGE}
have_ds = os.path.exists(f"{C.DATASETS_DIR}/variants.parquet")

### 5 · Environment

`datasets` needs only light deps. `extract` rebuilds `common.py`'s CUDA stack
(torch 2.4.1 + a prebuilt flash-attn wheel + TE 1.13 + evo2 from source) — the
fragile step; needs Internet on.

In [ ]:
def clone_evo2(submodules):
    if os.path.isdir(f"{TMP}/evo2"):
        return
    flag = "--recurse-submodules" if submodules else "--depth 1"
    sh(f"git clone {flag} https://github.com/ArcInstitute/evo2.git {TMP}/evo2")

FLASH_WHL = ("https://github.com/Dao-AILab/flash-attention/releases/download/"
             "v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch2.4cxx11abiFALSE"
             "-{py}-{py}-linux_x86_64.whl")


def _pkg_version(name):
    import importlib.metadata as _md          # reads dist-info, does NOT import
    try:
        return _md.version(name)
    except _md.PackageNotFoundError:
        return None


def _stale_torch(_s):
    # A torch left half-imported or at the wrong version by an earlier Run All
    # cannot be fixed in-process (NameError _C, or the docstring RuntimeError).
    t = _s.modules.get("torch")
    if t is not None and (not hasattr(t, "_C")
                          or not getattr(t, "__version__", "").startswith("2.4")):
        raise SystemExit(
            "  *** Run menu -> 'Restart & Run All'. A stale torch "
            f"({getattr(t, '__version__', 'partial import')}) is loaded in this "
            "kernel and cannot be swapped in place. Installs are cached, so the "
            "next pass is quick. ***")


def build_gpu_env():
    import sys as _s

    # torch cannot be re-imported in a live kernel (RuntimeError: '...already has
    # a docstring'). If a wrong torch is already loaded, only a restart fixes it.
    _stale_torch(_s)

    py = f"cp{_s.version_info.major}{_s.version_info.minor}"
    os.environ["NVTE_FRAMEWORK"] = "pytorch"
    os.environ.setdefault("CUDA_HOME", "/usr/local/cuda")
    os.environ["MAX_JOBS"] = "4"

    sh("pip -q install 'setuptools<70' packaging wheel ninja cmake pybind11")
    clone_evo2(True)

    # evo2's install no longer hard-pins torch: the current Kaggle image ships
    # torch 2.10, which satisfies evo2's range, so `pip install .` leaves it in
    # place and the flash-attn / TE 1.13 stack below (built for 2.4's ABI) then
    # mismatches. Pin torch 2.4.1+cu124 ourselves first -- the cu124 wheel pulls
    # the matching nvidia-* runtime deps -- so evo2's install is a no-op for it.
    sh("pip -q install --force-reinstall torch==2.4.1 "
       "--index-url https://download.pytorch.org/whl/cu124")
    sh(f"cd {TMP}/evo2 && pip -q install .", check=False)

    # flash-attn: force the wheel that matches torch 2.4's ABI (evo2 may have
    # left a mismatched one). --no-deps so it can't move torch.
    sh(f"pip install --no-deps --force-reinstall '{FLASH_WHL.format(py=py)}'")

    # transformer-engine 1.13: cu12 lib has a wheel; the pytorch bindings
    # (transformer_engine_torch) are a source build against the *installed*
    # torch's ABI. evo2_7b needs FP8 input projections -> TE is mandatory.
    #
    # Two traps, both about torch:
    #  * pip's wheel cache holds a _torch build from an earlier Run All linked
    #    to a different torch -- it installs but fails to import. --no-cache-dir
    #    --no-binary forces a fresh compile against the current torch.
    #  * transformer_engine_torch's setup.py lists `torch` with NO version
    #    bound, so --force-reinstall (or an unconstrained resolve) happily pulls
    #    the newest torch off PyPI and uninstalls our 2.4.1 mid-install, leaving
    #    the .so linked to 2.4 headers but 2.x loaded (undefined-symbol on
    #    import). So: NO --force-reinstall, and pin torch==2.4.1 in the same
    #    command to hold the resolver.
    sh("pip uninstall -y transformer-engine transformer_engine transformer_engine_cu12 "
       "transformer_engine_torch", check=False)
    sh("pip -q install transformer_engine_cu12==1.13.0", check=False)
    r = subprocess.run(
        f"pip install -v --no-cache-dir --no-build-isolation "
        f"--no-binary transformer_engine_torch "
        f"torch==2.4.1 transformer_engine_torch==1.13.0 "
        f"transformer_engine[pytorch]==1.13.0 "
        f"2>&1 | tee {WORK}/te_build.log",
        shell=True,
    )
    if r.returncode:
        raise SystemExit(
            "  *** transformer-engine build failed. evo2_7b requires TE (FP8 "
            f"input projections). See {WORK}/te_build.log. ***")

    # The TE install still lists torch unbounded; make sure nothing bumped it.
    tv_now = _pkg_version("torch")
    if tv_now is None or not tv_now.startswith("2.4"):
        raise SystemExit(
            f"  *** torch became {tv_now} during the transformer-engine install "
            "(its setup.py depends on unpinned `torch`). Run menu -> 'Restart & "
            "Run All'. ***")

    # Confirm the freshly built _torch extension actually imports against this
    # torch -- an ABI mismatch here is silent until vortex sets HAS_TE=False.
    chk = subprocess.run(
        "python -c 'import transformer_engine.pytorch, transformer_engine_torch'",
        shell=True,
    )
    if chk.returncode:
        raise SystemExit(
            "  *** transformer-engine installed but fails to import (ABI "
            "mismatch with torch). Run menu -> 'Restart & Run All'; the fresh "
            "kernel rebuilds it against the pinned torch. ***")

    # Verify on disk without importing torch.
    tv, fv = _pkg_version("torch"), _pkg_version("flash-attn")
    print(f"on disk: torch={tv}  flash-attn={fv}")
    if tv is None or not tv.startswith("2.4"):
        raise SystemExit(
            f"  *** torch on disk is {tv}, expected 2.4.x. The explicit "
            "torch==2.4.1 pin above did not take (PIP_CONSTRAINT? evo2 install "
            "moved it?) -- check the pip output above. ***")
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

sh("pip -q install pyfaidx pyarrow openpyxl requests pandas scikit-learn numpy")
os.environ.setdefault("HF_HOME", f"{TMP}/hf")

def gpu_present():
    # Deliberately does NOT import torch -- see build_gpu_env(). Checks the
    # driver directly.
    return os.path.isdir("/proc/driver/nvidia/gpus") and bool(
        os.listdir("/proc/driver/nvidia/gpus")
    )

if need & {"extract"} and not gpu_present() and STAGE == "extract":
    raise SystemExit("STAGE=extract needs a GPU. Set Accelerator -> 'GPU T4 x2'.")

if need & {"datasets", "extract"}:
    if need & {"extract"} and gpu_present():
        build_gpu_env()
    elif need & {"extract"}:
        print("no GPU -> skipping the CUDA build; this run can only do 'datasets'")
    clone_evo2(False)
    C.HG19_CHR17_FASTA_GZ = f"{TMP}/evo2/notebooks/brca1/GRCh37.p13_chr17.fna.gz"
    C.BRCA1_DMS_XLSX = f"{TMP}/evo2/notebooks/brca1/41586_2018_461_MOESM3_ESM.xlsx"

### 6 · Stage: build datasets

In [ ]:
if (STAGE == "datasets") or (STAGE == "auto" and not have_ds):
    from finetune import data, sequences
    cc = ClinVarConfig(min_review_stars=MIN_STARS, max_variants=MAX_VARIANTS, balance_classes=True)
    combined = data.assign_splits(data.build_clinvar_dataset(cc), data.build_brca1_dataset())
    data.save_dataset(combined)
    sequences.ensure_hg38()
    sequences.ensure_hg19_chr17()
    have_ds = True
    import pandas as pd
    print(pd.read_parquet(f"{C.DATASETS_DIR}/variants.parquet").groupby("split").size())
else:
    print("skip (STAGE=%s, datasets present=%s)" % (STAGE, have_ds))

### 7 · Stage: extract features  (GPU, resumable)

In [ ]:
if STAGE in ("extract", "auto") and (STAGE == "extract" or have_ds):
    import sys as _sys
    _stale_torch(_sys)          # bail with a clear message on a polluted kernel
    import pandas as pd, torch
    from finetune import features as F
    from finetune.loader import load_evo2
    from finetune.sequences import build_variant_window, open_genome

    frame = (pd.read_parquet(f"{C.DATASETS_DIR}/variants.parquet")
               .sort_values("variant_id").reset_index(drop=True))

    # evo2_7b's weights alone are ~14 GB -- they do not fit one 16 GB T4. vortex
    # pipeline-parallelises across every visible CUDA device (ceil(num_layers /
    # device_count) blocks per GPU), so two T4s (~7 GB of weights each) is enough
    # but one is not. features._forward puts inputs on cuda:0, matching vortex's
    # first-device placement, so no other wiring is needed.
    ngpu = torch.cuda.device_count()
    total_vram = sum(torch.cuda.get_device_properties(i).total_memory
                     for i in range(ngpu)) / 2**30
    print(f"visible CUDA devices: {ngpu}  ({total_vram:.0f} GiB total)")
    # evo2_7b weights are ~14 GiB; need real headroom for activations too. One
    # 24 GiB L4 is plenty; two 16 GiB T4s work via vortex's pipeline split; one
    # T4 does not.
    if MODEL_NAME == "evo2_7b" and total_vram < 20:
        raise SystemExit(
            f"  *** evo2_7b needs >=20 GiB of GPU memory ({total_vram:.0f} GiB "
            f"across {ngpu} device(s) here). Use Accelerator 'GPU T4 x2' or "
            "'GPU L4x4' -- not a single 'GPU T4' or 'GPU P100'. ***")

    model = load_evo2(MODEL_NAME)

    # smoke test — one variant end to end
    r = frame.iloc[0]; g = open_genome(r["assembly"])
    ref_w, st = g.window(r["chrom"], int(r["pos"]), FC.window_size)
    rel = int(r["pos"]) - 1 - st
    var_w, _ = build_variant_window(ref_w, rel, r["alt"], expected_reference=r["ref"])
    t0 = time.time(); out = F.extract_variant_features(model, ref_w, var_w, rel, FC); g.close()
    spv = time.time() - t0
    print(f"smoke ok  {spv:.2f}s/variant  peak {torch.cuda.max_memory_allocated()/2**30:.1f} GiB")
    print(f"~{len(frame)*spv/3600:.1f} GPU-h for all {len(frame)} variants")

    bounds = F.shard_bounds(len(frame), N_SHARDS)
    pending = [i for i in range(len(bounds)) if not os.path.exists(F.shard_path(i, FC))]
    if SHARD_LO is not None:
        pending = [i for i in pending if SHARD_LO <= i < SHARD_HI]
        print(f"worker mode: restricted to shards [{SHARD_LO}, {SHARD_HI})")
    print(f"{len(bounds)-len(pending)}/{len(bounds)} shards done, {len(pending)} pending")

    deadline = time.time() + EXTRACT_HOURS * 3600
    for i in pending:
        if time.time() > deadline:
            print("time budget hit — Save Version and re-run to continue"); break
        s, e = bounds[i]
        print(f"=== shard {i} rows [{s},{e}) ===", flush=True)
        try:
            F.extract_shard(frame.iloc[s:e], i, FC, model=model)
        except Exception as ex:
            print(f"  shard {i} failed: {ex}")
    done = sum(os.path.exists(F.shard_path(i, FC)) for i in range(N_SHARDS))
    print(f"{done}/{N_SHARDS} shards on disk")
else:
    print("skip extract")

### 8 · Stage: merge · train · publish

In [ ]:
from finetune import features as F, train as T

merged = os.path.exists(f"{F.features_dir(FC)}/features.npz")
all_shards = have_ds and all(os.path.exists(F.shard_path(i, FC)) for i in range(N_SHARDS))

if STAGE in ("merge", "auto") and all_shards and not merged:
    F.merge_shards(FC); merged = True

if STAGE in ("train", "auto") and merged:
    trained = T.train_head(feature_config=FC, train_config=TrainConfig(head="mlp"))
    T.train_head(feature_config=FC, train_config=TrainConfig(head="linear"))
    T.summarise_runs()
    import json as _j
    print(_j.dumps(trained.metrics["splits"], indent=2, default=float))

if STAGE == "publish":
    assert PUBLISH_RUN, "set PUBLISH_RUN to a run dir name from the train stage"
    T.publish(PUBLISH_RUN)
    sh(f"ls -la {C.ACTIVE_RUN_DIR}")

### 9 · Next step

**Save Version** now. For the next run, *Add Input → Notebook Output → this
notebook → latest version* so the shards/datasets/runs you just produced are
restored by cell 3.

When a head is trained and published, get `out/runs/active/` onto the Modal
volume (from anywhere with the `modal` CLI):

```
# Kaggle UI → this notebook → Output → Download, then:
modal volume put evo2-finetune-data ./out/runs/active /runs/active
modal deploy main.py
```